# Final Ultra — Tox21 Drug Toxicity Prediction

### Leakage-aware ML, Deep Learning, CNN, Multitask Learning, R-GCN and Ensemble Research Notebook

**Output file:** `Final_Ultra.ipynb`  
**Dataset:** Tox21, 12 toxicity endpoints  
**Primary evaluation:** 3-fold scaffold-balanced cross-validation  
**Official metrics:** Mean ROC-AUC, Mean Accuracy, Best Endpoint AUC, Best Endpoint Accuracy

> **Research integrity note:** `Mean AUC > 0.90` is treated as a research target, not a guaranteed result. The reported R-GCN mean AUC of approximately 0.925 in the uploaded 2025 paper used an external ToxKG knowledge graph in addition to Tox21. The uploaded CSV alone does not contain those chemical-gene-pathway relations. Therefore, this notebook implements a reproducible **molecular relation-aware R-GCN + fingerprint/descriptor fusion** model and reports only scores produced by the current data and frozen folds. No target label is imputed as 0 or 1.


## Research design followed in this notebook

The notebook follows the methodological roadmap in `Drug_Path.pdf` and the six uploaded papers:

- DeepTox-style normalization, multitask learning, missing-label masking, dropout and ensemble ideas.
- Multitask CapsNet with 166-bit MACCS input, 13 molecular properties, two RBM layers and dynamic routing.
- Six-channel molecular 2D-grid CNN inspired by the multi-channel CNN paper, implemented with a transparent RDKit-based approximation so it remains reproducible without proprietary ChemAxon/AutoDock components.
- Rich molecular graph learning with relation-specific graph convolutions. Bond types are modeled as graph relations and fused with ECFP4 and molecular descriptors.
- Frozen, leakage-aware 3-fold scaffold-balanced outer CV. Inner scaffold holdout is used for deep-model early stopping. Preprocessing is fitted on training data only.
- Elastic Net and GPS are intentionally excluded, as requested.

### Model list

1. Random Forest  
2. Extra Trees  
3. XGBoost  
4. SVM with RBF kernel  
5. DeepTox-style Multi-task DNN  
6. Multitask CapsNet + RBM + Dynamic Routing  
7. Multi-channel 2D CNN  
8. Compact MTL-DNN  
9. Molecular R-GCN + Fingerprint/Descriptor Fusion  
10. Leakage-safe equal-weight Soft-Voting Ensemble


## 0. Environment setup — automatic installation with Bangla status messages

এই cell প্রয়োজনীয় package পরীক্ষা করবে। কোনো package missing হলে আগে তার নাম দেখাবে, তারপর install করবে, এবং শেষে installation সম্পন্ন হওয়ার message দেবে।


In [ ]:
import sys, subprocess, importlib.util, platform, os

PACKAGE_MAP = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "joblib": "joblib",
    "tqdm": "tqdm",
    "psutil": "psutil",
    "dill": "dill",
    "rdkit": "rdkit",
    "xgboost": "xgboost",
    "torch": "torch",
}

print("🔎 প্রয়োজনীয় Python package পরীক্ষা করা হচ্ছে...")
missing = [pip_name for import_name, pip_name in PACKAGE_MAP.items()
           if importlib.util.find_spec(import_name) is None]

if missing:
    print("⚠️ নিচের package গুলো missing পাওয়া গেছে:")
    for name in missing:
        print("   •", name)
    print("📦 Missing package গুলো install করা হচ্ছে...")
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "--upgrade-strategy", "only-if-needed", *missing
        ])
    except subprocess.CalledProcessError:
        # Some older environments expose RDKit only as rdkit-pypi.
        if "rdkit" in missing:
            retry = ["rdkit-pypi" if x == "rdkit" else x for x in missing]
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *retry])
        else:
            raise
    print("✅ প্রয়োজনীয় package installation সম্পন্ন হয়েছে।")
else:
    print("✅ সব প্রয়োজনীয় package আগে থেকেই installed আছে।")

print("🐍 Python:", sys.version.split()[0])
print("💻 Platform:", platform.platform())
print("➡️ এখন পরের cell run করুন।")


## 1. Import libraries and automatic CPU/GPU selection

GPU পাওয়া গেলে PyTorch এবং compatible XGBoost computation GPU ব্যবহার করবে। GPU না থাকলে notebook স্বয়ংক্রিয়ভাবে CPU-তে চলবে।


In [ ]:
import os, gc, re, json, math, time, random, hashlib, warnings, shutil, sys
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Callable, Any

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, dill, psutil
from tqdm.auto import tqdm
from scipy.special import expit
from scipy.optimize import minimize

from IPython.display import display, Markdown
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neural_network import BernoulliRBM

from xgboost import XGBClassifier

from rdkit import Chem, DataStructs, RDLogger, RDConfig
from rdkit.Chem import AllChem, MACCSkeys, Descriptors, rdMolDescriptors, Draw, ChemicalFeatures
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"

print("✅ সব library সফলভাবে import হয়েছে।")
if DEVICE.type == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"🚀 GPU পাওয়া গেছে: {gpu_name} | VRAM ≈ {gpu_mem:.1f} GB")
    print("✅ Deep Learning model GPU-তে চলবে এবং mixed precision সক্রিয় থাকবে।")
else:
    print("ℹ️ GPU পাওয়া যায়নি। সব model CPU-তে চলবে।")
    print("✅ Code পরিবর্তন করার প্রয়োজন নেই; notebook স্বয়ংক্রিয়ভাবে CPU mode ব্যবহার করছে।")


## 2. Global configuration and project folders

এই cell Google Colab, Anaconda/JupyterLab, Windows এবং Linux environment-এর জন্য dataset path স্বয়ংক্রিয়ভাবে খুঁজবে। `SMOKE_TEST=False` থাকলে full research run হবে।


In [ ]:
SEED = 42
N_FOLDS = 3
TARGET_MEAN_AUC = 0.90
SPLIT_MODE = "scaffold_balanced"
THRESHOLD = 0.50
SMOKE_TEST = bool(int(os.environ.get("FINAL_ULTRA_SMOKE", "0")))
FORCE_REBUILD = False
SAVE_FITTED_CLASSICAL_MODELS = False
REFIT_DEEP_ON_FULL_OUTER_TRAIN = not SMOKE_TEST

# Optional Google Drive mount. It is attempted only inside Colab.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            print("🔗 Google Drive mount করা হচ্ছে, যাতে checkpoint বিদ্যুৎ/রানটাইম সমস্যার পরও থাকে...")
            drive.mount("/content/drive")
    except Exception as e:
        print("ℹ️ Google Drive mount করা যায়নি বা প্রয়োজন হয়নি:", e)

def _valid_tox21_csv(path: Path) -> bool:
    try:
        cols = pd.read_csv(path, nrows=2).columns.tolist()
        return "smiles" in cols and len([c for c in cols if c.startswith(("NR-", "SR-"))]) >= 10
    except Exception:
        return False

def find_tox21_dataset() -> Path:
    roots = [
        Path.cwd(), Path.cwd().parent,
        Path("D:/Drug_Toxicity"),
        Path("/content/drive/MyDrive/Drug_Toxicity"),
        Path("/content"), Path("/mnt/data"),
    ]
    names = ["tox21.csv", "tox21(1).csv", "tox21(2).csv", "tox21 (1).csv", "tox21 (2).csv"]
    candidates = []
    for root in roots:
        for name in names:
            candidates += [root / "datasets" / name, root / name]
    for path in candidates:
        if path.exists() and _valid_tox21_csv(path):
            return path.resolve()
    for root in roots:
        if root.exists():
            for path in root.glob("**/*tox21*.csv"):
                if not any(x in path.name.lower() for x in ["result", "audit", "clean", "prediction", "distribution"]):
                    if _valid_tox21_csv(path):
                        return path.resolve()
    raise FileNotFoundError(
        "tox21 CSV পাওয়া যায়নি। Colab-এ /content বা Google Drive-এর Drug_Toxicity/datasets folder-এ file রাখুন।"
    )

DATA_PATH = find_tox21_dataset()
if DATA_PATH.parent.name.lower() == "datasets":
    PROJECT_ROOT = DATA_PATH.parent.parent
elif IN_COLAB and Path("/content/drive/MyDrive").exists():
    PROJECT_ROOT = Path("/content/drive/MyDrive/Drug_Toxicity")
else:
    PROJECT_ROOT = DATA_PATH.parent / "Final_Ultra_Project"

WORK_ROOT = PROJECT_ROOT / "final_ultra"
CACHE_DIR = WORK_ROOT / "cache"
MODEL_DIR = WORK_ROOT / "models"
RESULT_DIR = WORK_ROOT / "results"
FIG_DIR = WORK_ROOT / "figures"
LOG_DIR = WORK_ROOT / "logs"
PRED_DIR = WORK_ROOT / "predictions"
for folder in [WORK_ROOT, CACHE_DIR, MODEL_DIR, RESULT_DIR, FIG_DIR, LOG_DIR, PRED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RAM_GB = psutil.virtual_memory().total / (1024**3)
CPU_COUNT = max(1, os.cpu_count() or 1)
LOW_RESOURCE = RAM_GB < 11 and DEVICE.type == "cpu"
try:
    torch.set_num_threads(2 if SMOKE_TEST else min(8, CPU_COUNT))
except Exception:
    pass

RESOURCE = {
    "tree_estimators": 8 if SMOKE_TEST else (450 if LOW_RESOURCE else 700),
    "xgb_estimators": 3 if SMOKE_TEST else (450 if LOW_RESOURCE else 700),
    "svd_components": 12 if SMOKE_TEST else (192 if LOW_RESOURCE else 320),
    "dnn_epochs": 1 if SMOKE_TEST else (70 if LOW_RESOURCE else 140),
    "caps_epochs": 1 if SMOKE_TEST else (60 if LOW_RESOURCE else 120),
    "cnn_epochs": 1 if SMOKE_TEST else (50 if LOW_RESOURCE else 100),
    "rgcn_epochs": 1 if SMOKE_TEST else (70 if LOW_RESOURCE else 140),
    "patience": 1 if SMOKE_TEST else (12 if LOW_RESOURCE else 18),
    "tab_batch": 64 if SMOKE_TEST else (128 if LOW_RESOURCE else (256 if DEVICE.type == "cuda" else 192)),
    "cnn_batch": 16 if SMOKE_TEST else (24 if LOW_RESOURCE else (128 if DEVICE.type == "cuda" else 48)),
    "graph_batch": 12 if SMOKE_TEST else (32 if LOW_RESOURCE else (128 if DEVICE.type == "cuda" else 64)),
    "num_workers": 0 if os.name == "nt" or SMOKE_TEST else min(4, max(1, CPU_COUNT // 4)),
}

print("✅ Project configuration তৈরি হয়েছে।")
print("📁 Dataset:", DATA_PATH)
print("📁 Project root:", PROJECT_ROOT)
print("📁 Restart-safe checkpoint root:", WORK_ROOT)
print(f"🧠 RAM ≈ {RAM_GB:.1f} GB | CPU threads = {CPU_COUNT} | Device = {DEVICE}")
print("🧪 Mode:", "SMOKE TEST" if SMOKE_TEST else "FULL RESEARCH RUN")


In [ ]:
def seed_everything(seed: int = SEED) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(False)
    except Exception:
        pass

seed_everything()

CONFIG = {
    "seed": SEED,
    "n_folds": N_FOLDS,
    "split_mode": SPLIT_MODE,
    "threshold": THRESHOLD,
    "target_mean_auc": TARGET_MEAN_AUC,
    "device": str(DEVICE),
    "smoke_test": SMOKE_TEST,
    "resource": RESOURCE,
    "dataset_path": str(DATA_PATH),
    "project_root": str(PROJECT_ROOT),
    "models": [
        "Random Forest", "Extra Trees", "XGBoost", "SVM-RBF",
        "DeepTox-style MT-DNN", "Multitask CapsNet + RBM",
        "Multi-channel 2D CNN", "Compact MTL-DNN",
        "Molecular R-GCN Fusion", "Soft-Voting Ensemble"
    ],
    "excluded_models": ["Elastic Net", "GPS"],
}
with open(WORK_ROOT / "final_ultra_config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)

print("✅ Random seed freeze করা হয়েছে এবং configuration JSON save হয়েছে।")
print("🚫 Excluded models: Elastic Net, GPS")


## 3. Load dataset and strict schema validation

Target columns-এ কেবল `0`, `1`, অথবা missing (`NaN`) গ্রহণ করা হবে। Missing target কোনো অবস্থাতেই inactive/active label হিসেবে ধরা হবে না।


In [ ]:
RAW_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
df_raw = pd.read_csv(DATA_PATH)

EXPECTED_TARGETS = [
    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase", "NR-ER", "NR-ER-LBD",
    "NR-PPAR-gamma", "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53"
]
missing_cols = [c for c in EXPECTED_TARGETS + ["smiles"] if c not in df_raw.columns]
if missing_cols:
    raise ValueError(f"Dataset schema ভুল। Missing columns: {missing_cols}")

if "mol_id" not in df_raw.columns:
    df_raw["mol_id"] = [f"MOL_{i:06d}" for i in range(len(df_raw))]

for target in EXPECTED_TARGETS:
    df_raw[target] = pd.to_numeric(df_raw[target], errors="coerce")
    invalid_values = set(df_raw[target].dropna().unique()) - {0.0, 1.0}
    if invalid_values:
        raise ValueError(f"{target} column-এ 0/1 ছাড়া value পাওয়া গেছে: {invalid_values}")

TARGETS = EXPECTED_TARGETS.copy()
if SMOKE_TEST:
    # Keep enough endpoints to exercise masking and multi-task logic.
    TARGETS = EXPECTED_TARGETS[:4]
    df_raw = df_raw.sample(n=min(240, len(df_raw)), random_state=SEED).reset_index(drop=True)
    print("🧪 Smoke test-এর জন্য 240 row এবং প্রথম 4 endpoint ব্যবহার করা হচ্ছে।")

print("✅ Tox21 dataset সফলভাবে load ও validate হয়েছে।")
print("📌 Shape:", df_raw.shape)
print("📌 Active target count:", len(TARGETS))
print("📌 SHA256:", RAW_SHA256[:24] + "...")
display(df_raw[["mol_id", "smiles"] + TARGETS].head(3))


## 4. Dataset audit — missing labels, class imbalance and endpoint statistics


In [ ]:
def build_endpoint_audit(frame: pd.DataFrame, targets: List[str]) -> pd.DataFrame:
    rows = []
    for target in targets:
        y = frame[target]
        valid = int(y.notna().sum())
        pos = int((y == 1).sum())
        neg = int((y == 0).sum())
        missing = int(y.isna().sum())
        rows.append({
            "Endpoint": target,
            "Labeled": valid,
            "Toxic (1)": pos,
            "Non-toxic (0)": neg,
            "Missing": missing,
            "Missing %": 100 * missing / len(frame),
            "Positive %": 100 * pos / max(valid, 1),
            "Neg:Pos": neg / max(pos, 1),
        })
    return pd.DataFrame(rows)

audit_df = build_endpoint_audit(df_raw, TARGETS)
audit_df.to_csv(RESULT_DIR / "01_endpoint_distribution_raw.csv", index=False)

print("✅ Endpoint audit সম্পন্ন হয়েছে।")
print("🔒 Missing label-কে 0 অথবা 1 হিসেবে count করা হয়নি।")
display(
    audit_df.style
    .format({"Missing %": "{:.2f}%", "Positive %": "{:.2f}%", "Neg:Pos": "{:.2f}:1"})
    .background_gradient(subset=["Missing %"], cmap="Oranges")
    .background_gradient(subset=["Neg:Pos"], cmap="Reds")
    .set_caption("Tox21 Endpoint-wise Label Audit")
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_df = audit_df.sort_values("Missing %")
axes[0].barh(plot_df["Endpoint"], plot_df["Missing %"], color="#F4A261")
axes[0].set_title("Missing Values by Endpoint")
axes[0].set_xlabel("Missing labels (%)")
for i, v in enumerate(plot_df["Missing %"]):
    axes[0].text(v + 0.2, i, f"{v:.1f}%", va="center", fontsize=9)

plot_df = audit_df.sort_values("Neg:Pos")
axes[1].barh(plot_df["Endpoint"], plot_df["Neg:Pos"], color="#2A9D8F")
axes[1].set_title("Class Imbalance — Negative : Positive Ratio")
axes[1].set_xlabel("Negative / Positive")
for i, v in enumerate(plot_df["Neg:Pos"]):
    axes[1].text(v + 0.25, i, f"{v:.1f}:1", va="center", fontsize=9)

plt.suptitle("Tox21 Dataset Challenges", fontsize=16, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "01_missing_and_imbalance.png", dpi=220, bbox_inches="tight")
plt.show(); plt.close()
print("✅ Missingness এবং imbalance visualization তৈরি ও save হয়েছে।")


In [ ]:
label_corr = df_raw[TARGETS].corr(method="pearson", min_periods=100)
plt.figure(figsize=(11, 8))
sns.heatmap(label_corr, cmap="vlag", center=0, annot=True, fmt=".2f", square=True,
            linewidths=0.5, cbar_kws={"label": "Pairwise correlation on observed labels"})
plt.title("Observed-label Correlation between Tox21 Endpoints", fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR / "02_endpoint_correlation.png", dpi=220, bbox_inches="tight")
plt.show(); plt.close()
print("✅ Endpoint correlation heatmap তৈরি হয়েছে।")


## 5. Chemical preprocessing — standardization, salts, stereochemistry and duplicate control

Pipeline:

1. RDKit sanitization  
2. Cleanup  
3. Largest organic fragment / fragment parent  
4. Uncharging  
5. Canonical tautomer  
6. Isomeric canonical SMILES and InChIKey  
7. Bemis-Murcko scaffold  
8. Invalid molecule quarantine  
9. Duplicate label resolution before splitting


In [ ]:
TAUTOMER_ENUMERATOR = rdMolStandardize.TautomerEnumerator()
UNCHARGER = rdMolStandardize.Uncharger()
try:
    FRAGMENT_CHOOSER = rdMolStandardize.LargestFragmentChooser(preferOrganic=True)
except TypeError:
    FRAGMENT_CHOOSER = rdMolStandardize.LargestFragmentChooser()

def standardize_molecule(smiles: str) -> Optional[Dict[str, str]]:
    try:
        mol = Chem.MolFromSmiles(str(smiles), sanitize=True)
        if mol is None:
            return None
        mol = rdMolStandardize.Cleanup(mol)
        mol = FRAGMENT_CHOOSER.choose(mol)
        mol = UNCHARGER.uncharge(mol)
        mol = TAUTOMER_ENUMERATOR.Canonicalize(mol)
        Chem.SanitizeMol(mol)
        std_smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
        inchikey = Chem.MolToInchiKey(mol)
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=True)
        if not scaffold:
            scaffold = "ACYCLIC_" + hashlib.sha1(std_smiles.encode()).hexdigest()[:16]
        return {
            "std_smiles": std_smiles,
            "inchikey": inchikey,
            "scaffold": scaffold,
            "has_mixture": int("." in str(smiles)),
        }
    except Exception:
        return None

print("✅ Chemical standardization function প্রস্তুত হয়েছে।")


In [ ]:
CURATED_PATH = CACHE_DIR / f"curated_{RAW_SHA256[:10]}_{len(TARGETS)}t.csv"
INVALID_PATH = RESULT_DIR / "02_invalid_smiles.csv"
PROGRESS_PATH = CACHE_DIR / "curation_progress.json"
PARTIAL_PATH = CACHE_DIR / "curation_partial.jsonl"

if CURATED_PATH.exists() and not FORCE_REBUILD:
    df_std = pd.read_csv(CURATED_PATH)
    print("♻️ Cached standardized dataset load হয়েছে।")
else:
    start = 0
    records = []
    invalid_records = []
    if PARTIAL_PATH.exists() and PROGRESS_PATH.exists() and not FORCE_REBUILD:
        with open(PARTIAL_PATH, "r", encoding="utf-8") as f:
            records = [json.loads(line) for line in f if line.strip()]
        state = json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
        start = int(state.get("next_index", len(records)))
        print(f"🔄 আগের chemical preprocessing checkpoint পাওয়া গেছে। Row {start} থেকে resume হচ্ছে।")
    else:
        PARTIAL_PATH.unlink(missing_ok=True)
        PROGRESS_PATH.unlink(missing_ok=True)

    with open(PARTIAL_PATH, "a", encoding="utf-8") as writer:
        for i in tqdm(range(start, len(df_raw)), desc="Chemical standardization"):
            row = df_raw.iloc[i]
            out = standardize_molecule(row["smiles"])
            if out is None:
                invalid_records.append(row.to_dict())
            else:
                rec = row.to_dict()
                rec["raw_smiles"] = rec.pop("smiles")
                rec.update(out)
                writer.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")
                records.append(rec)
            if (i + 1) % 100 == 0 or i + 1 == len(df_raw):
                writer.flush()
                PROGRESS_PATH.write_text(json.dumps({"next_index": i + 1}), encoding="utf-8")

    df_std = pd.DataFrame(records)
    df_std.to_csv(CURATED_PATH, index=False)
    if invalid_records:
        pd.DataFrame(invalid_records).to_csv(INVALID_PATH, index=False)
    else:
        pd.DataFrame(columns=df_raw.columns).to_csv(INVALID_PATH, index=False)
    PARTIAL_PATH.unlink(missing_ok=True)
    PROGRESS_PATH.unlink(missing_ok=True)
    print("✅ Chemical preprocessing সম্পন্ন এবং cache save হয়েছে।")

print("📌 Valid standardized molecules:", len(df_std))
print("📌 Invalid molecules removed:", len(df_raw) - len(df_std))
print("📌 Dot-separated mixtures/salts in raw input:", int(df_std["has_mixture"].sum()))


In [ ]:
def resolve_duplicate_group(group: pd.DataFrame) -> pd.Series:
    out = group.iloc[0].copy()
    out["source_mol_ids"] = "|".join(group["mol_id"].astype(str).tolist())
    out["duplicate_count"] = len(group)
    for target in TARGETS:
        values = sorted(group[target].dropna().unique().tolist())
        if len(values) == 1:
            out[target] = values[0]
        elif len(values) == 0:
            out[target] = np.nan
        else:
            # Conflicting duplicate measurements are unknown, not forced to 0/1.
            out[target] = np.nan
    return out

before_dup = len(df_std)
df = df_std.groupby("inchikey", sort=False, group_keys=False).apply(resolve_duplicate_group).reset_index(drop=True)
conflict_count = int(df[TARGETS].isna().sum().sum() - df_std[TARGETS].isna().sum().sum())

Y = df[TARGETS].to_numpy(dtype=np.float32)
LABEL_MASK = (~np.isnan(Y)).astype(np.float32)
Y_TENSOR_SAFE = np.nan_to_num(Y, nan=0.0).astype(np.float32)

print("✅ Duplicate resolution সম্পন্ন হয়েছে।")
print("📌 Rows before duplicate merge:", before_dup)
print("📌 Rows after duplicate merge:", len(df))
print("📌 Conflicting endpoint labels converted to missing:", max(0, conflict_count))
print("🔒 Target NaN tensor storage-এর জন্য temporary 0 placeholder পেয়েছে, কিন্তু mask=0 হওয়ায় loss/metric-এ কখনও label হিসেবে গণনা হবে না।")
print("📌 Observed labels:", int(LABEL_MASK.sum()), "| Missing labels:", int((1-LABEL_MASK).sum()))

df.to_csv(RESULT_DIR / "03_curated_molecule_mapping.csv", index=False)


In [ ]:
show_n = min(12, len(df))
show_idx = np.linspace(0, len(df)-1, show_n, dtype=int)
mols = [Chem.MolFromSmiles(df.iloc[i]["std_smiles"]) for i in show_idx]
legends = [str(df.iloc[i]["mol_id"]) for i in show_idx]
img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(260, 200), legends=legends)
display(img)
print("✅ Standardized molecule sample visualization সম্পন্ন হয়েছে।")


## 6. Frozen 3-fold scaffold-balanced cross-validation

একই Bemis-Murcko scaffold কেবল একটি outer fold-এ থাকবে। Greedy multi-objective assignment fold size, labeled count এবং positive count একসঙ্গে balance করার চেষ্টা করে।


In [ ]:
def make_scaffold_balanced_folds(frame: pd.DataFrame, targets: List[str], n_folds: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    groups = []
    for scaffold, part in frame.groupby("scaffold", sort=False):
        arr = part[targets].to_numpy(float)
        groups.append({
            "scaffold": scaffold,
            "indices": part.index.to_numpy(),
            "n": len(part),
            "pos": np.nansum(arr == 1, axis=0).astype(float),
            "labeled": np.sum(~np.isnan(arr), axis=0).astype(float),
        })
    # Large and rare-positive-rich scaffolds are placed first.
    rng.shuffle(groups)
    groups.sort(key=lambda g: (g["n"] + 3.0 * np.sqrt(g["pos"].sum() + 1)), reverse=True)

    total_n = len(frame)
    total_pos = np.sum([g["pos"] for g in groups], axis=0)
    total_lab = np.sum([g["labeled"] for g in groups], axis=0)
    target_n = total_n / n_folds
    target_pos = total_pos / n_folds
    target_lab = total_lab / n_folds

    fold_n = np.zeros(n_folds, dtype=float)
    fold_pos = np.zeros((n_folds, len(targets)), dtype=float)
    fold_lab = np.zeros((n_folds, len(targets)), dtype=float)
    assignment = {}

    def objective(n_arr, pos_arr, lab_arr):
        size_err = np.mean(((n_arr - target_n) / max(target_n, 1)) ** 2)
        pos_err = np.mean(((pos_arr - target_pos) / (target_pos + 1.0)) ** 2)
        lab_err = np.mean(((lab_arr - target_lab) / (target_lab + 1.0)) ** 2)
        return 0.25 * size_err + 0.55 * pos_err + 0.20 * lab_err

    for g in groups:
        scores = []
        for fold in range(n_folds):
            n2, p2, l2 = fold_n.copy(), fold_pos.copy(), fold_lab.copy()
            n2[fold] += g["n"]
            p2[fold] += g["pos"]
            l2[fold] += g["labeled"]
            scores.append(objective(n2, p2, l2))
        min_score = min(scores)
        ties = [i for i, s in enumerate(scores) if abs(s - min_score) < 1e-12]
        chosen = int(rng.choice(ties))
        assignment[g["scaffold"]] = chosen
        fold_n[chosen] += g["n"]
        fold_pos[chosen] += g["pos"]
        fold_lab[chosen] += g["labeled"]

    return frame["scaffold"].map(assignment).to_numpy(dtype=int)

print("✅ Scaffold-balanced fold generator প্রস্তুত হয়েছে।")


In [ ]:
FOLD_PATH = CACHE_DIR / f"folds_{RAW_SHA256[:10]}_{len(df)}_{len(TARGETS)}t_{N_FOLDS}f.csv"
if FOLD_PATH.exists() and not FORCE_REBUILD:
    fold_file = pd.read_csv(FOLD_PATH)
    if len(fold_file) != len(df):
        raise ValueError("Cached fold file row count বর্তমান curated dataset-এর সঙ্গে মিলছে না।")
    df["fold"] = fold_file["fold"].astype(int).to_numpy()
    print("♻️ Frozen fold assignment cache থেকে load হয়েছে।")
else:
    df["fold"] = make_scaffold_balanced_folds(df, TARGETS, N_FOLDS, SEED)
    df[["mol_id", "inchikey", "scaffold", "fold"]].to_csv(FOLD_PATH, index=False)
    print("✅ নতুন 3-fold scaffold-balanced assignment তৈরি ও freeze করা হয়েছে।")

# Leakage audit
scaffold_fold_count = df.groupby("scaffold")["fold"].nunique()
if scaffold_fold_count.max() != 1:
    raise AssertionError("Scaffold leakage পাওয়া গেছে।")

fold_rows=[]
for fold in range(N_FOLDS):
    part=df[df["fold"]==fold]
    row={"Fold":fold+1,"Molecules":len(part),"Unique scaffolds":part["scaffold"].nunique()}
    for t in TARGETS:
        row[f"{t} positive"] = int((part[t]==1).sum())
    fold_rows.append(row)
fold_quality_df=pd.DataFrame(fold_rows)
fold_quality_df.to_csv(RESULT_DIR / "04_fold_quality.csv", index=False)

print("✅ Scaffold leakage audit passed: কোনো scaffold একাধিক fold-এ নেই।")
display(fold_quality_df.style.set_caption("Frozen Outer-fold Quality Audit").background_gradient(cmap="Blues"))


In [ ]:
fold_dist=[]
for fold in range(N_FOLDS):
    part=df[df["fold"]==fold]
    for t in TARGETS:
        valid=part[t].notna().sum()
        fold_dist.append({"Fold":f"Fold {fold+1}","Endpoint":t,"Positive %":100*(part[t]==1).sum()/max(valid,1)})
fold_dist=pd.DataFrame(fold_dist)
plt.figure(figsize=(14,6))
sns.barplot(data=fold_dist, x="Endpoint", y="Positive %", hue="Fold", palette="Set2")
plt.xticks(rotation=35, ha="right")
plt.title("Positive-label Distribution across Frozen Scaffold Folds", fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR / "03_fold_positive_distribution.png", dpi=220, bbox_inches="tight")
plt.show(); plt.close()
print("✅ Fold balance visualization তৈরি হয়েছে।")


## 7. Feature engineering — ECFP4, MACCS and 13 validated RDKit descriptors

- ECFP4: radius 2, 2048 bits  
- MACCS: RDKit 167-bit representation; bit 0 is removed for the paper-style 166-bit CapsNet input  
- 13 descriptors: molecular weight, LogP, molar refractivity, TPSA, HBD, HBA, rotatable bonds, ring count, aromatic ring count, N/O count, fraction Csp3, heavy atom count and Labute ASA

Feature generation is checkpointed every 100 molecules. After interruption, rerunning this cell resumes from the saved index.


In [ ]:
DESC_SPECS = [
    ("MolWt", Descriptors.MolWt),
    ("MolLogP", Descriptors.MolLogP),
    ("MolMR", Descriptors.MolMR),
    ("TPSA", rdMolDescriptors.CalcTPSA),
    ("HBD", rdMolDescriptors.CalcNumHBD),
    ("HBA", rdMolDescriptors.CalcNumHBA),
    ("RotatableBonds", rdMolDescriptors.CalcNumRotatableBonds),
    ("RingCount", rdMolDescriptors.CalcNumRings),
    ("AromaticRingCount", rdMolDescriptors.CalcNumAromaticRings),
    ("NOCount", Descriptors.NOCount),
    ("FractionCSP3", rdMolDescriptors.CalcFractionCSP3),
    ("HeavyAtomCount", Descriptors.HeavyAtomCount),
    ("LabuteASA", rdMolDescriptors.CalcLabuteASA),
]
DESC_NAMES = [name for name, _ in DESC_SPECS]
COMPACT_DESC_NAMES = ["MolWt", "MolLogP", "TPSA", "HBD", "HBA", "RotatableBonds", "RingCount"]
COMPACT_DESC_IDX = [DESC_NAMES.index(x) for x in COMPACT_DESC_NAMES]

try:
    MORGAN_GENERATOR = AllChem.GetMorganGenerator(radius=2, fpSize=2048)
except Exception:
    MORGAN_GENERATOR = None

def fingerprint_and_descriptors(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Standardized SMILES থেকে molecule তৈরি হয়নি।")
    ecfp = np.zeros(2048, dtype=np.uint8)
    fp = MORGAN_GENERATOR.GetFingerprint(mol) if MORGAN_GENERATOR is not None else AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    DataStructs.ConvertToNumpyArray(fp, ecfp)
    maccs = np.zeros(167, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(MACCSkeys.GenMACCSKeys(mol), maccs)
    desc=[]
    for _, fn in DESC_SPECS:
        try:
            value=float(fn(mol))
            desc.append(value if np.isfinite(value) else np.nan)
        except Exception:
            desc.append(np.nan)
    return ecfp, maccs, np.asarray(desc, dtype=np.float32)

print("✅ Feature functions প্রস্তুত হয়েছে।")


In [ ]:
FEATURE_ROOT = CACHE_DIR / f"features_{RAW_SHA256[:10]}_{len(df)}"
FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
ECFP_PATH = FEATURE_ROOT / "ecfp4_2048.npy"
MACCS_PATH = FEATURE_ROOT / "maccs_167.npy"
DESC_PATH = FEATURE_ROOT / "rdkit_desc_13.npy"
FEATURE_PROGRESS = FEATURE_ROOT / "progress.json"

shape_ok = ECFP_PATH.exists() and MACCS_PATH.exists() and DESC_PATH.exists()
if shape_ok and not FORCE_REBUILD:
    X_ECFP = np.load(ECFP_PATH, mmap_mode="r")
    X_MACCS = np.load(MACCS_PATH, mmap_mode="r")
    X_DESC = np.load(DESC_PATH, mmap_mode="r")
    if X_ECFP.shape[0] != len(df):
        shape_ok=False

if not shape_ok or FORCE_REBUILD:
    if FORCE_REBUILD:
        for p in [ECFP_PATH, MACCS_PATH, DESC_PATH, FEATURE_PROGRESS]:
            p.unlink(missing_ok=True)
    if not ECFP_PATH.exists():
        X_ECFP_W = np.lib.format.open_memmap(ECFP_PATH, mode="w+", dtype=np.uint8, shape=(len(df), 2048))
        X_MACCS_W = np.lib.format.open_memmap(MACCS_PATH, mode="w+", dtype=np.uint8, shape=(len(df), 167))
        X_DESC_W = np.lib.format.open_memmap(DESC_PATH, mode="w+", dtype=np.float32, shape=(len(df), len(DESC_SPECS)))
        start=0
    else:
        X_ECFP_W = np.lib.format.open_memmap(ECFP_PATH, mode="r+")
        X_MACCS_W = np.lib.format.open_memmap(MACCS_PATH, mode="r+")
        X_DESC_W = np.lib.format.open_memmap(DESC_PATH, mode="r+")
        start=json.loads(FEATURE_PROGRESS.read_text())['next_index'] if FEATURE_PROGRESS.exists() else 0
        print(f"🔄 Feature checkpoint পাওয়া গেছে। Molecule {start} থেকে resume হচ্ছে।")

    for i in tqdm(range(start, len(df)), desc="ECFP4 + MACCS + descriptors"):
        e,m,d = fingerprint_and_descriptors(df.iloc[i]["std_smiles"])
        X_ECFP_W[i]=e; X_MACCS_W[i]=m; X_DESC_W[i]=d
        if (i+1)%100==0 or i+1==len(df):
            X_ECFP_W.flush(); X_MACCS_W.flush(); X_DESC_W.flush()
            FEATURE_PROGRESS.write_text(json.dumps({"next_index":i+1}))
    FEATURE_PROGRESS.unlink(missing_ok=True)
    del X_ECFP_W, X_MACCS_W, X_DESC_W
    X_ECFP=np.load(ECFP_PATH,mmap_mode="r")
    X_MACCS=np.load(MACCS_PATH,mmap_mode="r")
    X_DESC=np.load(DESC_PATH,mmap_mode="r")
    print("✅ Feature generation সম্পন্ন এবং cache save হয়েছে।")
else:
    print("♻️ Cached molecular features load হয়েছে।")

print("📌 ECFP4 shape:", X_ECFP.shape, X_ECFP.dtype)
print("📌 MACCS shape:", X_MACCS.shape, X_MACCS.dtype)
print("📌 Descriptor shape:", X_DESC.shape, X_DESC.dtype)


In [ ]:
desc_preview = pd.DataFrame(np.asarray(X_DESC), columns=DESC_NAMES)
desc_summary = desc_preview.describe().T[["mean","std","min","max"]]
display(desc_summary.style.format("{:.3f}").set_caption("RDKit Descriptor Summary"))

plt.figure(figsize=(11,8))
sns.heatmap(desc_preview.corr(), cmap="vlag", center=0, square=True, linewidths=.4)
plt.title("Correlation among 13 Molecular Descriptors", fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR / "04_descriptor_correlation.png", dpi=220, bbox_inches="tight")
plt.show(); plt.close()
print("✅ Molecular descriptor summary এবং correlation visualization সম্পন্ন হয়েছে।")


## 8. Fold-safe feature transformers and evaluation utilities

Imputer, scaler and SVD are always fitted on the relevant training portion only. Test-fold information is never used to fit preprocessing.


In [ ]:
def prepare_tabular_fold(train_idx, eval_idx, mode="tree", compact=False):
    desc_cols = COMPACT_DESC_IDX if compact else list(range(len(DESC_NAMES)))
    desc_train = np.asarray(X_DESC[train_idx])[:, desc_cols]
    desc_eval = np.asarray(X_DESC[eval_idx])[:, desc_cols]
    imputer = SimpleImputer(strategy="median")
    desc_train = imputer.fit_transform(desc_train)
    desc_eval = imputer.transform(desc_eval)

    if mode in {"scaled", "svm", "dnn"}:
        scaler = StandardScaler()
        desc_train = scaler.fit_transform(desc_train)
        desc_eval = scaler.transform(desc_eval)
    else:
        scaler = None

    fp_train = np.asarray(X_ECFP[train_idx], dtype=np.float32)
    fp_eval = np.asarray(X_ECFP[eval_idx], dtype=np.float32)
    x_train = np.concatenate([fp_train, desc_train.astype(np.float32)], axis=1)
    x_eval = np.concatenate([fp_eval, desc_eval.astype(np.float32)], axis=1)

    svd = None
    post_scaler = None
    if mode == "svm":
        n_comp = min(RESOURCE["svd_components"], x_train.shape[0]-1, x_train.shape[1]-1)
        svd = TruncatedSVD(n_components=max(2,n_comp), random_state=SEED, n_iter=5)
        x_train = svd.fit_transform(x_train)
        x_eval = svd.transform(x_eval)
        post_scaler = StandardScaler()
        x_train = post_scaler.fit_transform(x_train)
        x_eval = post_scaler.transform(x_eval)

    artifacts={"imputer":imputer,"scaler":scaler,"svd":svd,"post_scaler":post_scaler,"compact":compact,"mode":mode}
    return x_train.astype(np.float32), x_eval.astype(np.float32), artifacts

def prepare_capsnet_fold(train_idx, eval_idx):
    m_train=np.asarray(X_MACCS[train_idx,1:],dtype=np.float32)
    m_eval=np.asarray(X_MACCS[eval_idx,1:],dtype=np.float32)
    d_train=np.asarray(X_DESC[train_idx],dtype=np.float32)
    d_eval=np.asarray(X_DESC[eval_idx],dtype=np.float32)
    imp=SimpleImputer(strategy="median")
    d_train=imp.fit_transform(d_train); d_eval=imp.transform(d_eval)
    scaler=MinMaxScaler()
    d_train=scaler.fit_transform(d_train); d_eval=scaler.transform(d_eval)
    x_train=np.concatenate([m_train,d_train],axis=1).astype(np.float32)
    x_eval=np.concatenate([m_eval,d_eval],axis=1).astype(np.float32)
    return x_train,x_eval,{"imputer":imp,"scaler":scaler}

print("✅ Fold-safe feature transformers প্রস্তুত হয়েছে।")


In [ ]:
def safe_auc(y_true, y_prob):
    y_true=np.asarray(y_true)
    y_prob=np.asarray(y_prob)
    finite=np.isfinite(y_prob)
    y_true=y_true[finite]
    y_prob=y_prob[finite]
    if len(y_true)==0 or np.unique(y_true).size<2:
        return np.nan
    return float(roc_auc_score(y_true,y_prob))

def metrics_from_predictions(indices, probs, fold, model_name):
    rows=[]
    for j,target in enumerate(TARGETS):
        observed=LABEL_MASK[indices,j].astype(bool)
        finite=np.isfinite(probs[:,j])
        valid=observed & finite
        yt=Y[indices,j][valid]
        yp=probs[valid,j]
        if len(yt)==0:
            auc=acc=np.nan
        else:
            auc=safe_auc(yt,yp)
            acc=float(accuracy_score(yt,(yp>=THRESHOLD).astype(int)))
        rows.append({"Model":model_name,"Fold":fold+1,"Endpoint":target,"AUC":auc,"Accuracy":acc,"N":int(valid.sum())})
    return pd.DataFrame(rows)

def summarize_metric_table(metric_df, model_name):
    per_task=metric_df.groupby("Endpoint",as_index=False).agg(
        AUC_Mean=("AUC","mean"), AUC_SD=("AUC","std"),
        Accuracy_Mean=("Accuracy","mean"), Accuracy_SD=("Accuracy","std"),
        Valid_Folds=("AUC","count")
    )
    fold_macro=metric_df.groupby("Fold",as_index=False).agg(Mean_AUC=("AUC","mean"),Mean_Accuracy=("Accuracy","mean"))
    best_auc_row=per_task.loc[per_task["AUC_Mean"].idxmax()] if per_task["AUC_Mean"].notna().any() else None
    best_acc_row=per_task.loc[per_task["Accuracy_Mean"].idxmax()] if per_task["Accuracy_Mean"].notna().any() else None
    overall={
        "Model":model_name,
        "Mean AUC":float(fold_macro["Mean_AUC"].mean()),
        "AUC SD":float(fold_macro["Mean_AUC"].std(ddof=1)) if len(fold_macro)>1 else 0.0,
        "Mean Accuracy":float(fold_macro["Mean_Accuracy"].mean()),
        "Accuracy SD":float(fold_macro["Mean_Accuracy"].std(ddof=1)) if len(fold_macro)>1 else 0.0,
        "Best Endpoint AUC":float(best_auc_row["AUC_Mean"]) if best_auc_row is not None else np.nan,
        "Best AUC Endpoint":str(best_auc_row["Endpoint"]) if best_auc_row is not None else "N/A",
        "Best Endpoint Accuracy":float(best_acc_row["Accuracy_Mean"]) if best_acc_row is not None else np.nan,
        "Best Accuracy Endpoint":str(best_acc_row["Endpoint"]) if best_acc_row is not None else "N/A",
        "Target AUC Gap":float(fold_macro["Mean_AUC"].mean()-TARGET_MEAN_AUC),
    }
    return per_task,fold_macro,overall

def model_slug(name):
    return re.sub(r"[^a-z0-9]+","_",name.lower()).strip("_")

def save_model_outputs(model_name, oof_probs, metric_df, histories=None):
    slug=model_slug(model_name)
    out_dir=RESULT_DIR/slug; out_dir.mkdir(parents=True,exist_ok=True)
    np.save(out_dir/"oof_probabilities.npy",oof_probs.astype(np.float32))
    metric_df.to_csv(out_dir/"fold_endpoint_metrics.csv",index=False)
    per_task,fold_macro,overall=summarize_metric_table(metric_df,model_name)
    per_task.to_csv(out_dir/"per_task_summary.csv",index=False)
    fold_macro.to_csv(out_dir/"fold_macro_summary.csv",index=False)
    with open(out_dir/"overall_summary.json","w",encoding="utf-8") as f:
        json.dump(overall,f,indent=2,ensure_ascii=False)
    if histories is not None:
        with open(out_dir/"training_histories.json","w",encoding="utf-8") as f:
            json.dump(histories,f,indent=2,ensure_ascii=False,default=float)
    return per_task,fold_macro,overall

def show_model_result(model_name, per_task, overall):
    print("\n"+"="*92)
    print(f"✅ {model_name} সম্পন্ন হয়েছে")
    print(f"Mean AUC              : {overall['Mean AUC']:.4f} ± {overall['AUC SD']:.4f}")
    print(f"Mean Accuracy         : {overall['Mean Accuracy']:.4f} ± {overall['Accuracy SD']:.4f}")
    print(f"Best Endpoint AUC     : {overall['Best AUC Endpoint']} = {overall['Best Endpoint AUC']:.4f}")
    print(f"Best Endpoint Accuracy: {overall['Best Accuracy Endpoint']} = {overall['Best Endpoint Accuracy']:.4f}")
    print("="*92)
    display(per_task.style.format({"AUC_Mean":"{:.4f}","AUC_SD":"{:.4f}","Accuracy_Mean":"{:.4f}","Accuracy_SD":"{:.4f}"})
            .background_gradient(subset=["AUC_Mean"],cmap="YlGn")
            .background_gradient(subset=["Accuracy_Mean"],cmap="Blues")
            .set_caption(f"{model_name} — 3-fold Per-endpoint Performance"))
    plot=per_task.set_index("Endpoint")[["AUC_Mean","Accuracy_Mean"]]
    ax=plot.plot(kind="bar",figsize=(14,5),color=["#2A9D8F","#F4A261"],ylim=(0,1.03))
    ax.axhline(TARGET_MEAN_AUC,color="#D62828",linestyle="--",linewidth=1.2,label="AUC target 0.90")
    ax.set_ylabel("Score"); ax.set_title(f"{model_name} — Per-endpoint AUC and Accuracy",fontweight="bold")
    ax.legend(loc="lower right"); plt.xticks(rotation=35,ha="right"); plt.tight_layout()
    plt.savefig(FIG_DIR/f"model_{model_slug(model_name)}_per_task.png",dpi=200,bbox_inches="tight")
    plt.show(); plt.close()

print("✅ Evaluation, result saving এবং formal display utilities প্রস্তুত হয়েছে।")


## 9. Kernel-restart recovery and cell-level checkpoints

এই notebook দুই স্তরে restart-safe:

1. **Model-level checkpoint:** classical model-এর task/fold এবং deep model-এর প্রতি epoch-এর state disk-এ save হয়। একই model cell পুনরায় run করলে completed work skip হয় এবং অসম্পূর্ণ fold/epoch থেকে resume হয়।
2. **Kernel-restart bootstrap:** প্রথমবার এই cell পর্যন্ত run হলে একটি lightweight `runtime_bootstrap.py` তৈরি হয়। Kernel restart-এর পর যেকোনো model cell সরাসরি run করলে cell নিজেই setup, curated data, frozen folds এবং cached features restore করে নেয়। পূর্ণ Python session pickle করা হয়নি, কারণ RDKit-এর কিছু C++ object নিরাপদভাবে pickle করা যায় না।

> Colab-এ Google Drive mount থাকলে checkpoint Drive-এ থাকবে। Local Jupyter-এ project folder-এর `final_ultra` directory-তে থাকবে।

In [ ]:
RUNTIME_BOOTSTRAP_SOURCE = '# Auto-generated by Final_Ultra.ipynb for kernel-restart recovery.\n# Do not edit manually. Re-run the Runtime Recovery cell to regenerate it.\n# ===== Recovered from notebook cell 3 =====\nimport sys, subprocess, importlib.util, platform, os\n\nPACKAGE_MAP = {\n    "numpy": "numpy",\n    "pandas": "pandas",\n    "sklearn": "scikit-learn",\n    "scipy": "scipy",\n    "matplotlib": "matplotlib",\n    "seaborn": "seaborn",\n    "joblib": "joblib",\n    "tqdm": "tqdm",\n    "psutil": "psutil",\n    "dill": "dill",\n    "rdkit": "rdkit",\n    "xgboost": "xgboost",\n    "torch": "torch",\n}\n\nprint("🔎 প্রয়োজনীয় Python package পরীক্ষা করা হচ্ছে...")\nmissing = [pip_name for import_name, pip_name in PACKAGE_MAP.items()\n           if importlib.util.find_spec(import_name) is None]\n\nif missing:\n    print("⚠️ নিচের package গুলো missing পাওয়া গেছে:")\n    for name in missing:\n        print("   •", name)\n    print("📦 Missing package গুলো install করা হচ্ছে...")\n    try:\n        subprocess.check_call([\n            sys.executable, "-m", "pip", "install", "-q",\n            "--upgrade-strategy", "only-if-needed", *missing\n        ])\n    except subprocess.CalledProcessError:\n        # Some older environments expose RDKit only as rdkit-pypi.\n        if "rdkit" in missing:\n            retry = ["rdkit-pypi" if x == "rdkit" else x for x in missing]\n            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *retry])\n        else:\n            raise\n    print("✅ প্রয়োজনীয় package installation সম্পন্ন হয়েছে।")\nelse:\n    print("✅ সব প্রয়োজনীয় package আগে থেকেই installed আছে।")\n\nprint("🐍 Python:", sys.version.split()[0])\nprint("💻 Platform:", platform.platform())\nprint("➡️ এখন পরের cell run করুন।")\n\n\n# ===== Recovered from notebook cell 5 =====\nimport os, gc, re, json, math, time, random, hashlib, warnings, shutil, sys\nfrom pathlib import Path\nfrom dataclasses import dataclass, asdict\nfrom typing import Dict, List, Tuple, Optional, Callable, Any\n\nwarnings.filterwarnings("ignore")\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport joblib, dill, psutil\nfrom tqdm.auto import tqdm\nfrom scipy.special import expit\nfrom scipy.optimize import minimize\n\nfrom IPython.display import display, Markdown\nfrom sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix\nfrom sklearn.preprocessing import StandardScaler, MinMaxScaler\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.decomposition import TruncatedSVD\nfrom sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier\nfrom sklearn.svm import SVC\nfrom sklearn.neural_network import BernoulliRBM\n\nfrom xgboost import XGBClassifier\n\nfrom rdkit import Chem, DataStructs, RDLogger, RDConfig\nfrom rdkit.Chem import AllChem, MACCSkeys, Descriptors, rdMolDescriptors, Draw, ChemicalFeatures\nfrom rdkit.Chem.MolStandardize import rdMolStandardize\nfrom rdkit.Chem.Scaffolds import MurckoScaffold\nRDLogger.DisableLog("rdApp.*")\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import Dataset, DataLoader\n\nsns.set_theme(style="whitegrid", context="notebook")\npd.set_option("display.max_columns", 100)\npd.set_option("display.max_rows", 100)\npd.set_option("display.width", 180)\n\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nAMP_ENABLED = DEVICE.type == "cuda"\n\nprint("✅ সব library সফলভাবে import হয়েছে।")\nif DEVICE.type == "cuda":\n    gpu_name = torch.cuda.get_device_name(0)\n    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)\n    print(f"🚀 GPU পাওয়া গেছে: {gpu_name} | VRAM ≈ {gpu_mem:.1f} GB")\n    print("✅ Deep Learning model GPU-তে চলবে এবং mixed precision সক্রিয় থাকবে।")\nelse:\n    print("ℹ️ GPU পাওয়া যায়নি। সব model CPU-তে চলবে।")\n    print("✅ Code পরিবর্তন করার প্রয়োজন নেই; notebook স্বয়ংক্রিয়ভাবে CPU mode ব্যবহার করছে।")\n\n\n# ===== Recovered from notebook cell 7 =====\nSEED = 42\nN_FOLDS = 3\nTARGET_MEAN_AUC = 0.90\nSPLIT_MODE = "scaffold_balanced"\nTHRESHOLD = 0.50\nSMOKE_TEST = bool(int(os.environ.get("FINAL_ULTRA_SMOKE", "0")))\nFORCE_REBUILD = False\nSAVE_FITTED_CLASSICAL_MODELS = False\nREFIT_DEEP_ON_FULL_OUTER_TRAIN = not SMOKE_TEST\n\n# Optional Google Drive mount. It is attempted only inside Colab.\nIN_COLAB = "google.colab" in sys.modules\nif IN_COLAB:\n    try:\n        from google.colab import drive\n        if not Path("/content/drive/MyDrive").exists():\n            print("🔗 Google Drive mount করা হচ্ছে, যাতে checkpoint বিদ্যুৎ/রানটাইম সমস্যার পরও থাকে...")\n            drive.mount("/content/drive")\n    except Exception as e:\n        print("ℹ️ Google Drive mount করা যায়নি বা প্রয়োজন হয়নি:", e)\n\ndef _valid_tox21_csv(path: Path) -> bool:\n    try:\n        cols = pd.read_csv(path, nrows=2).columns.tolist()\n        return "smiles" in cols and len([c for c in cols if c.startswith(("NR-", "SR-"))]) >= 10\n    except Exception:\n        return False\n\ndef find_tox21_dataset() -> Path:\n    roots = [\n        Path.cwd(), Path.cwd().parent,\n        Path("D:/Drug_Toxicity"),\n        Path("/content/drive/MyDrive/Drug_Toxicity"),\n        Path("/content"), Path("/mnt/data"),\n    ]\n    names = ["tox21.csv", "tox21(1).csv", "tox21(2).csv", "tox21 (1).csv", "tox21 (2).csv"]\n    candidates = []\n    for root in roots:\n        for name in names:\n            candidates += [root / "datasets" / name, root / name]\n    for path in candidates:\n        if path.exists() and _valid_tox21_csv(path):\n            return path.resolve()\n    for root in roots:\n        if root.exists():\n            for path in root.glob("**/*tox21*.csv"):\n                if not any(x in path.name.lower() for x in ["result", "audit", "clean", "prediction", "distribution"]):\n                    if _valid_tox21_csv(path):\n                        return path.resolve()\n    raise FileNotFoundError(\n        "tox21 CSV পাওয়া যায়নি। Colab-এ /content বা Google Drive-এর Drug_Toxicity/datasets folder-এ file রাখুন।"\n    )\n\nDATA_PATH = find_tox21_dataset()\nif DATA_PATH.parent.name.lower() == "datasets":\n    PROJECT_ROOT = DATA_PATH.parent.parent\nelif IN_COLAB and Path("/content/drive/MyDrive").exists():\n    PROJECT_ROOT = Path("/content/drive/MyDrive/Drug_Toxicity")\nelse:\n    PROJECT_ROOT = DATA_PATH.parent / "Final_Ultra_Project"\n\nWORK_ROOT = PROJECT_ROOT / "final_ultra"\nCACHE_DIR = WORK_ROOT / "cache"\nMODEL_DIR = WORK_ROOT / "models"\nRESULT_DIR = WORK_ROOT / "results"\nFIG_DIR = WORK_ROOT / "figures"\nLOG_DIR = WORK_ROOT / "logs"\nPRED_DIR = WORK_ROOT / "predictions"\nfor folder in [WORK_ROOT, CACHE_DIR, MODEL_DIR, RESULT_DIR, FIG_DIR, LOG_DIR, PRED_DIR]:\n    folder.mkdir(parents=True, exist_ok=True)\n\nRAM_GB = psutil.virtual_memory().total / (1024**3)\nCPU_COUNT = max(1, os.cpu_count() or 1)\nLOW_RESOURCE = RAM_GB < 11 and DEVICE.type == "cpu"\ntry:\n    torch.set_num_threads(2 if SMOKE_TEST else min(8, CPU_COUNT))\nexcept Exception:\n    pass\n\nRESOURCE = {\n    "tree_estimators": 8 if SMOKE_TEST else (450 if LOW_RESOURCE else 700),\n    "xgb_estimators": 3 if SMOKE_TEST else (450 if LOW_RESOURCE else 700),\n    "svd_components": 12 if SMOKE_TEST else (192 if LOW_RESOURCE else 320),\n    "dnn_epochs": 1 if SMOKE_TEST else (70 if LOW_RESOURCE else 140),\n    "caps_epochs": 1 if SMOKE_TEST else (60 if LOW_RESOURCE else 120),\n    "cnn_epochs": 1 if SMOKE_TEST else (50 if LOW_RESOURCE else 100),\n    "rgcn_epochs": 1 if SMOKE_TEST else (70 if LOW_RESOURCE else 140),\n    "patience": 1 if SMOKE_TEST else (12 if LOW_RESOURCE else 18),\n    "tab_batch": 64 if SMOKE_TEST else (128 if LOW_RESOURCE else (256 if DEVICE.type == "cuda" else 192)),\n    "cnn_batch": 16 if SMOKE_TEST else (24 if LOW_RESOURCE else (128 if DEVICE.type == "cuda" else 48)),\n    "graph_batch": 12 if SMOKE_TEST else (32 if LOW_RESOURCE else (128 if DEVICE.type == "cuda" else 64)),\n    "num_workers": 0 if os.name == "nt" or SMOKE_TEST else min(4, max(1, CPU_COUNT // 4)),\n}\n\nprint("✅ Project configuration তৈরি হয়েছে।")\nprint("📁 Dataset:", DATA_PATH)\nprint("📁 Project root:", PROJECT_ROOT)\nprint("📁 Restart-safe checkpoint root:", WORK_ROOT)\nprint(f"🧠 RAM ≈ {RAM_GB:.1f} GB | CPU threads = {CPU_COUNT} | Device = {DEVICE}")\nprint("🧪 Mode:", "SMOKE TEST" if SMOKE_TEST else "FULL RESEARCH RUN")\n\n\n# ===== Recovered from notebook cell 8 =====\ndef seed_everything(seed: int = SEED) -> None:\n    os.environ["PYTHONHASHSEED"] = str(seed)\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    try:\n        torch.use_deterministic_algorithms(False)\n    except Exception:\n        pass\n\nseed_everything()\n\nCONFIG = {\n    "seed": SEED,\n    "n_folds": N_FOLDS,\n    "split_mode": SPLIT_MODE,\n    "threshold": THRESHOLD,\n    "target_mean_auc": TARGET_MEAN_AUC,\n    "device": str(DEVICE),\n    "smoke_test": SMOKE_TEST,\n    "resource": RESOURCE,\n    "dataset_path": str(DATA_PATH),\n    "project_root": str(PROJECT_ROOT),\n    "models": [\n        "Random Forest", "Extra Trees", "XGBoost", "SVM-RBF",\n        "DeepTox-style MT-DNN", "Multitask CapsNet + RBM",\n        "Multi-channel 2D CNN", "Compact MTL-DNN",\n        "Molecular R-GCN Fusion", "Soft-Voting Ensemble"\n    ],\n    "excluded_models": ["Elastic Net", "GPS"],\n}\nwith open(WORK_ROOT / "final_ultra_config.json", "w", encoding="utf-8") as f:\n    json.dump(CONFIG, f, indent=2, ensure_ascii=False)\n\nprint("✅ Random seed freeze করা হয়েছে এবং configuration JSON save হয়েছে।")\nprint("🚫 Excluded models: Elastic Net, GPS")\n\n\n# ===== Recovered from notebook cell 10 =====\nRAW_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()\ndf_raw = pd.read_csv(DATA_PATH)\n\nEXPECTED_TARGETS = [\n    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase", "NR-ER", "NR-ER-LBD",\n    "NR-PPAR-gamma", "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53"\n]\nmissing_cols = [c for c in EXPECTED_TARGETS + ["smiles"] if c not in df_raw.columns]\nif missing_cols:\n    raise ValueError(f"Dataset schema ভুল। Missing columns: {missing_cols}")\n\nif "mol_id" not in df_raw.columns:\n    df_raw["mol_id"] = [f"MOL_{i:06d}" for i in range(len(df_raw))]\n\nfor target in EXPECTED_TARGETS:\n    df_raw[target] = pd.to_numeric(df_raw[target], errors="coerce")\n    invalid_values = set(df_raw[target].dropna().unique()) - {0.0, 1.0}\n    if invalid_values:\n        raise ValueError(f"{target} column-এ 0/1 ছাড়া value পাওয়া গেছে: {invalid_values}")\n\nTARGETS = EXPECTED_TARGETS.copy()\nif SMOKE_TEST:\n    # Keep enough endpoints to exercise masking and multi-task logic.\n    TARGETS = EXPECTED_TARGETS[:4]\n    df_raw = df_raw.sample(n=min(240, len(df_raw)), random_state=SEED).reset_index(drop=True)\n    print("🧪 Smoke test-এর জন্য 240 row এবং প্রথম 4 endpoint ব্যবহার করা হচ্ছে।")\n\nprint("✅ Tox21 dataset সফলভাবে load ও validate হয়েছে।")\nprint("📌 Shape:", df_raw.shape)\nprint("📌 Active target count:", len(TARGETS))\nprint("📌 SHA256:", RAW_SHA256[:24] + "...")\ndisplay(df_raw[["mol_id", "smiles"] + TARGETS].head(3))\n\n\n# ===== Recovered from notebook cell 16 =====\nTAUTOMER_ENUMERATOR = rdMolStandardize.TautomerEnumerator()\nUNCHARGER = rdMolStandardize.Uncharger()\ntry:\n    FRAGMENT_CHOOSER = rdMolStandardize.LargestFragmentChooser(preferOrganic=True)\nexcept TypeError:\n    FRAGMENT_CHOOSER = rdMolStandardize.LargestFragmentChooser()\n\ndef standardize_molecule(smiles: str) -> Optional[Dict[str, str]]:\n    try:\n        mol = Chem.MolFromSmiles(str(smiles), sanitize=True)\n        if mol is None:\n            return None\n        mol = rdMolStandardize.Cleanup(mol)\n        mol = FRAGMENT_CHOOSER.choose(mol)\n        mol = UNCHARGER.uncharge(mol)\n        mol = TAUTOMER_ENUMERATOR.Canonicalize(mol)\n        Chem.SanitizeMol(mol)\n        std_smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)\n        inchikey = Chem.MolToInchiKey(mol)\n        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=True)\n        if not scaffold:\n            scaffold = "ACYCLIC_" + hashlib.sha1(std_smiles.encode()).hexdigest()[:16]\n        return {\n            "std_smiles": std_smiles,\n            "inchikey": inchikey,\n            "scaffold": scaffold,\n            "has_mixture": int("." in str(smiles)),\n        }\n    except Exception:\n        return None\n\nprint("✅ Chemical standardization function প্রস্তুত হয়েছে।")\n\n\n# ===== Recovered from notebook cell 17 =====\nCURATED_PATH = CACHE_DIR / f"curated_{RAW_SHA256[:10]}_{len(TARGETS)}t.csv"\nINVALID_PATH = RESULT_DIR / "02_invalid_smiles.csv"\nPROGRESS_PATH = CACHE_DIR / "curation_progress.json"\nPARTIAL_PATH = CACHE_DIR / "curation_partial.jsonl"\n\nif CURATED_PATH.exists() and not FORCE_REBUILD:\n    df_std = pd.read_csv(CURATED_PATH)\n    print("♻️ Cached standardized dataset load হয়েছে।")\nelse:\n    start = 0\n    records = []\n    invalid_records = []\n    if PARTIAL_PATH.exists() and PROGRESS_PATH.exists() and not FORCE_REBUILD:\n        with open(PARTIAL_PATH, "r", encoding="utf-8") as f:\n            records = [json.loads(line) for line in f if line.strip()]\n        state = json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))\n        start = int(state.get("next_index", len(records)))\n        print(f"🔄 আগের chemical preprocessing checkpoint পাওয়া গেছে। Row {start} থেকে resume হচ্ছে।")\n    else:\n        PARTIAL_PATH.unlink(missing_ok=True)\n        PROGRESS_PATH.unlink(missing_ok=True)\n\n    with open(PARTIAL_PATH, "a", encoding="utf-8") as writer:\n        for i in tqdm(range(start, len(df_raw)), desc="Chemical standardization"):\n            row = df_raw.iloc[i]\n            out = standardize_molecule(row["smiles"])\n            if out is None:\n                invalid_records.append(row.to_dict())\n            else:\n                rec = row.to_dict()\n                rec["raw_smiles"] = rec.pop("smiles")\n                rec.update(out)\n                writer.write(json.dumps(rec, ensure_ascii=False, default=str) + "\\n")\n                records.append(rec)\n            if (i + 1) % 100 == 0 or i + 1 == len(df_raw):\n                writer.flush()\n                PROGRESS_PATH.write_text(json.dumps({"next_index": i + 1}), encoding="utf-8")\n\n    df_std = pd.DataFrame(records)\n    df_std.to_csv(CURATED_PATH, index=False)\n    if invalid_records:\n        pd.DataFrame(invalid_records).to_csv(INVALID_PATH, index=False)\n    else:\n        pd.DataFrame(columns=df_raw.columns).to_csv(INVALID_PATH, index=False)\n    PARTIAL_PATH.unlink(missing_ok=True)\n    PROGRESS_PATH.unlink(missing_ok=True)\n    print("✅ Chemical preprocessing সম্পন্ন এবং cache save হয়েছে।")\n\nprint("📌 Valid standardized molecules:", len(df_std))\nprint("📌 Invalid molecules removed:", len(df_raw) - len(df_std))\nprint("📌 Dot-separated mixtures/salts in raw input:", int(df_std["has_mixture"].sum()))\n\n\n# ===== Recovered from notebook cell 18 =====\ndef resolve_duplicate_group(group: pd.DataFrame) -> pd.Series:\n    out = group.iloc[0].copy()\n    out["source_mol_ids"] = "|".join(group["mol_id"].astype(str).tolist())\n    out["duplicate_count"] = len(group)\n    for target in TARGETS:\n        values = sorted(group[target].dropna().unique().tolist())\n        if len(values) == 1:\n            out[target] = values[0]\n        elif len(values) == 0:\n            out[target] = np.nan\n        else:\n            # Conflicting duplicate measurements are unknown, not forced to 0/1.\n            out[target] = np.nan\n    return out\n\nbefore_dup = len(df_std)\ndf = df_std.groupby("inchikey", sort=False, group_keys=False).apply(resolve_duplicate_group).reset_index(drop=True)\nconflict_count = int(df[TARGETS].isna().sum().sum() - df_std[TARGETS].isna().sum().sum())\n\nY = df[TARGETS].to_numpy(dtype=np.float32)\nLABEL_MASK = (~np.isnan(Y)).astype(np.float32)\nY_TENSOR_SAFE = np.nan_to_num(Y, nan=0.0).astype(np.float32)\n\nprint("✅ Duplicate resolution সম্পন্ন হয়েছে।")\nprint("📌 Rows before duplicate merge:", before_dup)\nprint("📌 Rows after duplicate merge:", len(df))\nprint("📌 Conflicting endpoint labels converted to missing:", max(0, conflict_count))\nprint("🔒 Target NaN tensor storage-এর জন্য temporary 0 placeholder পেয়েছে, কিন্তু mask=0 হওয়ায় loss/metric-এ কখনও label হিসেবে গণনা হবে না।")\nprint("📌 Observed labels:", int(LABEL_MASK.sum()), "| Missing labels:", int((1-LABEL_MASK).sum()))\n\ndf.to_csv(RESULT_DIR / "03_curated_molecule_mapping.csv", index=False)\n\n\n# ===== Recovered from notebook cell 21 =====\ndef make_scaffold_balanced_folds(frame: pd.DataFrame, targets: List[str], n_folds: int, seed: int) -> np.ndarray:\n    rng = np.random.default_rng(seed)\n    groups = []\n    for scaffold, part in frame.groupby("scaffold", sort=False):\n        arr = part[targets].to_numpy(float)\n        groups.append({\n            "scaffold": scaffold,\n            "indices": part.index.to_numpy(),\n            "n": len(part),\n            "pos": np.nansum(arr == 1, axis=0).astype(float),\n            "labeled": np.sum(~np.isnan(arr), axis=0).astype(float),\n        })\n    # Large and rare-positive-rich scaffolds are placed first.\n    rng.shuffle(groups)\n    groups.sort(key=lambda g: (g["n"] + 3.0 * np.sqrt(g["pos"].sum() + 1)), reverse=True)\n\n    total_n = len(frame)\n    total_pos = np.sum([g["pos"] for g in groups], axis=0)\n    total_lab = np.sum([g["labeled"] for g in groups], axis=0)\n    target_n = total_n / n_folds\n    target_pos = total_pos / n_folds\n    target_lab = total_lab / n_folds\n\n    fold_n = np.zeros(n_folds, dtype=float)\n    fold_pos = np.zeros((n_folds, len(targets)), dtype=float)\n    fold_lab = np.zeros((n_folds, len(targets)), dtype=float)\n    assignment = {}\n\n    def objective(n_arr, pos_arr, lab_arr):\n        size_err = np.mean(((n_arr - target_n) / max(target_n, 1)) ** 2)\n        pos_err = np.mean(((pos_arr - target_pos) / (target_pos + 1.0)) ** 2)\n        lab_err = np.mean(((lab_arr - target_lab) / (target_lab + 1.0)) ** 2)\n        return 0.25 * size_err + 0.55 * pos_err + 0.20 * lab_err\n\n    for g in groups:\n        scores = []\n        for fold in range(n_folds):\n            n2, p2, l2 = fold_n.copy(), fold_pos.copy(), fold_lab.copy()\n            n2[fold] += g["n"]\n            p2[fold] += g["pos"]\n            l2[fold] += g["labeled"]\n            scores.append(objective(n2, p2, l2))\n        min_score = min(scores)\n        ties = [i for i, s in enumerate(scores) if abs(s - min_score) < 1e-12]\n        chosen = int(rng.choice(ties))\n        assignment[g["scaffold"]] = chosen\n        fold_n[chosen] += g["n"]\n        fold_pos[chosen] += g["pos"]\n        fold_lab[chosen] += g["labeled"]\n\n    return frame["scaffold"].map(assignment).to_numpy(dtype=int)\n\nprint("✅ Scaffold-balanced fold generator প্রস্তুত হয়েছে।")\n\n\n# ===== Recovered from notebook cell 22 =====\nFOLD_PATH = CACHE_DIR / f"folds_{RAW_SHA256[:10]}_{len(df)}_{len(TARGETS)}t_{N_FOLDS}f.csv"\nif FOLD_PATH.exists() and not FORCE_REBUILD:\n    fold_file = pd.read_csv(FOLD_PATH)\n    if len(fold_file) != len(df):\n        raise ValueError("Cached fold file row count বর্তমান curated dataset-এর সঙ্গে মিলছে না।")\n    df["fold"] = fold_file["fold"].astype(int).to_numpy()\n    print("♻️ Frozen fold assignment cache থেকে load হয়েছে।")\nelse:\n    df["fold"] = make_scaffold_balanced_folds(df, TARGETS, N_FOLDS, SEED)\n    df[["mol_id", "inchikey", "scaffold", "fold"]].to_csv(FOLD_PATH, index=False)\n    print("✅ নতুন 3-fold scaffold-balanced assignment তৈরি ও freeze করা হয়েছে।")\n\n# Leakage audit\nscaffold_fold_count = df.groupby("scaffold")["fold"].nunique()\nif scaffold_fold_count.max() != 1:\n    raise AssertionError("Scaffold leakage পাওয়া গেছে।")\n\nfold_rows=[]\nfor fold in range(N_FOLDS):\n    part=df[df["fold"]==fold]\n    row={"Fold":fold+1,"Molecules":len(part),"Unique scaffolds":part["scaffold"].nunique()}\n    for t in TARGETS:\n        row[f"{t} positive"] = int((part[t]==1).sum())\n    fold_rows.append(row)\nfold_quality_df=pd.DataFrame(fold_rows)\nfold_quality_df.to_csv(RESULT_DIR / "04_fold_quality.csv", index=False)\n\nprint("✅ Scaffold leakage audit passed: কোনো scaffold একাধিক fold-এ নেই।")\ndisplay(fold_quality_df.style.set_caption("Frozen Outer-fold Quality Audit").background_gradient(cmap="Blues"))\n\n\n# ===== Recovered from notebook cell 25 =====\nDESC_SPECS = [\n    ("MolWt", Descriptors.MolWt),\n    ("MolLogP", Descriptors.MolLogP),\n    ("MolMR", Descriptors.MolMR),\n    ("TPSA", rdMolDescriptors.CalcTPSA),\n    ("HBD", rdMolDescriptors.CalcNumHBD),\n    ("HBA", rdMolDescriptors.CalcNumHBA),\n    ("RotatableBonds", rdMolDescriptors.CalcNumRotatableBonds),\n    ("RingCount", rdMolDescriptors.CalcNumRings),\n    ("AromaticRingCount", rdMolDescriptors.CalcNumAromaticRings),\n    ("NOCount", Descriptors.NOCount),\n    ("FractionCSP3", rdMolDescriptors.CalcFractionCSP3),\n    ("HeavyAtomCount", Descriptors.HeavyAtomCount),\n    ("LabuteASA", rdMolDescriptors.CalcLabuteASA),\n]\nDESC_NAMES = [name for name, _ in DESC_SPECS]\nCOMPACT_DESC_NAMES = ["MolWt", "MolLogP", "TPSA", "HBD", "HBA", "RotatableBonds", "RingCount"]\nCOMPACT_DESC_IDX = [DESC_NAMES.index(x) for x in COMPACT_DESC_NAMES]\n\ntry:\n    MORGAN_GENERATOR = AllChem.GetMorganGenerator(radius=2, fpSize=2048)\nexcept Exception:\n    MORGAN_GENERATOR = None\n\ndef fingerprint_and_descriptors(smiles: str):\n    mol = Chem.MolFromSmiles(smiles)\n    if mol is None:\n        raise ValueError("Standardized SMILES থেকে molecule তৈরি হয়নি।")\n    ecfp = np.zeros(2048, dtype=np.uint8)\n    fp = MORGAN_GENERATOR.GetFingerprint(mol) if MORGAN_GENERATOR is not None else AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)\n    DataStructs.ConvertToNumpyArray(fp, ecfp)\n    maccs = np.zeros(167, dtype=np.uint8)\n    DataStructs.ConvertToNumpyArray(MACCSkeys.GenMACCSKeys(mol), maccs)\n    desc=[]\n    for _, fn in DESC_SPECS:\n        try:\n            value=float(fn(mol))\n            desc.append(value if np.isfinite(value) else np.nan)\n        except Exception:\n            desc.append(np.nan)\n    return ecfp, maccs, np.asarray(desc, dtype=np.float32)\n\nprint("✅ Feature functions প্রস্তুত হয়েছে।")\n\n\n# ===== Recovered from notebook cell 26 =====\nFEATURE_ROOT = CACHE_DIR / f"features_{RAW_SHA256[:10]}_{len(df)}"\nFEATURE_ROOT.mkdir(parents=True, exist_ok=True)\nECFP_PATH = FEATURE_ROOT / "ecfp4_2048.npy"\nMACCS_PATH = FEATURE_ROOT / "maccs_167.npy"\nDESC_PATH = FEATURE_ROOT / "rdkit_desc_13.npy"\nFEATURE_PROGRESS = FEATURE_ROOT / "progress.json"\n\nshape_ok = ECFP_PATH.exists() and MACCS_PATH.exists() and DESC_PATH.exists()\nif shape_ok and not FORCE_REBUILD:\n    X_ECFP = np.load(ECFP_PATH, mmap_mode="r")\n    X_MACCS = np.load(MACCS_PATH, mmap_mode="r")\n    X_DESC = np.load(DESC_PATH, mmap_mode="r")\n    if X_ECFP.shape[0] != len(df):\n        shape_ok=False\n\nif not shape_ok or FORCE_REBUILD:\n    if FORCE_REBUILD:\n        for p in [ECFP_PATH, MACCS_PATH, DESC_PATH, FEATURE_PROGRESS]:\n            p.unlink(missing_ok=True)\n    if not ECFP_PATH.exists():\n        X_ECFP_W = np.lib.format.open_memmap(ECFP_PATH, mode="w+", dtype=np.uint8, shape=(len(df), 2048))\n        X_MACCS_W = np.lib.format.open_memmap(MACCS_PATH, mode="w+", dtype=np.uint8, shape=(len(df), 167))\n        X_DESC_W = np.lib.format.open_memmap(DESC_PATH, mode="w+", dtype=np.float32, shape=(len(df), len(DESC_SPECS)))\n        start=0\n    else:\n        X_ECFP_W = np.lib.format.open_memmap(ECFP_PATH, mode="r+")\n        X_MACCS_W = np.lib.format.open_memmap(MACCS_PATH, mode="r+")\n        X_DESC_W = np.lib.format.open_memmap(DESC_PATH, mode="r+")\n        start=json.loads(FEATURE_PROGRESS.read_text())[\'next_index\'] if FEATURE_PROGRESS.exists() else 0\n        print(f"🔄 Feature checkpoint পাওয়া গেছে। Molecule {start} থেকে resume হচ্ছে।")\n\n    for i in tqdm(range(start, len(df)), desc="ECFP4 + MACCS + descriptors"):\n        e,m,d = fingerprint_and_descriptors(df.iloc[i]["std_smiles"])\n        X_ECFP_W[i]=e; X_MACCS_W[i]=m; X_DESC_W[i]=d\n        if (i+1)%100==0 or i+1==len(df):\n            X_ECFP_W.flush(); X_MACCS_W.flush(); X_DESC_W.flush()\n            FEATURE_PROGRESS.write_text(json.dumps({"next_index":i+1}))\n    FEATURE_PROGRESS.unlink(missing_ok=True)\n    del X_ECFP_W, X_MACCS_W, X_DESC_W\n    X_ECFP=np.load(ECFP_PATH,mmap_mode="r")\n    X_MACCS=np.load(MACCS_PATH,mmap_mode="r")\n    X_DESC=np.load(DESC_PATH,mmap_mode="r")\n    print("✅ Feature generation সম্পন্ন এবং cache save হয়েছে।")\nelse:\n    print("♻️ Cached molecular features load হয়েছে।")\n\nprint("📌 ECFP4 shape:", X_ECFP.shape, X_ECFP.dtype)\nprint("📌 MACCS shape:", X_MACCS.shape, X_MACCS.dtype)\nprint("📌 Descriptor shape:", X_DESC.shape, X_DESC.dtype)\n\n\n# ===== Recovered from notebook cell 29 =====\ndef prepare_tabular_fold(train_idx, eval_idx, mode="tree", compact=False):\n    desc_cols = COMPACT_DESC_IDX if compact else list(range(len(DESC_NAMES)))\n    desc_train = np.asarray(X_DESC[train_idx])[:, desc_cols]\n    desc_eval = np.asarray(X_DESC[eval_idx])[:, desc_cols]\n    imputer = SimpleImputer(strategy="median")\n    desc_train = imputer.fit_transform(desc_train)\n    desc_eval = imputer.transform(desc_eval)\n\n    if mode in {"scaled", "svm", "dnn"}:\n        scaler = StandardScaler()\n        desc_train = scaler.fit_transform(desc_train)\n        desc_eval = scaler.transform(desc_eval)\n    else:\n        scaler = None\n\n    fp_train = np.asarray(X_ECFP[train_idx], dtype=np.float32)\n    fp_eval = np.asarray(X_ECFP[eval_idx], dtype=np.float32)\n    x_train = np.concatenate([fp_train, desc_train.astype(np.float32)], axis=1)\n    x_eval = np.concatenate([fp_eval, desc_eval.astype(np.float32)], axis=1)\n\n    svd = None\n    post_scaler = None\n    if mode == "svm":\n        n_comp = min(RESOURCE["svd_components"], x_train.shape[0]-1, x_train.shape[1]-1)\n        svd = TruncatedSVD(n_components=max(2,n_comp), random_state=SEED, n_iter=5)\n        x_train = svd.fit_transform(x_train)\n        x_eval = svd.transform(x_eval)\n        post_scaler = StandardScaler()\n        x_train = post_scaler.fit_transform(x_train)\n        x_eval = post_scaler.transform(x_eval)\n\n    artifacts={"imputer":imputer,"scaler":scaler,"svd":svd,"post_scaler":post_scaler,"compact":compact,"mode":mode}\n    return x_train.astype(np.float32), x_eval.astype(np.float32), artifacts\n\ndef prepare_capsnet_fold(train_idx, eval_idx):\n    m_train=np.asarray(X_MACCS[train_idx,1:],dtype=np.float32)\n    m_eval=np.asarray(X_MACCS[eval_idx,1:],dtype=np.float32)\n    d_train=np.asarray(X_DESC[train_idx],dtype=np.float32)\n    d_eval=np.asarray(X_DESC[eval_idx],dtype=np.float32)\n    imp=SimpleImputer(strategy="median")\n    d_train=imp.fit_transform(d_train); d_eval=imp.transform(d_eval)\n    scaler=MinMaxScaler()\n    d_train=scaler.fit_transform(d_train); d_eval=scaler.transform(d_eval)\n    x_train=np.concatenate([m_train,d_train],axis=1).astype(np.float32)\n    x_eval=np.concatenate([m_eval,d_eval],axis=1).astype(np.float32)\n    return x_train,x_eval,{"imputer":imp,"scaler":scaler}\n\nprint("✅ Fold-safe feature transformers প্রস্তুত হয়েছে।")\n\n\n# ===== Recovered from notebook cell 30 =====\ndef safe_auc(y_true, y_prob):\n    y_true=np.asarray(y_true)\n    y_prob=np.asarray(y_prob)\n    finite=np.isfinite(y_prob)\n    y_true=y_true[finite]\n    y_prob=y_prob[finite]\n    if len(y_true)==0 or np.unique(y_true).size<2:\n        return np.nan\n    return float(roc_auc_score(y_true,y_prob))\n\ndef metrics_from_predictions(indices, probs, fold, model_name):\n    rows=[]\n    for j,target in enumerate(TARGETS):\n        observed=LABEL_MASK[indices,j].astype(bool)\n        finite=np.isfinite(probs[:,j])\n        valid=observed & finite\n        yt=Y[indices,j][valid]\n        yp=probs[valid,j]\n        if len(yt)==0:\n            auc=acc=np.nan\n        else:\n            auc=safe_auc(yt,yp)\n            acc=float(accuracy_score(yt,(yp>=THRESHOLD).astype(int)))\n        rows.append({"Model":model_name,"Fold":fold+1,"Endpoint":target,"AUC":auc,"Accuracy":acc,"N":int(valid.sum())})\n    return pd.DataFrame(rows)\n\ndef summarize_metric_table(metric_df, model_name):\n    per_task=metric_df.groupby("Endpoint",as_index=False).agg(\n        AUC_Mean=("AUC","mean"), AUC_SD=("AUC","std"),\n        Accuracy_Mean=("Accuracy","mean"), Accuracy_SD=("Accuracy","std"),\n        Valid_Folds=("AUC","count")\n    )\n    fold_macro=metric_df.groupby("Fold",as_index=False).agg(Mean_AUC=("AUC","mean"),Mean_Accuracy=("Accuracy","mean"))\n    best_auc_row=per_task.loc[per_task["AUC_Mean"].idxmax()] if per_task["AUC_Mean"].notna().any() else None\n    best_acc_row=per_task.loc[per_task["Accuracy_Mean"].idxmax()] if per_task["Accuracy_Mean"].notna().any() else None\n    overall={\n        "Model":model_name,\n        "Mean AUC":float(fold_macro["Mean_AUC"].mean()),\n        "AUC SD":float(fold_macro["Mean_AUC"].std(ddof=1)) if len(fold_macro)>1 else 0.0,\n        "Mean Accuracy":float(fold_macro["Mean_Accuracy"].mean()),\n        "Accuracy SD":float(fold_macro["Mean_Accuracy"].std(ddof=1)) if len(fold_macro)>1 else 0.0,\n        "Best Endpoint AUC":float(best_auc_row["AUC_Mean"]) if best_auc_row is not None else np.nan,\n        "Best AUC Endpoint":str(best_auc_row["Endpoint"]) if best_auc_row is not None else "N/A",\n        "Best Endpoint Accuracy":float(best_acc_row["Accuracy_Mean"]) if best_acc_row is not None else np.nan,\n        "Best Accuracy Endpoint":str(best_acc_row["Endpoint"]) if best_acc_row is not None else "N/A",\n        "Target AUC Gap":float(fold_macro["Mean_AUC"].mean()-TARGET_MEAN_AUC),\n    }\n    return per_task,fold_macro,overall\n\ndef model_slug(name):\n    return re.sub(r"[^a-z0-9]+","_",name.lower()).strip("_")\n\ndef save_model_outputs(model_name, oof_probs, metric_df, histories=None):\n    slug=model_slug(model_name)\n    out_dir=RESULT_DIR/slug; out_dir.mkdir(parents=True,exist_ok=True)\n    np.save(out_dir/"oof_probabilities.npy",oof_probs.astype(np.float32))\n    metric_df.to_csv(out_dir/"fold_endpoint_metrics.csv",index=False)\n    per_task,fold_macro,overall=summarize_metric_table(metric_df,model_name)\n    per_task.to_csv(out_dir/"per_task_summary.csv",index=False)\n    fold_macro.to_csv(out_dir/"fold_macro_summary.csv",index=False)\n    with open(out_dir/"overall_summary.json","w",encoding="utf-8") as f:\n        json.dump(overall,f,indent=2,ensure_ascii=False)\n    if histories is not None:\n        with open(out_dir/"training_histories.json","w",encoding="utf-8") as f:\n            json.dump(histories,f,indent=2,ensure_ascii=False,default=float)\n    return per_task,fold_macro,overall\n\ndef show_model_result(model_name, per_task, overall):\n    print("\\n"+"="*92)\n    print(f"✅ {model_name} সম্পন্ন হয়েছে")\n    print(f"Mean AUC              : {overall[\'Mean AUC\']:.4f} ± {overall[\'AUC SD\']:.4f}")\n    print(f"Mean Accuracy         : {overall[\'Mean Accuracy\']:.4f} ± {overall[\'Accuracy SD\']:.4f}")\n    print(f"Best Endpoint AUC     : {overall[\'Best AUC Endpoint\']} = {overall[\'Best Endpoint AUC\']:.4f}")\n    print(f"Best Endpoint Accuracy: {overall[\'Best Accuracy Endpoint\']} = {overall[\'Best Endpoint Accuracy\']:.4f}")\n    print("="*92)\n    display(per_task.style.format({"AUC_Mean":"{:.4f}","AUC_SD":"{:.4f}","Accuracy_Mean":"{:.4f}","Accuracy_SD":"{:.4f}"})\n            .background_gradient(subset=["AUC_Mean"],cmap="YlGn")\n            .background_gradient(subset=["Accuracy_Mean"],cmap="Blues")\n            .set_caption(f"{model_name} — 3-fold Per-endpoint Performance"))\n    plot=per_task.set_index("Endpoint")[["AUC_Mean","Accuracy_Mean"]]\n    ax=plot.plot(kind="bar",figsize=(14,5),color=["#2A9D8F","#F4A261"],ylim=(0,1.03))\n    ax.axhline(TARGET_MEAN_AUC,color="#D62828",linestyle="--",linewidth=1.2,label="AUC target 0.90")\n    ax.set_ylabel("Score"); ax.set_title(f"{model_name} — Per-endpoint AUC and Accuracy",fontweight="bold")\n    ax.legend(loc="lower right"); plt.xticks(rotation=35,ha="right"); plt.tight_layout()\n    plt.savefig(FIG_DIR/f"model_{model_slug(model_name)}_per_task.png",dpi=200,bbox_inches="tight")\n    plt.show(); plt.close()\n\nprint("✅ Evaluation, result saving এবং formal display utilities প্রস্তুত হয়েছে।")\n\n\n# ===== Recovered from notebook cell 34 =====\ndef _predict_positive(model, X):\n    if hasattr(model,"predict_proba"):\n        return np.asarray(model.predict_proba(X))[:,1]\n    if hasattr(model,"decision_function"):\n        return expit(np.asarray(model.decision_function(X)))\n    return np.asarray(model.predict(X),dtype=float)\n\ndef build_classical_model(kind, y_train):\n    if kind=="rf":\n        return RandomForestClassifier(\n            n_estimators=RESOURCE["tree_estimators"], max_features="sqrt", min_samples_leaf=1,\n            class_weight="balanced_subsample", bootstrap=True, n_jobs=-1, random_state=SEED\n        )\n    if kind=="et":\n        return ExtraTreesClassifier(\n            n_estimators=RESOURCE["tree_estimators"], max_features="sqrt", min_samples_leaf=1,\n            class_weight="balanced", bootstrap=False, n_jobs=-1, random_state=SEED\n        )\n    if kind=="xgb":\n        pos=max(1,int(np.sum(y_train==1))); neg=max(1,int(np.sum(y_train==0)))\n        params=dict(\n            n_estimators=RESOURCE["xgb_estimators"], max_depth=(3 if SMOKE_TEST else 5), learning_rate=(0.15 if SMOKE_TEST else 0.035),\n            subsample=0.85, colsample_bytree=0.70, min_child_weight=2,\n            reg_alpha=0.05, reg_lambda=2.0, objective="binary:logistic", eval_metric="auc",\n            tree_method="hist", n_jobs=(2 if SMOKE_TEST else min(8, max(1, CPU_COUNT-1))), random_state=SEED,\n            scale_pos_weight=neg/pos\n        )\n        if DEVICE.type=="cuda": params["device"]="cuda"\n        return XGBClassifier(**params)\n    if kind=="svm":\n        return SVC(\n            kernel="rbf", C=10.0, gamma="scale", class_weight="balanced",\n            probability=False, cache_size=384 if LOW_RESOURCE else 1536, random_state=SEED\n        )\n    raise ValueError(kind)\n\ndef run_classical_cv(model_name, kind, feature_mode):\n    slug=model_slug(model_name)\n    unit_dir=PRED_DIR/slug; unit_dir.mkdir(parents=True,exist_ok=True)\n    oof=np.full((len(df),len(TARGETS)),np.nan,dtype=np.float32)\n    metric_parts=[]\n\n    for fold in range(N_FOLDS):\n        train_idx=np.where(df["fold"].to_numpy()!=fold)[0]\n        test_idx=np.where(df["fold"].to_numpy()==fold)[0]\n        print(f"\\n📘 {model_name} | Fold {fold+1}/{N_FOLDS} শুরু")\n        Xtr_all,Xte_all,artifacts=prepare_tabular_fold(train_idx,test_idx,mode=feature_mode,compact=False)\n        joblib.dump(artifacts, MODEL_DIR/f"{slug}_fold{fold+1}_preprocess.joblib", compress=3)\n\n        fold_probs=np.full((len(test_idx),len(TARGETS)),np.nan,dtype=np.float32)\n        for j,target in enumerate(TARGETS):\n            unit_path=unit_dir/f"fold{fold+1}_{model_slug(target)}.npz"\n            if unit_path.exists() and not FORCE_REBUILD:\n                saved=np.load(unit_path)\n                fold_probs[:,j]=saved["prob"]\n                print(f"   ♻️ {target}: completed checkpoint load")\n                continue\n            tr_valid=~np.isnan(Y[train_idx,j]); te_valid=~np.isnan(Y[test_idx,j])\n            ytr=Y[train_idx,j][tr_valid].astype(int)\n            if np.unique(ytr).size<2:\n                print(f"   ⚠️ {target}: training fold-এ দুই class নেই; skip")\n                np.savez_compressed(unit_path,prob=fold_probs[:,j])\n                continue\n            model=build_classical_model(kind,ytr)\n            try:\n                model.fit(Xtr_all[tr_valid],ytr)\n            except Exception as e:\n                # XGBoost GPU compatibility fallback.\n                if kind=="xgb" and "cuda" in str(e).lower():\n                    print("   ⚠️ XGBoost GPU mode কাজ করেনি; CPU hist mode-এ retry হচ্ছে।")\n                    model=build_classical_model(kind,ytr)\n                    try: model.set_params(device="cpu")\n                    except Exception: pass\n                    model.fit(Xtr_all[tr_valid],ytr)\n                else:\n                    raise\n            if te_valid.any():\n                fold_probs[te_valid,j]=_predict_positive(model,Xte_all[te_valid]).astype(np.float32)\n            np.savez_compressed(unit_path,prob=fold_probs[:,j])\n            if SAVE_FITTED_CLASSICAL_MODELS:\n                joblib.dump(model,MODEL_DIR/f"{slug}_fold{fold+1}_{model_slug(target)}.joblib",compress=3)\n            print(f"   ✅ {target}: save complete")\n            del model; gc.collect()\n\n        oof[test_idx]=fold_probs\n        part=metrics_from_predictions(test_idx,fold_probs,fold,model_name)\n        metric_parts.append(part)\n        print(f"✅ Fold {fold+1} সম্পন্ন | Macro AUC={part[\'AUC\'].mean():.4f} | Macro Accuracy={part[\'Accuracy\'].mean():.4f}")\n        del Xtr_all,Xte_all; gc.collect()\n\n    metrics=pd.concat(metric_parts,ignore_index=True)\n    per_task,fold_macro,overall=save_model_outputs(model_name,oof,metrics)\n    show_model_result(model_name,per_task,overall)\n    return {"oof":oof,"metrics":metrics,"per_task":per_task,"fold_macro":fold_macro,"overall":overall}\n\nprint("✅ Classical ML CV engine প্রস্তুত হয়েছে।")\n\n\n# ===== Recovered from notebook cell 44 =====\ndef make_inner_group_holdout(train_pool_idx, val_fraction=0.15, seed=SEED):\n    rng=np.random.default_rng(seed)\n    temp=df.iloc[train_pool_idx]\n    groups=[]\n    for scaf,part in temp.groupby("scaffold"):\n        idx=part.index.to_numpy()\n        pos=np.nansum(part[TARGETS].to_numpy(float)==1,axis=0)\n        groups.append((scaf,idx,pos))\n    rng.shuffle(groups)\n    groups.sort(key=lambda x:(len(x[1])+2*np.sqrt(x[2].sum()+1)),reverse=True)\n    desired=max(1,int(round(len(train_pool_idx)*val_fraction)))\n    val=[]; val_pos=np.zeros(len(TARGETS)); total_pos=np.nansum(df.iloc[train_pool_idx][TARGETS].to_numpy(float)==1,axis=0)\n    desired_pos=total_pos*val_fraction\n    for _,idx,pos in groups:\n        if len(val)>=desired and np.all(val_pos >= np.minimum(desired_pos,1)):\n            break\n        # Add group when it improves size/positive coverage.\n        current=np.mean(((val_pos-desired_pos)/(desired_pos+1))**2)+((len(val)-desired)/max(desired,1))**2\n        new=np.mean(((val_pos+pos-desired_pos)/(desired_pos+1))**2)+((len(val)+len(idx)-desired)/max(desired,1))**2\n        if new<=current or len(val)<desired*0.65:\n            val.extend(idx.tolist()); val_pos+=pos\n    val_idx=np.array(sorted(set(val)),dtype=int)\n    train_idx=np.array(sorted(set(train_pool_idx)-set(val_idx)),dtype=int)\n    if len(train_idx)==0 or len(val_idx)==0:\n        cut=max(1,int(len(train_pool_idx)*(1-val_fraction)))\n        train_idx=np.asarray(train_pool_idx[:cut]); val_idx=np.asarray(train_pool_idx[cut:])\n    return train_idx,val_idx\n\ndef compute_pos_weight(indices):\n    out=[]\n    for j in range(len(TARGETS)):\n        y=Y[indices,j]; y=y[~np.isnan(y)]\n        pos=max(1,np.sum(y==1)); neg=max(1,np.sum(y==0))\n        out.append(min(25.0,max(1.0,float((neg/pos)**0.75))))\n    return torch.tensor(out,dtype=torch.float32)\n\ndef masked_weighted_bce(logits,y,mask,pos_weight):\n    loss=F.binary_cross_entropy_with_logits(logits,y,reduction="none",pos_weight=pos_weight)\n    loss=loss*mask\n    return loss.sum()/mask.sum().clamp_min(1.0)\n\nclass ArrayMultiTaskDataset(Dataset):\n    def __init__(self,X,indices,augment=None):\n        self.X=X; self.indices=np.asarray(indices,dtype=int); self.augment=augment\n    def __len__(self): return len(self.indices)\n    def __getitem__(self,k):\n        idx=int(self.indices[k]); x=np.asarray(self.X[k],dtype=np.float32)\n        if self.augment is not None: x=self.augment(x)\n        return {"x":torch.from_numpy(np.ascontiguousarray(x)),"y":torch.from_numpy(Y_TENSOR_SAFE[idx]),\n                "mask":torch.from_numpy(LABEL_MASK[idx]),"idx":torch.tensor(idx,dtype=torch.long)}\n\ndef make_loader(dataset,batch_size,shuffle):\n    return DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,num_workers=RESOURCE["num_workers"],\n                      pin_memory=(DEVICE.type=="cuda"),drop_last=False,persistent_workers=(RESOURCE["num_workers"]>0))\n\ndef move_batch(batch):\n    return {k:(v.to(DEVICE,non_blocking=True) if torch.is_tensor(v) else v) for k,v in batch.items()}\n\ndef torch_load_compat(path,map_location=DEVICE):\n    try: return torch.load(path,map_location=map_location,weights_only=False)\n    except TypeError: return torch.load(path,map_location=map_location)\n\ndef predict_loader(model,loader,forward_fn):\n    model.eval(); probs=[]; indices=[]\n    with torch.no_grad():\n        for batch in loader:\n            batch=move_batch(batch)\n            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=AMP_ENABLED):\n                logits=forward_fn(model,batch)\n            probs.append(torch.sigmoid(logits).detach().cpu().numpy())\n            indices.append(batch["idx"].detach().cpu().numpy())\n    return np.concatenate(indices),np.concatenate(probs)\n\ndef macro_auc_loader(model,loader,forward_fn):\n    idx,prob=predict_loader(model,loader,forward_fn)\n    vals=[]\n    for j in range(len(TARGETS)):\n        valid=LABEL_MASK[idx,j].astype(bool)\n        if valid.any(): vals.append(safe_auc(Y[idx,j][valid],prob[valid,j]))\n    return float(np.nanmean(vals)) if vals else np.nan\n\ndef _new_grad_scaler():\n    try: return torch.amp.GradScaler("cuda",enabled=AMP_ENABLED)\n    except Exception: return torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)\n\ndef train_with_early_stopping(model,train_loader,val_loader,forward_fn,pos_weight,fold_dir,max_epochs,patience,lr=3e-4,weight_decay=1e-5):\n    fold_dir.mkdir(parents=True,exist_ok=True)\n    last_path=fold_dir/"selection_last.pt"; best_path=fold_dir/"selection_best.pt"; done_path=fold_dir/"selection_done.json"\n    optimizer=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=weight_decay)\n    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="max",factor=0.5,patience=max(2,patience//3),min_lr=1e-6)\n    scaler=_new_grad_scaler(); start=0; best=-np.inf; wait=0; history=[]; best_epoch=1\n    if last_path.exists() and not FORCE_REBUILD:\n        ck=torch_load_compat(last_path)\n        model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])\n        try: scheduler.load_state_dict(ck["scheduler"])\n        except Exception: pass\n        try: scaler.load_state_dict(ck["scaler"])\n        except Exception: pass\n        start=int(ck["epoch"])+1; best=float(ck["best"]); wait=int(ck["wait"]); history=ck.get("history",[]); best_epoch=int(ck.get("best_epoch",1))\n        print(f"      🔄 Epoch checkpoint restore: {start}/{max_epochs}")\n    pos_weight=pos_weight.to(DEVICE)\n    for epoch in range(start,max_epochs):\n        model.train(); losses=[]\n        for batch in train_loader:\n            batch=move_batch(batch); optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=AMP_ENABLED):\n                logits=forward_fn(model,batch)\n                loss=masked_weighted_bce(logits,batch["y"],batch["mask"],pos_weight)\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)\n            scaler.step(optimizer); scaler.update(); losses.append(float(loss.detach().cpu()))\n        val_auc=macro_auc_loader(model,val_loader,forward_fn)\n        scheduler.step(val_auc if np.isfinite(val_auc) else -1)\n        train_loss=float(np.mean(losses)) if losses else np.nan\n        history.append({"epoch":epoch+1,"train_loss":train_loss,"val_auc":val_auc,"lr":optimizer.param_groups[0]["lr"]})\n        improved=np.isfinite(val_auc) and val_auc>best+1e-5\n        if improved:\n            best=val_auc; wait=0; best_epoch=epoch+1\n            torch.save({"model":model.state_dict(),"epoch":epoch,"best":best},best_path)\n        else: wait+=1\n        torch.save({"epoch":epoch,"model":model.state_dict(),"optimizer":optimizer.state_dict(),"scheduler":scheduler.state_dict(),\n                    "scaler":scaler.state_dict(),"best":best,"wait":wait,"history":history,"best_epoch":best_epoch},last_path)\n        print(f"      Epoch {epoch+1:03d} | loss={train_loss:.5f} | val AUC={val_auc:.4f} | best={best:.4f}")\n        if wait>=patience:\n            print("      ⏹️ Early stopping triggered."); break\n    if best_path.exists(): model.load_state_dict(torch_load_compat(best_path)["model"])\n    done_path.write_text(json.dumps({"best_epoch":best_epoch,"best_val_auc":best,"history":history},default=float))\n    return best_epoch,best,history\n\ndef train_fixed_epochs(model,train_loader,forward_fn,pos_weight,fold_dir,epochs,lr=3e-4,weight_decay=1e-5):\n    fold_dir.mkdir(parents=True,exist_ok=True)\n    last_path=fold_dir/"refit_last.pt"; done_path=fold_dir/"refit_done.json"; final_path=fold_dir/"refit_final.pt"\n    optimizer=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=weight_decay); scaler=_new_grad_scaler(); start=0; history=[]\n    if last_path.exists() and not FORCE_REBUILD:\n        ck=torch_load_compat(last_path); model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])\n        try: scaler.load_state_dict(ck["scaler"])\n        except Exception: pass\n        start=int(ck["epoch"])+1; history=ck.get("history",[]); print(f"      🔄 Refit checkpoint restore: {start}/{epochs}")\n    pos_weight=pos_weight.to(DEVICE)\n    for epoch in range(start,epochs):\n        model.train(); losses=[]\n        for batch in train_loader:\n            batch=move_batch(batch); optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=AMP_ENABLED):\n                logits=forward_fn(model,batch); loss=masked_weighted_bce(logits,batch["y"],batch["mask"],pos_weight)\n            scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)\n            scaler.step(optimizer); scaler.update(); losses.append(float(loss.detach().cpu()))\n        history.append({"epoch":epoch+1,"train_loss":float(np.mean(losses))})\n        torch.save({"epoch":epoch,"model":model.state_dict(),"optimizer":optimizer.state_dict(),"scaler":scaler.state_dict(),"history":history},last_path)\n        print(f"      Refit epoch {epoch+1:03d}/{epochs} | loss={history[-1][\'train_loss\']:.5f}")\n    torch.save({"model":model.state_dict(),"epochs":epochs},final_path)\n    done_path.write_text(json.dumps({"epochs":epochs,"history":history},default=float))\n    return history\n\nprint("✅ Deep Learning training, early stopping এবং per-epoch resume engine প্রস্তুত হয়েছে।")\n\n\n# ===== Recovered from notebook cell 45 =====\nclass TabularMTDNN(nn.Module):\n    def __init__(self,input_dim,hidden_dims,dropouts,input_dropout=0.1):\n        super().__init__(); self.input_dropout=nn.Dropout(input_dropout); blocks=[]; prev=input_dim\n        for h,d in zip(hidden_dims,dropouts):\n            blocks += [nn.Linear(prev,h),nn.LayerNorm(h),nn.ReLU(),nn.Dropout(d)]\n            prev=h\n        self.backbone=nn.Sequential(*blocks); self.out=nn.Linear(prev,len(TARGETS))\n    def forward(self,x): return self.out(self.backbone(self.input_dropout(x)))\n\ndef build_tabular_resources(train_idx,val_idx,test_idx,fold,stage,architecture):\n    compact=architecture=="compact"\n    mode="dnn"\n    Xtr,Xva,art=prepare_tabular_fold(train_idx,val_idx,mode=mode,compact=compact)\n    _,Xte,_=prepare_tabular_fold(train_idx,test_idx,mode=mode,compact=compact)\n    # The second call above fits an equivalent train-only transformer. For exact consistency, transform test with the first artifacts:\n    desc_cols=COMPACT_DESC_IDX if compact else list(range(len(DESC_NAMES)))\n    dte=art["imputer"].transform(np.asarray(X_DESC[test_idx])[:,desc_cols])\n    dte=art["scaler"].transform(dte)\n    Xte=np.concatenate([np.asarray(X_ECFP[test_idx],dtype=np.float32),dte.astype(np.float32)],axis=1).astype(np.float32)\n    if SMOKE_TEST:\n        hidden=[64,32]; drops=[0.15,0.10]\n    else:\n        hidden=[1024,512,128] if compact else [2048,1024,512]\n        drops=[0.30,0.30,0.25] if compact else [0.35,0.40,0.35]\n    model=TabularMTDNN(Xtr.shape[1],hidden,drops,input_dropout=0.08).to(DEVICE)\n    return {\n        "model":model,\n        "train_loader":make_loader(ArrayMultiTaskDataset(Xtr,train_idx),RESOURCE["tab_batch"],True),\n        "val_loader":make_loader(ArrayMultiTaskDataset(Xva,val_idx),RESOURCE["tab_batch"],False),\n        "test_loader":make_loader(ArrayMultiTaskDataset(Xte,test_idx),RESOURCE["tab_batch"],False),\n        "forward_fn":lambda m,b:m(b["x"]),\n        "pos_weight":compute_pos_weight(train_idx),\n    }\n\ndef run_torch_cv(model_name,resource_builder,max_epochs,lr=3e-4,weight_decay=1e-5):\n    slug=model_slug(model_name); pred_dir=PRED_DIR/slug; pred_dir.mkdir(parents=True,exist_ok=True)\n    histories={}; oof=np.full((len(df),len(TARGETS)),np.nan,dtype=np.float32); metric_parts=[]\n    for fold in range(N_FOLDS):\n        fold_pred=pred_dir/f"fold{fold+1}.npz"\n        if fold_pred.exists() and not FORCE_REBUILD:\n            z=np.load(fold_pred); idx=z["idx"].astype(int); prob=z["prob"]\n            oof[idx]=prob; metric_parts.append(metrics_from_predictions(idx,prob,fold,model_name))\n            print(f"♻️ {model_name} Fold {fold+1}: completed prediction checkpoint load")\n            continue\n        print(f"\\n📘 {model_name} | Fold {fold+1}/{N_FOLDS} শুরু")\n        test_idx=np.where(df["fold"].to_numpy()==fold)[0]\n        train_pool=np.where(df["fold"].to_numpy()!=fold)[0]\n        inner_train,val_idx=make_inner_group_holdout(train_pool,0.15,SEED+fold)\n        fold_dir=MODEL_DIR/slug/f"fold{fold+1}"\n\n        res=resource_builder(inner_train,val_idx,test_idx,fold,"selection")\n        best_epoch,best_val,hist=train_with_early_stopping(\n            res["model"],res["train_loader"],res["val_loader"],res["forward_fn"],res["pos_weight"],\n            fold_dir,max_epochs,RESOURCE["patience"],lr,weight_decay\n        )\n        histories[f"fold{fold+1}_selection"]={"best_epoch":best_epoch,"best_val_auc":best_val,"history":hist}\n\n        if REFIT_DEEP_ON_FULL_OUTER_TRAIN:\n            # Use a tiny validation placeholder only to build consistent resources; refit uses full train loader.\n            placeholder=val_idx[:max(1,min(8,len(val_idx)))]\n            full=resource_builder(train_pool,placeholder,test_idx,fold,"refit")\n            refit_epochs=(1 if SMOKE_TEST else max(2,best_epoch))\n            refit_hist=train_fixed_epochs(full["model"],full["train_loader"],full["forward_fn"],full["pos_weight"],fold_dir,refit_epochs,lr,weight_decay)\n            model=full["model"]; test_loader=full["test_loader"]; forward_fn=full["forward_fn"]\n            histories[f"fold{fold+1}_refit"]={"epochs":refit_epochs,"history":refit_hist}\n        else:\n            model=res["model"]; test_loader=res["test_loader"]; forward_fn=res["forward_fn"]\n\n        idx,prob=predict_loader(model,test_loader,forward_fn)\n        order=np.argsort(idx); idx=idx[order]; prob=prob[order]\n        np.savez_compressed(fold_pred,idx=idx,prob=prob)\n        oof[idx]=prob; part=metrics_from_predictions(idx,prob,fold,model_name); metric_parts.append(part)\n        print(f"✅ Fold {fold+1} prediction save হয়েছে | Macro AUC={part[\'AUC\'].mean():.4f}")\n        del res,model,test_loader; gc.collect()\n        if torch.cuda.is_available(): torch.cuda.empty_cache()\n\n    metrics=pd.concat(metric_parts,ignore_index=True)\n    per_task,fold_macro,overall=save_model_outputs(model_name,oof,metrics,histories)\n    show_model_result(model_name,per_task,overall)\n    return {"oof":oof,"metrics":metrics,"per_task":per_task,"fold_macro":fold_macro,"overall":overall,"histories":histories}\n\nprint("✅ Multi-task tabular neural architecture এবং CV runner প্রস্তুত হয়েছে।")\n\n\n# ===== Recovered from notebook cell 49 =====\ndef squash_caps(s,dim=-1,eps=1e-8):\n    norm2=(s*s).sum(dim=dim,keepdim=True)\n    scale=norm2/(1.0+norm2)/torch.sqrt(norm2+eps)\n    return scale*s\n\nclass DynamicTaskCapsules(nn.Module):\n    def __init__(self,n_primary=8,in_dim=8,n_tasks=None,out_dim=8,routing_iters=2):\n        super().__init__(); n_tasks=n_tasks or len(TARGETS)\n        self.n_primary=n_primary; self.n_tasks=n_tasks; self.out_dim=out_dim; self.routing_iters=routing_iters\n        self.W=nn.Parameter(torch.empty(n_primary,n_tasks,out_dim,in_dim)); nn.init.xavier_uniform_(self.W)\n    def forward(self,u):\n        # u: [B, primary, in_dim], u_hat: [B, primary, task, out_dim]\n        u_hat=torch.einsum("bpi,ptoi->bpto",u,self.W)\n        b=torch.zeros(u.shape[0],self.n_primary,self.n_tasks,device=u.device,dtype=u.dtype)\n        for r in range(self.routing_iters):\n            c=torch.softmax(b,dim=2)\n            s=(c.unsqueeze(-1)*u_hat).sum(dim=1)\n            v=squash_caps(s)\n            if r<self.routing_iters-1:\n                b=b+(u_hat*v.unsqueeze(1)).sum(dim=-1)\n        return v\n\nclass RBMCapsNet(nn.Module):\n    def __init__(self,rbm1=None,rbm2=None):\n        super().__init__(); self.enc1=nn.Linear(179,256); self.enc2=nn.Linear(256,128); self.dropout=nn.Dropout(0.25)\n        if rbm1 is not None:\n            self.enc1.weight.data.copy_(torch.tensor(rbm1.components_,dtype=torch.float32))\n            self.enc1.bias.data.copy_(torch.tensor(rbm1.intercept_hidden_,dtype=torch.float32))\n        if rbm2 is not None:\n            self.enc2.weight.data.copy_(torch.tensor(rbm2.components_,dtype=torch.float32))\n            self.enc2.bias.data.copy_(torch.tensor(rbm2.intercept_hidden_,dtype=torch.float32))\n        self.primary=nn.Linear(128,8*8)\n        self.routing=DynamicTaskCapsules(8,8,len(TARGETS),8,2)\n        self.task_head=nn.Linear(8,1)\n    def forward(self,x):\n        h=torch.sigmoid(self.enc1(x)); h=self.dropout(h); h=torch.sigmoid(self.enc2(h));\n        u=squash_caps(self.primary(h).view(-1,8,8)); v=self.routing(u)\n        return self.task_head(v).squeeze(-1)\n\ndef fit_or_load_rbms(Xtrain,cache_prefix):\n    p1=Path(str(cache_prefix)+"_rbm1.joblib"); p2=Path(str(cache_prefix)+"_rbm2.joblib")\n    if p1.exists() and p2.exists() and not FORCE_REBUILD:\n        return joblib.load(p1),joblib.load(p2)\n    rbm1=BernoulliRBM(n_components=256,learning_rate=0.02,batch_size=64,n_iter=1 if SMOKE_TEST else 15,verbose=0,random_state=SEED)\n    h1=rbm1.fit_transform(Xtrain)\n    rbm2=BernoulliRBM(n_components=128,learning_rate=0.02,batch_size=64,n_iter=1 if SMOKE_TEST else 12,verbose=0,random_state=SEED+1)\n    rbm2.fit(h1)\n    joblib.dump(rbm1,p1,compress=3); joblib.dump(rbm2,p2,compress=3)\n    return rbm1,rbm2\n\ndef build_caps_resources(train_idx,val_idx,test_idx,fold,stage):\n    Xtr,Xva,art=prepare_capsnet_fold(train_idx,val_idx)\n    # Transform test with train-fitted artifacts.\n    dte=art["imputer"].transform(np.asarray(X_DESC[test_idx],dtype=np.float32)); dte=art["scaler"].transform(dte)\n    Xte=np.concatenate([np.asarray(X_MACCS[test_idx,1:],dtype=np.float32),dte],axis=1).astype(np.float32)\n    prefix=MODEL_DIR/model_slug("Multitask CapsNet + RBM")/f"fold{fold+1}_{stage}"\n    prefix.parent.mkdir(parents=True,exist_ok=True)\n    rbm1,rbm2=fit_or_load_rbms(Xtr,prefix)\n    model=RBMCapsNet(rbm1,rbm2).to(DEVICE)\n    return {"model":model,"train_loader":make_loader(ArrayMultiTaskDataset(Xtr,train_idx),RESOURCE["tab_batch"],True),\n            "val_loader":make_loader(ArrayMultiTaskDataset(Xva,val_idx),RESOURCE["tab_batch"],False),\n            "test_loader":make_loader(ArrayMultiTaskDataset(Xte,test_idx),RESOURCE["tab_batch"],False),\n            "forward_fn":lambda m,b:m(b["x"]),"pos_weight":compute_pos_weight(train_idx)}\n\nprint("✅ RBM-initialized Multitask CapsNet architecture প্রস্তুত হয়েছে।")\n\n\n# ===== Recovered from notebook cell 52 =====\nFEATURE_FACTORY = ChemicalFeatures.BuildFeatureFactory(str(Path(RDConfig.RDDataDir)/"BaseFeatures.fdef"))\nGRID_SIZE=48; GRID_HALF=12.0; GRID_AXIS=np.linspace(-GRID_HALF,GRID_HALF,GRID_SIZE,dtype=np.float32)\nGX,GY=np.meshgrid(GRID_AXIS,GRID_AXIS,indexing="xy")\nMETALS={3,4,11,12,13,19,20,21,22,23,24,25,26,27,28,29,30,31,37,38,39,40,41,42,43,44,45,46,47,48,49,50,55,56,57,78,79,80}\n\ndef molecule_to_six_channel_grid(smiles):\n    mol=Chem.MolFromSmiles(smiles)\n    if mol is None: return np.zeros((6,GRID_SIZE,GRID_SIZE),dtype=np.float16)\n    AllChem.Compute2DCoords(mol)\n    conf=mol.GetConformer(); coords=np.array([[conf.GetAtomPosition(i).x,conf.GetAtomPosition(i).y] for i in range(mol.GetNumAtoms())],dtype=np.float32)\n    coords-=coords.mean(axis=0,keepdims=True)\n    max_abs=max(1.0,float(np.abs(coords).max()))\n    if max_abs>10.5: coords*=10.5/max_abs\n    fam_atoms={"Donor":set(),"Acceptor":set(),"Hydrophobe":set(),"LumpedHydrophobe":set(),"PosIonizable":set(),"NegIonizable":set()}\n    try:\n        for feat in FEATURE_FACTORY.GetFeaturesForMol(mol):\n            if feat.GetFamily() in fam_atoms: fam_atoms[feat.GetFamily()].update(feat.GetAtomIds())\n    except Exception: pass\n    grid=np.zeros((6,GRID_SIZE,GRID_SIZE),dtype=np.float32)\n    pt=Chem.GetPeriodicTable()\n    for i,atom in enumerate(mol.GetAtoms()):\n        x,y=coords[i]; d2=(GX-x)**2+(GY-y)**2\n        z=atom.GetAtomicNum(); r=max(0.7,float(pt.GetRvdw(z)))\n        g=np.exp(-d2/(2*(0.55+0.15*r)**2))\n        if i in fam_atoms["Donor"]: grid[0]+=1.0*g\n        if i in fam_atoms["Acceptor"]: grid[0]+=0.85*g\n        if i in fam_atoms["Hydrophobe"] or i in fam_atoms["LumpedHydrophobe"] or z in {6,9,17,35,53}: grid[1]+=g\n        if z in METALS: grid[2]+=1.2*g\n        grid[3]+=r*g\n        if i in fam_atoms["PosIonizable"] or atom.GetFormalCharge()>0: grid[4]+=g\n        if i in fam_atoms["NegIonizable"] or atom.GetFormalCharge()<0: grid[5]+=g\n    for c in range(6):\n        mx=float(np.max(np.abs(grid[c])))\n        if mx>0: grid[c]/=mx\n    return grid.astype(np.float16)\n\nGRID_PATH=FEATURE_ROOT/"six_channel_48x48_float16.npy"\nGRID_PROGRESS=FEATURE_ROOT/"grid_progress.json"\n\ndef build_or_load_grids():\n    valid=False\n    if GRID_PATH.exists() and not FORCE_REBUILD:\n        try: valid=np.load(GRID_PATH,mmap_mode="r").shape==(len(df),6,GRID_SIZE,GRID_SIZE)\n        except Exception: valid=False\n    if not valid:\n        if FORCE_REBUILD: GRID_PATH.unlink(missing_ok=True); GRID_PROGRESS.unlink(missing_ok=True)\n        if not GRID_PATH.exists():\n            arr=np.lib.format.open_memmap(GRID_PATH,mode="w+",dtype=np.float16,shape=(len(df),6,GRID_SIZE,GRID_SIZE)); start=0\n        else:\n            arr=np.lib.format.open_memmap(GRID_PATH,mode="r+"); start=json.loads(GRID_PROGRESS.read_text())["next_index"] if GRID_PROGRESS.exists() else 0\n            print(f"🔄 Grid checkpoint পাওয়া গেছে। Molecule {start} থেকে resume হচ্ছে।")\n        for i in tqdm(range(start,len(df)),desc="Six-channel molecular grids"):\n            arr[i]=molecule_to_six_channel_grid(df.iloc[i]["std_smiles"])\n            if (i+1)%50==0 or i+1==len(df):\n                arr.flush(); GRID_PROGRESS.write_text(json.dumps({"next_index":i+1}))\n        GRID_PROGRESS.unlink(missing_ok=True); del arr\n        print("✅ Six-channel grid generation সম্পন্ন হয়েছে।")\n    else: print("♻️ Cached six-channel grids পাওয়া গেছে।")\n    return np.load(GRID_PATH,mmap_mode="r")\n\nprint("✅ Six-channel grid generator প্রস্তুত হয়েছে।")\n\n\n# ===== Recovered from notebook cell 54 =====\nclass GridDataset(Dataset):\n    def __init__(self,grid_path,indices,augment=False): self.grid_path=grid_path; self.indices=np.asarray(indices); self.augment=augment; self._grid=None\n    def _arr(self):\n        if self._grid is None: self._grid=np.load(self.grid_path,mmap_mode="r")\n        return self._grid\n    def __len__(self): return len(self.indices)\n    def __getitem__(self,k):\n        idx=int(self.indices[k]); x=np.asarray(self._arr()[idx],dtype=np.float32)\n        if self.augment:\n            rot=np.random.randint(0,4); x=np.rot90(x,rot,axes=(1,2)).copy()\n            if np.random.rand()<.5: x=x[:,:,::-1].copy()\n        return {"x":torch.from_numpy(np.ascontiguousarray(x)),"y":torch.from_numpy(Y_TENSOR_SAFE[idx]),\n                "mask":torch.from_numpy(LABEL_MASK[idx]),"idx":torch.tensor(idx,dtype=torch.long)}\n\nclass MultiChannelCNN(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.features=nn.Sequential(\n            nn.Conv2d(6,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),\n            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2),\n            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),\n            nn.Conv2d(128,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),nn.AdaptiveAvgPool2d(1)\n        )\n        self.head=nn.Sequential(nn.Flatten(),nn.Linear(128,256),nn.ReLU(),nn.Dropout(.35),nn.Linear(256,len(TARGETS)))\n    def forward(self,x): return self.head(self.features(x))\n\ndef build_cnn_resources(train_idx,val_idx,test_idx,fold,stage):\n    model=MultiChannelCNN().to(DEVICE)\n    return {"model":model,"train_loader":make_loader(GridDataset(GRID_PATH,train_idx,True),RESOURCE["cnn_batch"],True),\n            "val_loader":make_loader(GridDataset(GRID_PATH,val_idx,False),RESOURCE["cnn_batch"],False),\n            "test_loader":make_loader(GridDataset(GRID_PATH,test_idx,False),RESOURCE["cnn_batch"],False),\n            "forward_fn":lambda m,b:m(b["x"]),"pos_weight":compute_pos_weight(train_idx)}\n\nprint("✅ Multi-channel 2D CNN architecture প্রস্তুত হয়েছে।")\n\n\n# ===== Recovered from notebook cell 59 =====\nATOM_NUMS=[1,5,6,7,8,9,14,15,16,17,35,53,3,4,11,12,19,20,26,30]\nHYBRIDS=[Chem.rdchem.HybridizationType.SP,Chem.rdchem.HybridizationType.SP2,Chem.rdchem.HybridizationType.SP3,\n         Chem.rdchem.HybridizationType.SP3D,Chem.rdchem.HybridizationType.SP3D2,Chem.rdchem.HybridizationType.UNSPECIFIED]\nCHIRALS=[Chem.rdchem.ChiralType.CHI_UNSPECIFIED,Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,\n         Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,Chem.rdchem.ChiralType.CHI_OTHER]\n\ndef one_hot_unknown(value,choices):\n    return [float(value==x) for x in choices]+[float(value not in choices)]\n\ndef atom_feature(atom):\n    feat=[]\n    feat+=one_hot_unknown(atom.GetAtomicNum(),ATOM_NUMS)\n    feat+=one_hot_unknown(atom.GetDegree(),list(range(6)))\n    feat+=one_hot_unknown(atom.GetFormalCharge(),[-2,-1,0,1,2])\n    feat+=one_hot_unknown(atom.GetHybridization(),HYBRIDS[:-1])\n    feat+=one_hot_unknown(atom.GetChiralTag(),CHIRALS[:-1])\n    feat += [float(atom.GetIsAromatic()),float(atom.IsInRing()),float(atom.GetMass()/200.0),float(atom.GetTotalValence()/8.0)]\n    return np.asarray(feat,dtype=np.float32)\n\ndef bond_relation(bond):\n    bt=bond.GetBondType()\n    if bt==Chem.rdchem.BondType.SINGLE:return 0\n    if bt==Chem.rdchem.BondType.DOUBLE:return 1\n    if bt==Chem.rdchem.BondType.TRIPLE:return 2\n    if bt==Chem.rdchem.BondType.AROMATIC:return 3\n    return 4\n\ndef smiles_to_rel_graph(smiles):\n    mol=Chem.MolFromSmiles(smiles)\n    x=np.stack([atom_feature(a) for a in mol.GetAtoms()]).astype(np.float32)\n    src=[];dst=[];rel=[]\n    for b in mol.GetBonds():\n        i=b.GetBeginAtomIdx();j=b.GetEndAtomIdx();r=bond_relation(b)\n        src += [i,j]; dst += [j,i]; rel += [r,r]\n    edge_index=np.asarray([src,dst],dtype=np.int64)\n    edge_type=np.asarray(rel,dtype=np.int64)\n    return {"x":x,"edge_index":edge_index,"edge_type":edge_type}\n\nGRAPH_ROOT=FEATURE_ROOT/"rel_graph_shards"; GRAPH_ROOT.mkdir(parents=True,exist_ok=True)\nGRAPH_MANIFEST=GRAPH_ROOT/"manifest.json"; GRAPH_SHARD_SIZE=500\n\ndef build_graph_cache():\n    n_shards=math.ceil(len(df)/GRAPH_SHARD_SIZE)\n    for s in range(n_shards):\n        path=GRAPH_ROOT/f"shard_{s:03d}.joblib"\n        if path.exists() and not FORCE_REBUILD: continue\n        lo=s*GRAPH_SHARD_SIZE; hi=min(len(df),(s+1)*GRAPH_SHARD_SIZE)\n        graphs=[]\n        for i in tqdm(range(lo,hi),desc=f"Graph shard {s+1}/{n_shards}",leave=False):\n            graphs.append(smiles_to_rel_graph(df.iloc[i]["std_smiles"]))\n        joblib.dump(graphs,path,compress=3)\n        print(f"   ✅ Graph shard {s+1}/{n_shards} save হয়েছে।")\n    manifest={"n":len(df),"shard_size":GRAPH_SHARD_SIZE,"n_shards":n_shards,"atom_dim":len(atom_feature(Chem.MolFromSmiles(\'C\').GetAtomWithIdx(0))),"relations":5}\n    GRAPH_MANIFEST.write_text(json.dumps(manifest,indent=2))\n    print("✅ Molecular relational graph cache সম্পন্ন হয়েছে।")\n\ndef load_all_graphs():\n    manifest=json.loads(GRAPH_MANIFEST.read_text()); graphs=[]\n    for s in range(manifest["n_shards"]): graphs.extend(joblib.load(GRAPH_ROOT/f"shard_{s:03d}.joblib"))\n    return graphs[:manifest["n"]]\n\nprint("✅ Molecular relation-graph feature functions প্রস্তুত হয়েছে।")\n\n\n# ===== Recovered from notebook cell 61 =====\nclass GraphMultiTaskDataset(Dataset):\n    def __init__(self,graphs,tabular,indices): self.graphs=graphs; self.tabular=tabular; self.indices=np.asarray(indices,dtype=int)\n    def __len__(self): return len(self.indices)\n    def __getitem__(self,k):\n        idx=int(self.indices[k]); g=self.graphs[idx]\n        return {"node_x":torch.from_numpy(g["x"]),"edge_index":torch.from_numpy(g["edge_index"]),"edge_type":torch.from_numpy(g["edge_type"]),\n                "tabular":torch.from_numpy(np.asarray(self.tabular[k],dtype=np.float32)),"y":torch.from_numpy(Y_TENSOR_SAFE[idx]),\n                "mask":torch.from_numpy(LABEL_MASK[idx]),"idx":torch.tensor(idx,dtype=torch.long)}\n\ndef graph_collate(items):\n    node_x=[]; edges=[]; types=[]; batch=[]; tabs=[];ys=[];masks=[];idxs=[];offset=0\n    for b,it in enumerate(items):\n        n=it["node_x"].shape[0]; node_x.append(it["node_x"]); edges.append(it["edge_index"]+offset); types.append(it["edge_type"])\n        batch.append(torch.full((n,),b,dtype=torch.long)); tabs.append(it["tabular"]);ys.append(it["y"]);masks.append(it["mask"]);idxs.append(it["idx"]);offset+=n\n    return {"node_x":torch.cat(node_x),"edge_index":torch.cat(edges,dim=1) if edges else torch.empty((2,0),dtype=torch.long),\n            "edge_type":torch.cat(types) if types else torch.empty((0,),dtype=torch.long),"batch":torch.cat(batch),\n            "tabular":torch.stack(tabs),"y":torch.stack(ys),"mask":torch.stack(masks),"idx":torch.stack(idxs)}\n\ndef make_graph_loader(dataset,shuffle):\n    return DataLoader(dataset,batch_size=RESOURCE["graph_batch"],shuffle=shuffle,num_workers=RESOURCE["num_workers"],\n                      pin_memory=(DEVICE.type=="cuda"),collate_fn=graph_collate,persistent_workers=(RESOURCE["num_workers"]>0))\n\nclass RelGraphConv(nn.Module):\n    def __init__(self,in_dim,out_dim,num_rel=5):\n        super().__init__(); self.weight=nn.Parameter(torch.empty(num_rel,in_dim,out_dim)); self.self_lin=nn.Linear(in_dim,out_dim,bias=False); self.bias=nn.Parameter(torch.zeros(out_dim)); nn.init.xavier_uniform_(self.weight)\n    def forward(self,x,edge_index,edge_type):\n        out=self.self_lin(x); deg=torch.ones(x.shape[0],device=x.device,dtype=x.dtype)\n        if edge_index.numel()>0:\n            src,dst=edge_index\n            agg=torch.zeros(x.shape[0],self.weight.shape[-1],device=x.device,dtype=x.dtype)\n            dadd=torch.zeros(x.shape[0],device=x.device,dtype=x.dtype)\n            for r in range(self.weight.shape[0]):\n                mask=edge_type==r\n                if mask.any():\n                    s=src[mask]; d=dst[mask]; msg=x[s]@self.weight[r]; agg.index_add_(0,d,msg); dadd.index_add_(0,d,torch.ones_like(d,dtype=x.dtype))\n            out=out+agg/(dadd.clamp_min(1).unsqueeze(1)); deg=deg+dadd\n        return out+self.bias\n\ndef global_mean_max(x,batch):\n    B=int(batch.max().item())+1 if batch.numel() else 0; D=x.shape[1]\n    mean=torch.zeros(B,D,device=x.device,dtype=x.dtype); mean.index_add_(0,batch,x)\n    cnt=torch.bincount(batch,minlength=B).clamp_min(1).to(x.dtype).unsqueeze(1); mean=mean/cnt\n    try:\n        mx=torch.full((B,D),-torch.inf,device=x.device,dtype=x.dtype)\n        mx.scatter_reduce_(0,batch[:,None].expand(-1,D),x,reduce="amax",include_self=True)\n    except Exception:\n        mx=torch.stack([x[batch==i].max(dim=0).values for i in range(B)])\n    return torch.cat([mean,mx],dim=1)\n\nclass MolecularRGCNFusion(nn.Module):\n    def __init__(self,atom_dim,tab_dim):\n        super().__init__()\n        self.c1=RelGraphConv(atom_dim,128); self.n1=nn.LayerNorm(128)\n        self.c2=RelGraphConv(128,256); self.n2=nn.LayerNorm(256)\n        self.c3=RelGraphConv(256,256); self.n3=nn.LayerNorm(256)\n        self.tab=nn.Sequential(nn.Linear(tab_dim,512),nn.LayerNorm(512),nn.LeakyReLU(.1),nn.Dropout(.25),nn.Linear(512,256),nn.LeakyReLU(.1))\n        self.head=nn.Sequential(nn.Linear(512+256,512),nn.LayerNorm(512),nn.LeakyReLU(.1),nn.Dropout(.35),nn.Linear(512,256),nn.LeakyReLU(.1),nn.Dropout(.25),nn.Linear(256,len(TARGETS)))\n    def forward(self,node_x,edge_index,edge_type,batch,tabular):\n        h=F.leaky_relu(self.n1(self.c1(node_x,edge_index,edge_type)),.1); h=F.dropout(h,.15,self.training)\n        h=F.leaky_relu(self.n2(self.c2(h,edge_index,edge_type)),.1); h=F.dropout(h,.20,self.training)\n        h2=F.leaky_relu(self.n3(self.c3(h,edge_index,edge_type)),.1); h=h+h2\n        graph=global_mean_max(h,batch); tab=self.tab(tabular); return self.head(torch.cat([graph,tab],dim=1))\n\ndef build_rgcn_resources(train_idx,val_idx,test_idx,fold,stage):\n    Xtr,Xva,art=prepare_tabular_fold(train_idx,val_idx,mode="dnn",compact=False)\n    dte=art["imputer"].transform(np.asarray(X_DESC[test_idx])); dte=art["scaler"].transform(dte)\n    Xte=np.concatenate([np.asarray(X_ECFP[test_idx],dtype=np.float32),dte.astype(np.float32)],axis=1).astype(np.float32)\n    atom_dim=GRAPH_LIST[0]["x"].shape[1]; model=MolecularRGCNFusion(atom_dim,Xtr.shape[1]).to(DEVICE)\n    return {"model":model,"train_loader":make_graph_loader(GraphMultiTaskDataset(GRAPH_LIST,Xtr,train_idx),True),\n            "val_loader":make_graph_loader(GraphMultiTaskDataset(GRAPH_LIST,Xva,val_idx),False),\n            "test_loader":make_graph_loader(GraphMultiTaskDataset(GRAPH_LIST,Xte,test_idx),False),\n            "forward_fn":lambda m,b:m(b["node_x"],b["edge_index"],b["edge_type"],b["batch"],b["tabular"]),\n            "pos_weight":compute_pos_weight(train_idx)}\n\nprint("✅ Custom relation-specific R-GCN + tabular fusion architecture প্রস্তুত হয়েছে।")\n\n\nRUNTIME_READY = True\nprint("✅ Final_Ultra runtime bootstrap সম্পন্ন হয়েছে। Cached preprocessing/features load করা হয়েছে।")\n'
RUNTIME_BOOTSTRAP = CACHE_DIR / "runtime_bootstrap.py"
RUNTIME_MANIFEST = CACHE_DIR / "runtime_manifest.json"

def save_runtime_session():
    """Save a lightweight, portable recovery manifest instead of pickling RDKit C++ objects."""
    RUNTIME_BOOTSTRAP.write_text(RUNTIME_BOOTSTRAP_SOURCE, encoding="utf-8")
    try:
        (Path.cwd() / ".final_ultra_runtime_bootstrap.py").write_text(RUNTIME_BOOTSTRAP_SOURCE, encoding="utf-8")
    except Exception:
        pass
    manifest={
        "dataset": str(DATA_PATH), "dataset_sha256": RAW_SHA256, "work_root": str(WORK_ROOT),
        "bootstrap": str(RUNTIME_BOOTSTRAP), "targets": TARGETS, "folds": N_FOLDS,
        "updated_at": pd.Timestamp.now().isoformat(),
    }
    RUNTIME_MANIFEST.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("💾 Restart recovery bootstrap ও manifest save হয়েছে:", RUNTIME_BOOTSTRAP)

RUNTIME_READY = True
save_runtime_session()
print("✅ এখন model cell-গুলো kernel restart-এর পর নিজে থেকেই runtime restore করতে পারবে।")


# Classical Machine Learning Models

প্রতিটি endpoint আলাদাভাবে train হবে। ঐ endpoint-এর missing-label row বাদ যাবে। প্রতিটি fold-task শেষ হওয়ার সঙ্গে সঙ্গে prediction save হবে; cell পুনরায় run করলে completed task skip হবে।


In [ ]:
def _predict_positive(model, X):
    if hasattr(model,"predict_proba"):
        return np.asarray(model.predict_proba(X))[:,1]
    if hasattr(model,"decision_function"):
        return expit(np.asarray(model.decision_function(X)))
    return np.asarray(model.predict(X),dtype=float)

def build_classical_model(kind, y_train):
    if kind=="rf":
        return RandomForestClassifier(
            n_estimators=RESOURCE["tree_estimators"], max_features="sqrt", min_samples_leaf=1,
            class_weight="balanced_subsample", bootstrap=True, n_jobs=-1, random_state=SEED
        )
    if kind=="et":
        return ExtraTreesClassifier(
            n_estimators=RESOURCE["tree_estimators"], max_features="sqrt", min_samples_leaf=1,
            class_weight="balanced", bootstrap=False, n_jobs=-1, random_state=SEED
        )
    if kind=="xgb":
        pos=max(1,int(np.sum(y_train==1))); neg=max(1,int(np.sum(y_train==0)))
        params=dict(
            n_estimators=RESOURCE["xgb_estimators"], max_depth=(3 if SMOKE_TEST else 5), learning_rate=(0.15 if SMOKE_TEST else 0.035),
            subsample=0.85, colsample_bytree=0.70, min_child_weight=2,
            reg_alpha=0.05, reg_lambda=2.0, objective="binary:logistic", eval_metric="auc",
            tree_method="hist", n_jobs=(2 if SMOKE_TEST else min(8, max(1, CPU_COUNT-1))), random_state=SEED,
            scale_pos_weight=neg/pos
        )
        if DEVICE.type=="cuda": params["device"]="cuda"
        return XGBClassifier(**params)
    if kind=="svm":
        return SVC(
            kernel="rbf", C=10.0, gamma="scale", class_weight="balanced",
            probability=False, cache_size=384 if LOW_RESOURCE else 1536, random_state=SEED
        )
    raise ValueError(kind)

def run_classical_cv(model_name, kind, feature_mode):
    slug=model_slug(model_name)
    unit_dir=PRED_DIR/slug; unit_dir.mkdir(parents=True,exist_ok=True)
    oof=np.full((len(df),len(TARGETS)),np.nan,dtype=np.float32)
    metric_parts=[]

    for fold in range(N_FOLDS):
        train_idx=np.where(df["fold"].to_numpy()!=fold)[0]
        test_idx=np.where(df["fold"].to_numpy()==fold)[0]
        print(f"\n📘 {model_name} | Fold {fold+1}/{N_FOLDS} শুরু")
        Xtr_all,Xte_all,artifacts=prepare_tabular_fold(train_idx,test_idx,mode=feature_mode,compact=False)
        joblib.dump(artifacts, MODEL_DIR/f"{slug}_fold{fold+1}_preprocess.joblib", compress=3)

        fold_probs=np.full((len(test_idx),len(TARGETS)),np.nan,dtype=np.float32)
        for j,target in enumerate(TARGETS):
            unit_path=unit_dir/f"fold{fold+1}_{model_slug(target)}.npz"
            if unit_path.exists() and not FORCE_REBUILD:
                saved=np.load(unit_path)
                fold_probs[:,j]=saved["prob"]
                print(f"   ♻️ {target}: completed checkpoint load")
                continue
            tr_valid=~np.isnan(Y[train_idx,j]); te_valid=~np.isnan(Y[test_idx,j])
            ytr=Y[train_idx,j][tr_valid].astype(int)
            if np.unique(ytr).size<2:
                print(f"   ⚠️ {target}: training fold-এ দুই class নেই; skip")
                np.savez_compressed(unit_path,prob=fold_probs[:,j])
                continue
            model=build_classical_model(kind,ytr)
            try:
                model.fit(Xtr_all[tr_valid],ytr)
            except Exception as e:
                # XGBoost GPU compatibility fallback.
                if kind=="xgb" and "cuda" in str(e).lower():
                    print("   ⚠️ XGBoost GPU mode কাজ করেনি; CPU hist mode-এ retry হচ্ছে।")
                    model=build_classical_model(kind,ytr)
                    try: model.set_params(device="cpu")
                    except Exception: pass
                    model.fit(Xtr_all[tr_valid],ytr)
                else:
                    raise
            if te_valid.any():
                fold_probs[te_valid,j]=_predict_positive(model,Xte_all[te_valid]).astype(np.float32)
            np.savez_compressed(unit_path,prob=fold_probs[:,j])
            if SAVE_FITTED_CLASSICAL_MODELS:
                joblib.dump(model,MODEL_DIR/f"{slug}_fold{fold+1}_{model_slug(target)}.joblib",compress=3)
            print(f"   ✅ {target}: save complete")
            del model; gc.collect()

        oof[test_idx]=fold_probs
        part=metrics_from_predictions(test_idx,fold_probs,fold,model_name)
        metric_parts.append(part)
        print(f"✅ Fold {fold+1} সম্পন্ন | Macro AUC={part['AUC'].mean():.4f} | Macro Accuracy={part['Accuracy'].mean():.4f}")
        del Xtr_all,Xte_all; gc.collect()

    metrics=pd.concat(metric_parts,ignore_index=True)
    per_task,fold_macro,overall=save_model_outputs(model_name,oof,metrics)
    show_model_result(model_name,per_task,overall)
    return {"oof":oof,"metrics":metrics,"per_task":per_task,"fold_macro":fold_macro,"overall":overall}

print("✅ Classical ML CV engine প্রস্তুত হয়েছে।")
save_runtime_session()


## Model 1 — Random Forest


In [ ]:
if "RUNTIME_READY" not in globals():
    from pathlib import Path
    _candidates = [
        Path.cwd() / ".final_ultra_runtime_bootstrap.py",
        Path("D:/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/content/drive/MyDrive/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/mnt/data/Final_Ultra_Project/final_ultra/cache/runtime_bootstrap.py"),
        Path.cwd() / "final_ultra" / "cache" / "runtime_bootstrap.py",
        Path.cwd() / "cache" / "runtime_bootstrap.py",
    ]
    _bootstrap = next((p for p in _candidates if p.exists()), None)
    if _bootstrap is None:
        _hits = list(Path.cwd().rglob("runtime_bootstrap.py"))
        _bootstrap = _hits[0] if _hits else None
    if _bootstrap is None:
        raise RuntimeError("Recovery bootstrap পাওয়া যায়নি। প্রথমবার হলে setup থেকে Kernel-restart Recovery cell পর্যন্ত run করুন।")
    exec(compile(_bootstrap.read_text(encoding="utf-8"), str(_bootstrap), "exec"), globals())
    print("🔄 Kernel restart-এর পর runtime cache থেকে restore হয়েছে।")

RF_RESULT = run_classical_cv("Random Forest", "rf", "tree")


## Model 2 — Extra Trees


In [ ]:
if "RUNTIME_READY" not in globals():
    from pathlib import Path
    _candidates = [
        Path.cwd() / ".final_ultra_runtime_bootstrap.py",
        Path("D:/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/content/drive/MyDrive/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/mnt/data/Final_Ultra_Project/final_ultra/cache/runtime_bootstrap.py"),
        Path.cwd() / "final_ultra" / "cache" / "runtime_bootstrap.py",
        Path.cwd() / "cache" / "runtime_bootstrap.py",
    ]
    _bootstrap = next((p for p in _candidates if p.exists()), None)
    if _bootstrap is None:
        _hits = list(Path.cwd().rglob("runtime_bootstrap.py"))
        _bootstrap = _hits[0] if _hits else None
    if _bootstrap is None:
        raise RuntimeError("Recovery bootstrap পাওয়া যায়নি। প্রথমবার হলে setup থেকে Kernel-restart Recovery cell পর্যন্ত run করুন।")
    exec(compile(_bootstrap.read_text(encoding="utf-8"), str(_bootstrap), "exec"), globals())
    print("🔄 Kernel restart-এর পর runtime cache থেকে restore হয়েছে।")

ET_RESULT = run_classical_cv("Extra Trees", "et", "tree")


## Model 3 — XGBoost


In [ ]:
if "RUNTIME_READY" not in globals():
    from pathlib import Path
    _candidates = [
        Path.cwd() / ".final_ultra_runtime_bootstrap.py",
        Path("D:/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/content/drive/MyDrive/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/mnt/data/Final_Ultra_Project/final_ultra/cache/runtime_bootstrap.py"),
        Path.cwd() / "final_ultra" / "cache" / "runtime_bootstrap.py",
        Path.cwd() / "cache" / "runtime_bootstrap.py",
    ]
    _bootstrap = next((p for p in _candidates if p.exists()), None)
    if _bootstrap is None:
        _hits = list(Path.cwd().rglob("runtime_bootstrap.py"))
        _bootstrap = _hits[0] if _hits else None
    if _bootstrap is None:
        raise RuntimeError("Recovery bootstrap পাওয়া যায়নি। প্রথমবার হলে setup থেকে Kernel-restart Recovery cell পর্যন্ত run করুন।")
    exec(compile(_bootstrap.read_text(encoding="utf-8"), str(_bootstrap), "exec"), globals())
    print("🔄 Kernel restart-এর পর runtime cache থেকে restore হয়েছে।")

XGB_RESULT = run_classical_cv("XGBoost", "xgb", "tree")


## Model 4 — SVM with RBF Kernel

High-dimensional ECFP4 + descriptor input training-fold SVD দিয়ে reduce করা হয়, তারপর RBF-SVM fit হয়। AUC-এর জন্য decision score sigmoid mapping ব্যবহার করা হয়; এটি monotonic হওয়ায় ranking/AUC অপরিবর্তিত থাকে।


In [ ]:
if "RUNTIME_READY" not in globals():
    from pathlib import Path
    _candidates = [
        Path.cwd() / ".final_ultra_runtime_bootstrap.py",
        Path("D:/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/content/drive/MyDrive/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/mnt/data/Final_Ultra_Project/final_ultra/cache/runtime_bootstrap.py"),
        Path.cwd() / "final_ultra" / "cache" / "runtime_bootstrap.py",
        Path.cwd() / "cache" / "runtime_bootstrap.py",
    ]
    _bootstrap = next((p for p in _candidates if p.exists()), None)
    if _bootstrap is None:
        _hits = list(Path.cwd().rglob("runtime_bootstrap.py"))
        _bootstrap = _hits[0] if _hits else None
    if _bootstrap is None:
        raise RuntimeError("Recovery bootstrap পাওয়া যায়নি। প্রথমবার হলে setup থেকে Kernel-restart Recovery cell পর্যন্ত run করুন।")
    exec(compile(_bootstrap.read_text(encoding="utf-8"), str(_bootstrap), "exec"), globals())
    print("🔄 Kernel restart-এর পর runtime cache থেকে restore হয়েছে।")

SVM_RESULT = run_classical_cv("SVM-RBF", "svm", "svm")


# Deep Learning Infrastructure

All neural models use:

- 12-output (or active target count) multi-task prediction  
- masked weighted BCE loss  
- training-only positive weights  
- inner scaffold holdout for early stopping  
- best-checkpoint restoration  
- optional full outer-training refit for the selected epoch count  
- per-epoch checkpoint, so a restarted kernel resumes from the next epoch


In [ ]:
def make_inner_group_holdout(train_pool_idx, val_fraction=0.15, seed=SEED):
    rng=np.random.default_rng(seed)
    temp=df.iloc[train_pool_idx]
    groups=[]
    for scaf,part in temp.groupby("scaffold"):
        idx=part.index.to_numpy()
        pos=np.nansum(part[TARGETS].to_numpy(float)==1,axis=0)
        groups.append((scaf,idx,pos))
    rng.shuffle(groups)
    groups.sort(key=lambda x:(len(x[1])+2*np.sqrt(x[2].sum()+1)),reverse=True)
    desired=max(1,int(round(len(train_pool_idx)*val_fraction)))
    val=[]; val_pos=np.zeros(len(TARGETS)); total_pos=np.nansum(df.iloc[train_pool_idx][TARGETS].to_numpy(float)==1,axis=0)
    desired_pos=total_pos*val_fraction
    for _,idx,pos in groups:
        if len(val)>=desired and np.all(val_pos >= np.minimum(desired_pos,1)):
            break
        # Add group when it improves size/positive coverage.
        current=np.mean(((val_pos-desired_pos)/(desired_pos+1))**2)+((len(val)-desired)/max(desired,1))**2
        new=np.mean(((val_pos+pos-desired_pos)/(desired_pos+1))**2)+((len(val)+len(idx)-desired)/max(desired,1))**2
        if new<=current or len(val)<desired*0.65:
            val.extend(idx.tolist()); val_pos+=pos
    val_idx=np.array(sorted(set(val)),dtype=int)
    train_idx=np.array(sorted(set(train_pool_idx)-set(val_idx)),dtype=int)
    if len(train_idx)==0 or len(val_idx)==0:
        cut=max(1,int(len(train_pool_idx)*(1-val_fraction)))
        train_idx=np.asarray(train_pool_idx[:cut]); val_idx=np.asarray(train_pool_idx[cut:])
    return train_idx,val_idx

def compute_pos_weight(indices):
    out=[]
    for j in range(len(TARGETS)):
        y=Y[indices,j]; y=y[~np.isnan(y)]
        pos=max(1,np.sum(y==1)); neg=max(1,np.sum(y==0))
        out.append(min(25.0,max(1.0,float((neg/pos)**0.75))))
    return torch.tensor(out,dtype=torch.float32)

def masked_weighted_bce(logits,y,mask,pos_weight):
    loss=F.binary_cross_entropy_with_logits(logits,y,reduction="none",pos_weight=pos_weight)
    loss=loss*mask
    return loss.sum()/mask.sum().clamp_min(1.0)

class ArrayMultiTaskDataset(Dataset):
    def __init__(self,X,indices,augment=None):
        self.X=X; self.indices=np.asarray(indices,dtype=int); self.augment=augment
    def __len__(self): return len(self.indices)
    def __getitem__(self,k):
        idx=int(self.indices[k]); x=np.asarray(self.X[k],dtype=np.float32)
        if self.augment is not None: x=self.augment(x)
        return {"x":torch.from_numpy(np.ascontiguousarray(x)),"y":torch.from_numpy(Y_TENSOR_SAFE[idx]),
                "mask":torch.from_numpy(LABEL_MASK[idx]),"idx":torch.tensor(idx,dtype=torch.long)}

def make_loader(dataset,batch_size,shuffle):
    return DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,num_workers=RESOURCE["num_workers"],
                      pin_memory=(DEVICE.type=="cuda"),drop_last=False,persistent_workers=(RESOURCE["num_workers"]>0))

def move_batch(batch):
    return {k:(v.to(DEVICE,non_blocking=True) if torch.is_tensor(v) else v) for k,v in batch.items()}

def torch_load_compat(path,map_location=DEVICE):
    try: return torch.load(path,map_location=map_location,weights_only=False)
    except TypeError: return torch.load(path,map_location=map_location)

def predict_loader(model,loader,forward_fn):
    model.eval(); probs=[]; indices=[]
    with torch.no_grad():
        for batch in loader:
            batch=move_batch(batch)
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=AMP_ENABLED):
                logits=forward_fn(model,batch)
            probs.append(torch.sigmoid(logits).detach().cpu().numpy())
            indices.append(batch["idx"].detach().cpu().numpy())
    return np.concatenate(indices),np.concatenate(probs)

def macro_auc_loader(model,loader,forward_fn):
    idx,prob=predict_loader(model,loader,forward_fn)
    vals=[]
    for j in range(len(TARGETS)):
        valid=LABEL_MASK[idx,j].astype(bool)
        if valid.any(): vals.append(safe_auc(Y[idx,j][valid],prob[valid,j]))
    return float(np.nanmean(vals)) if vals else np.nan

def _new_grad_scaler():
    try: return torch.amp.GradScaler("cuda",enabled=AMP_ENABLED)
    except Exception: return torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

def train_with_early_stopping(model,train_loader,val_loader,forward_fn,pos_weight,fold_dir,max_epochs,patience,lr=3e-4,weight_decay=1e-5):
    fold_dir.mkdir(parents=True,exist_ok=True)
    last_path=fold_dir/"selection_last.pt"; best_path=fold_dir/"selection_best.pt"; done_path=fold_dir/"selection_done.json"
    optimizer=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=weight_decay)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="max",factor=0.5,patience=max(2,patience//3),min_lr=1e-6)
    scaler=_new_grad_scaler(); start=0; best=-np.inf; wait=0; history=[]; best_epoch=1
    if last_path.exists() and not FORCE_REBUILD:
        ck=torch_load_compat(last_path)
        model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
        try: scheduler.load_state_dict(ck["scheduler"])
        except Exception: pass
        try: scaler.load_state_dict(ck["scaler"])
        except Exception: pass
        start=int(ck["epoch"])+1; best=float(ck["best"]); wait=int(ck["wait"]); history=ck.get("history",[]); best_epoch=int(ck.get("best_epoch",1))
        print(f"      🔄 Epoch checkpoint restore: {start}/{max_epochs}")
    pos_weight=pos_weight.to(DEVICE)
    for epoch in range(start,max_epochs):
        model.train(); losses=[]
        for batch in train_loader:
            batch=move_batch(batch); optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=AMP_ENABLED):
                logits=forward_fn(model,batch)
                loss=masked_weighted_bce(logits,batch["y"],batch["mask"],pos_weight)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            scaler.step(optimizer); scaler.update(); losses.append(float(loss.detach().cpu()))
        val_auc=macro_auc_loader(model,val_loader,forward_fn)
        scheduler.step(val_auc if np.isfinite(val_auc) else -1)
        train_loss=float(np.mean(losses)) if losses else np.nan
        history.append({"epoch":epoch+1,"train_loss":train_loss,"val_auc":val_auc,"lr":optimizer.param_groups[0]["lr"]})
        improved=np.isfinite(val_auc) and val_auc>best+1e-5
        if improved:
            best=val_auc; wait=0; best_epoch=epoch+1
            torch.save({"model":model.state_dict(),"epoch":epoch,"best":best},best_path)
        else: wait+=1
        torch.save({"epoch":epoch,"model":model.state_dict(),"optimizer":optimizer.state_dict(),"scheduler":scheduler.state_dict(),
                    "scaler":scaler.state_dict(),"best":best,"wait":wait,"history":history,"best_epoch":best_epoch},last_path)
        print(f"      Epoch {epoch+1:03d} | loss={train_loss:.5f} | val AUC={val_auc:.4f} | best={best:.4f}")
        if wait>=patience:
            print("      ⏹️ Early stopping triggered."); break
    if best_path.exists(): model.load_state_dict(torch_load_compat(best_path)["model"])
    done_path.write_text(json.dumps({"best_epoch":best_epoch,"best_val_auc":best,"history":history},default=float))
    return best_epoch,best,history

def train_fixed_epochs(model,train_loader,forward_fn,pos_weight,fold_dir,epochs,lr=3e-4,weight_decay=1e-5):
    fold_dir.mkdir(parents=True,exist_ok=True)
    last_path=fold_dir/"refit_last.pt"; done_path=fold_dir/"refit_done.json"; final_path=fold_dir/"refit_final.pt"
    optimizer=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=weight_decay); scaler=_new_grad_scaler(); start=0; history=[]
    if last_path.exists() and not FORCE_REBUILD:
        ck=torch_load_compat(last_path); model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
        try: scaler.load_state_dict(ck["scaler"])
        except Exception: pass
        start=int(ck["epoch"])+1; history=ck.get("history",[]); print(f"      🔄 Refit checkpoint restore: {start}/{epochs}")
    pos_weight=pos_weight.to(DEVICE)
    for epoch in range(start,epochs):
        model.train(); losses=[]
        for batch in train_loader:
            batch=move_batch(batch); optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=AMP_ENABLED):
                logits=forward_fn(model,batch); loss=masked_weighted_bce(logits,batch["y"],batch["mask"],pos_weight)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            scaler.step(optimizer); scaler.update(); losses.append(float(loss.detach().cpu()))
        history.append({"epoch":epoch+1,"train_loss":float(np.mean(losses))})
        torch.save({"epoch":epoch,"model":model.state_dict(),"optimizer":optimizer.state_dict(),"scaler":scaler.state_dict(),"history":history},last_path)
        print(f"      Refit epoch {epoch+1:03d}/{epochs} | loss={history[-1]['train_loss']:.5f}")
    torch.save({"model":model.state_dict(),"epochs":epochs},final_path)
    done_path.write_text(json.dumps({"epochs":epochs,"history":history},default=float))
    return history

print("✅ Deep Learning training, early stopping এবং per-epoch resume engine প্রস্তুত হয়েছে।")


In [ ]:
class TabularMTDNN(nn.Module):
    def __init__(self,input_dim,hidden_dims,dropouts,input_dropout=0.1):
        super().__init__(); self.input_dropout=nn.Dropout(input_dropout); blocks=[]; prev=input_dim
        for h,d in zip(hidden_dims,dropouts):
            blocks += [nn.Linear(prev,h),nn.LayerNorm(h),nn.ReLU(),nn.Dropout(d)]
            prev=h
        self.backbone=nn.Sequential(*blocks); self.out=nn.Linear(prev,len(TARGETS))
    def forward(self,x): return self.out(self.backbone(self.input_dropout(x)))

def build_tabular_resources(train_idx,val_idx,test_idx,fold,stage,architecture):
    compact=architecture=="compact"
    mode="dnn"
    Xtr,Xva,art=prepare_tabular_fold(train_idx,val_idx,mode=mode,compact=compact)
    _,Xte,_=prepare_tabular_fold(train_idx,test_idx,mode=mode,compact=compact)
    # The second call above fits an equivalent train-only transformer. For exact consistency, transform test with the first artifacts:
    desc_cols=COMPACT_DESC_IDX if compact else list(range(len(DESC_NAMES)))
    dte=art["imputer"].transform(np.asarray(X_DESC[test_idx])[:,desc_cols])
    dte=art["scaler"].transform(dte)
    Xte=np.concatenate([np.asarray(X_ECFP[test_idx],dtype=np.float32),dte.astype(np.float32)],axis=1).astype(np.float32)
    if SMOKE_TEST:
        hidden=[64,32]; drops=[0.15,0.10]
    else:
        hidden=[1024,512,128] if compact else [2048,1024,512]
        drops=[0.30,0.30,0.25] if compact else [0.35,0.40,0.35]
    model=TabularMTDNN(Xtr.shape[1],hidden,drops,input_dropout=0.08).to(DEVICE)
    return {
        "model":model,
        "train_loader":make_loader(ArrayMultiTaskDataset(Xtr,train_idx),RESOURCE["tab_batch"],True),
        "val_loader":make_loader(ArrayMultiTaskDataset(Xva,val_idx),RESOURCE["tab_batch"],False),
        "test_loader":make_loader(ArrayMultiTaskDataset(Xte,test_idx),RESOURCE["tab_batch"],False),
        "forward_fn":lambda m,b:m(b["x"]),
        "pos_weight":compute_pos_weight(train_idx),
    }

def run_torch_cv(model_name,resource_builder,max_epochs,lr=3e-4,weight_decay=1e-5):
    slug=model_slug(model_name); pred_dir=PRED_DIR/slug; pred_dir.mkdir(parents=True,exist_ok=True)
    histories={}; oof=np.full((len(df),len(TARGETS)),np.nan,dtype=np.float32); metric_parts=[]
    for fold in range(N_FOLDS):
        fold_pred=pred_dir/f"fold{fold+1}.npz"
        if fold_pred.exists() and not FORCE_REBUILD:
            z=np.load(fold_pred); idx=z["idx"].astype(int); prob=z["prob"]
            oof[idx]=prob; metric_parts.append(metrics_from_predictions(idx,prob,fold,model_name))
            print(f"♻️ {model_name} Fold {fold+1}: completed prediction checkpoint load")
            continue
        print(f"\n📘 {model_name} | Fold {fold+1}/{N_FOLDS} শুরু")
        test_idx=np.where(df["fold"].to_numpy()==fold)[0]
        train_pool=np.where(df["fold"].to_numpy()!=fold)[0]
        inner_train,val_idx=make_inner_group_holdout(train_pool,0.15,SEED+fold)
        fold_dir=MODEL_DIR/slug/f"fold{fold+1}"

        res=resource_builder(inner_train,val_idx,test_idx,fold,"selection")
        best_epoch,best_val,hist=train_with_early_stopping(
            res["model"],res["train_loader"],res["val_loader"],res["forward_fn"],res["pos_weight"],
            fold_dir,max_epochs,RESOURCE["patience"],lr,weight_decay
        )
        histories[f"fold{fold+1}_selection"]={"best_epoch":best_epoch,"best_val_auc":best_val,"history":hist}

        if REFIT_DEEP_ON_FULL_OUTER_TRAIN:
            # Use a tiny validation placeholder only to build consistent resources; refit uses full train loader.
            placeholder=val_idx[:max(1,min(8,len(val_idx)))]
            full=resource_builder(train_pool,placeholder,test_idx,fold,"refit")
            refit_epochs=(1 if SMOKE_TEST else max(2,best_epoch))
            refit_hist=train_fixed_epochs(full["model"],full["train_loader"],full["forward_fn"],full["pos_weight"],fold_dir,refit_epochs,lr,weight_decay)
            model=full["model"]; test_loader=full["test_loader"]; forward_fn=full["forward_fn"]
            histories[f"fold{fold+1}_refit"]={"epochs":refit_epochs,"history":refit_hist}
        else:
            model=res["model"]; test_loader=res["test_loader"]; forward_fn=res["forward_fn"]

        idx,prob=predict_loader(model,test_loader,forward_fn)
        order=np.argsort(idx); idx=idx[order]; prob=prob[order]
        np.savez_compressed(fold_pred,idx=idx,prob=prob)
        oof[idx]=prob; part=metrics_from_predictions(idx,prob,fold,model_name); metric_parts.append(part)
        print(f"✅ Fold {fold+1} prediction save হয়েছে | Macro AUC={part['AUC'].mean():.4f}")
        del res,model,test_loader; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    metrics=pd.concat(metric_parts,ignore_index=True)
    per_task,fold_macro,overall=save_model_outputs(model_name,oof,metrics,histories)
    show_model_result(model_name,per_task,overall)
    return {"oof":oof,"metrics":metrics,"per_task":per_task,"fold_macro":fold_macro,"overall":overall,"histories":histories}

print("✅ Multi-task tabular neural architecture এবং CV runner প্রস্তুত হয়েছে।")
save_runtime_session()


## Model 5 — DeepTox-style Multi-task DNN

Architecture: ECFP4 + 13 descriptors → 2048 → 1024 → 512 → multi-task logits, with ReLU, LayerNorm, dropout, AdamW, masked class-weighted BCE and early stopping.


In [ ]:
if "RUNTIME_READY" not in globals():
    from pathlib import Path
    _candidates = [
        Path.cwd() / ".final_ultra_runtime_bootstrap.py",
        Path("D:/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/content/drive/MyDrive/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/mnt/data/Final_Ultra_Project/final_ultra/cache/runtime_bootstrap.py"),
        Path.cwd() / "final_ultra" / "cache" / "runtime_bootstrap.py",
        Path.cwd() / "cache" / "runtime_bootstrap.py",
    ]
    _bootstrap = next((p for p in _candidates if p.exists()), None)
    if _bootstrap is None:
        _hits = list(Path.cwd().rglob("runtime_bootstrap.py"))
        _bootstrap = _hits[0] if _hits else None
    if _bootstrap is None:
        raise RuntimeError("Recovery bootstrap পাওয়া যায়নি। প্রথমবার হলে setup থেকে Kernel-restart Recovery cell পর্যন্ত run করুন।")
    exec(compile(_bootstrap.read_text(encoding="utf-8"), str(_bootstrap), "exec"), globals())
    print("🔄 Kernel restart-এর পর runtime cache থেকে restore হয়েছে।")

DEEPTOX_RESULT = run_torch_cv(
    "DeepTox-style MT-DNN",
    lambda tr,va,te,f,s: build_tabular_resources(tr,va,te,f,s,"deeptox"),
    RESOURCE["dnn_epochs"], lr=3e-4, weight_decay=1e-5
)


## Model 6 — Multitask CapsNet + RBM + Dynamic Routing

Paper-inspired input: MACCS 166 bits + 13 descriptors = 179 features. Two Bernoulli RBMs (256 and 128 hidden units) are fitted only on the current training data. Their weights initialize the shared encoder. Primary capsules and task capsules use two routing iterations.


In [ ]:
def squash_caps(s,dim=-1,eps=1e-8):
    norm2=(s*s).sum(dim=dim,keepdim=True)
    scale=norm2/(1.0+norm2)/torch.sqrt(norm2+eps)
    return scale*s

class DynamicTaskCapsules(nn.Module):
    def __init__(self,n_primary=8,in_dim=8,n_tasks=None,out_dim=8,routing_iters=2):
        super().__init__(); n_tasks=n_tasks or len(TARGETS)
        self.n_primary=n_primary; self.n_tasks=n_tasks; self.out_dim=out_dim; self.routing_iters=routing_iters
        self.W=nn.Parameter(torch.empty(n_primary,n_tasks,out_dim,in_dim)); nn.init.xavier_uniform_(self.W)
    def forward(self,u):
        # u: [B, primary, in_dim], u_hat: [B, primary, task, out_dim]
        u_hat=torch.einsum("bpi,ptoi->bpto",u,self.W)
        b=torch.zeros(u.shape[0],self.n_primary,self.n_tasks,device=u.device,dtype=u.dtype)
        for r in range(self.routing_iters):
            c=torch.softmax(b,dim=2)
            s=(c.unsqueeze(-1)*u_hat).sum(dim=1)
            v=squash_caps(s)
            if r<self.routing_iters-1:
                b=b+(u_hat*v.unsqueeze(1)).sum(dim=-1)
        return v

class RBMCapsNet(nn.Module):
    def __init__(self,rbm1=None,rbm2=None):
        super().__init__(); self.enc1=nn.Linear(179,256); self.enc2=nn.Linear(256,128); self.dropout=nn.Dropout(0.25)
        if rbm1 is not None:
            self.enc1.weight.data.copy_(torch.tensor(rbm1.components_,dtype=torch.float32))
            self.enc1.bias.data.copy_(torch.tensor(rbm1.intercept_hidden_,dtype=torch.float32))
        if rbm2 is not None:
            self.enc2.weight.data.copy_(torch.tensor(rbm2.components_,dtype=torch.float32))
            self.enc2.bias.data.copy_(torch.tensor(rbm2.intercept_hidden_,dtype=torch.float32))
        self.primary=nn.Linear(128,8*8)
        self.routing=DynamicTaskCapsules(8,8,len(TARGETS),8,2)
        self.task_head=nn.Linear(8,1)
    def forward(self,x):
        h=torch.sigmoid(self.enc1(x)); h=self.dropout(h); h=torch.sigmoid(self.enc2(h));
        u=squash_caps(self.primary(h).view(-1,8,8)); v=self.routing(u)
        return self.task_head(v).squeeze(-1)

def fit_or_load_rbms(Xtrain,cache_prefix):
    p1=Path(str(cache_prefix)+"_rbm1.joblib"); p2=Path(str(cache_prefix)+"_rbm2.joblib")
    if p1.exists() and p2.exists() and not FORCE_REBUILD:
        return joblib.load(p1),joblib.load(p2)
    rbm1=BernoulliRBM(n_components=256,learning_rate=0.02,batch_size=64,n_iter=1 if SMOKE_TEST else 15,verbose=0,random_state=SEED)
    h1=rbm1.fit_transform(Xtrain)
    rbm2=BernoulliRBM(n_components=128,learning_rate=0.02,batch_size=64,n_iter=1 if SMOKE_TEST else 12,verbose=0,random_state=SEED+1)
    rbm2.fit(h1)
    joblib.dump(rbm1,p1,compress=3); joblib.dump(rbm2,p2,compress=3)
    return rbm1,rbm2

def build_caps_resources(train_idx,val_idx,test_idx,fold,stage):
    Xtr,Xva,art=prepare_capsnet_fold(train_idx,val_idx)
    # Transform test with train-fitted artifacts.
    dte=art["imputer"].transform(np.asarray(X_DESC[test_idx],dtype=np.float32)); dte=art["scaler"].transform(dte)
    Xte=np.concatenate([np.asarray(X_MACCS[test_idx,1:],dtype=np.float32),dte],axis=1).astype(np.float32)
    prefix=MODEL_DIR/model_slug("Multitask CapsNet + RBM")/f"fold{fold+1}_{stage}"
    prefix.parent.mkdir(parents=True,exist_ok=True)
    rbm1,rbm2=fit_or_load_rbms(Xtr,prefix)
    model=RBMCapsNet(rbm1,rbm2).to(DEVICE)
    return {"model":model,"train_loader":make_loader(ArrayMultiTaskDataset(Xtr,train_idx),RESOURCE["tab_batch"],True),
            "val_loader":make_loader(ArrayMultiTaskDataset(Xva,val_idx),RESOURCE["tab_batch"],False),
            "test_loader":make_loader(ArrayMultiTaskDataset(Xte,test_idx),RESOURCE["tab_batch"],False),
            "forward_fn":lambda m,b:m(b["x"]),"pos_weight":compute_pos_weight(train_idx)}

print("✅ RBM-initialized Multitask CapsNet architecture প্রস্তুত হয়েছে।")
save_runtime_session()


In [ ]:
if "RUNTIME_READY" not in globals():
    from pathlib import Path
    _candidates = [
        Path.cwd() / ".final_ultra_runtime_bootstrap.py",
        Path("D:/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/content/drive/MyDrive/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/mnt/data/Final_Ultra_Project/final_ultra/cache/runtime_bootstrap.py"),
        Path.cwd() / "final_ultra" / "cache" / "runtime_bootstrap.py",
        Path.cwd() / "cache" / "runtime_bootstrap.py",
    ]
    _bootstrap = next((p for p in _candidates if p.exists()), None)
    if _bootstrap is None:
        _hits = list(Path.cwd().rglob("runtime_bootstrap.py"))
        _bootstrap = _hits[0] if _hits else None
    if _bootstrap is None:
        raise RuntimeError("Recovery bootstrap পাওয়া যায়নি। প্রথমবার হলে setup থেকে Kernel-restart Recovery cell পর্যন্ত run করুন।")
    exec(compile(_bootstrap.read_text(encoding="utf-8"), str(_bootstrap), "exec"), globals())
    print("🔄 Kernel restart-এর পর runtime cache থেকে restore হয়েছে।")

CAPSNET_RESULT = run_torch_cv(
    "Multitask CapsNet + RBM",
    build_caps_resources,
    RESOURCE["caps_epochs"], lr=2e-4, weight_decay=1e-5
)


## Model 7 — Multi-channel 2D CNN

The paper used six physicochemical 2D grids and proprietary/external calculations. For a fully reproducible notebook, the following transparent RDKit approximation is used:

1. H-bond donor/acceptor field  
2. Hydrophobic atom field  
3. Metal atom field  
4. Excluded-volume / van der Waals field  
5. Positive-ionizable field  
6. Negative-ionizable field

Grid: 24 Å × 24 Å, 0.5 Å resolution → 48 × 48. Grid tensors are float16 memory maps and generation is checkpointed every 50 molecules.


In [ ]:
FEATURE_FACTORY = ChemicalFeatures.BuildFeatureFactory(str(Path(RDConfig.RDDataDir)/"BaseFeatures.fdef"))
GRID_SIZE=48; GRID_HALF=12.0; GRID_AXIS=np.linspace(-GRID_HALF,GRID_HALF,GRID_SIZE,dtype=np.float32)
GX,GY=np.meshgrid(GRID_AXIS,GRID_AXIS,indexing="xy")
METALS={3,4,11,12,13,19,20,21,22,23,24,25,26,27,28,29,30,31,37,38,39,40,41,42,43,44,45,46,47,48,49,50,55,56,57,78,79,80}

def molecule_to_six_channel_grid(smiles):
    mol=Chem.MolFromSmiles(smiles)
    if mol is None: return np.zeros((6,GRID_SIZE,GRID_SIZE),dtype=np.float16)
    AllChem.Compute2DCoords(mol)
    conf=mol.GetConformer(); coords=np.array([[conf.GetAtomPosition(i).x,conf.GetAtomPosition(i).y] for i in range(mol.GetNumAtoms())],dtype=np.float32)
    coords-=coords.mean(axis=0,keepdims=True)
    max_abs=max(1.0,float(np.abs(coords).max()))
    if max_abs>10.5: coords*=10.5/max_abs
    fam_atoms={"Donor":set(),"Acceptor":set(),"Hydrophobe":set(),"LumpedHydrophobe":set(),"PosIonizable":set(),"NegIonizable":set()}
    try:
        for feat in FEATURE_FACTORY.GetFeaturesForMol(mol):
            if feat.GetFamily() in fam_atoms: fam_atoms[feat.GetFamily()].update(feat.GetAtomIds())
    except Exception: pass
    grid=np.zeros((6,GRID_SIZE,GRID_SIZE),dtype=np.float32)
    pt=Chem.GetPeriodicTable()
    for i,atom in enumerate(mol.GetAtoms()):
        x,y=coords[i]; d2=(GX-x)**2+(GY-y)**2
        z=atom.GetAtomicNum(); r=max(0.7,float(pt.GetRvdw(z)))
        g=np.exp(-d2/(2*(0.55+0.15*r)**2))
        if i in fam_atoms["Donor"]: grid[0]+=1.0*g
        if i in fam_atoms["Acceptor"]: grid[0]+=0.85*g
        if i in fam_atoms["Hydrophobe"] or i in fam_atoms["LumpedHydrophobe"] or z in {6,9,17,35,53}: grid[1]+=g
        if z in METALS: grid[2]+=1.2*g
        grid[3]+=r*g
        if i in fam_atoms["PosIonizable"] or atom.GetFormalCharge()>0: grid[4]+=g
        if i in fam_atoms["NegIonizable"] or atom.GetFormalCharge()<0: grid[5]+=g
    for c in range(6):
        mx=float(np.max(np.abs(grid[c])))
        if mx>0: grid[c]/=mx
    return grid.astype(np.float16)

GRID_PATH=FEATURE_ROOT/"six_channel_48x48_float16.npy"
GRID_PROGRESS=FEATURE_ROOT/"grid_progress.json"

def build_or_load_grids():
    valid=False
    if GRID_PATH.exists() and not FORCE_REBUILD:
        try: valid=np.load(GRID_PATH,mmap_mode="r").shape==(len(df),6,GRID_SIZE,GRID_SIZE)
        except Exception: valid=False
    if not valid:
        if FORCE_REBUILD: GRID_PATH.unlink(missing_ok=True); GRID_PROGRESS.unlink(missing_ok=True)
        if not GRID_PATH.exists():
            arr=np.lib.format.open_memmap(GRID_PATH,mode="w+",dtype=np.float16,shape=(len(df),6,GRID_SIZE,GRID_SIZE)); start=0
        else:
            arr=np.lib.format.open_memmap(GRID_PATH,mode="r+"); start=json.loads(GRID_PROGRESS.read_text())["next_index"] if GRID_PROGRESS.exists() else 0
            print(f"🔄 Grid checkpoint পাওয়া গেছে। Molecule {start} থেকে resume হচ্ছে।")
        for i in tqdm(range(start,len(df)),desc="Six-channel molecular grids"):
            arr[i]=molecule_to_six_channel_grid(df.iloc[i]["std_smiles"])
            if (i+1)%50==0 or i+1==len(df):
                arr.flush(); GRID_PROGRESS.write_text(json.dumps({"next_index":i+1}))
        GRID_PROGRESS.unlink(missing_ok=True); del arr
        print("✅ Six-channel grid generation সম্পন্ন হয়েছে।")
    else: print("♻️ Cached six-channel grids পাওয়া গেছে।")
    return np.load(GRID_PATH,mmap_mode="r")

print("✅ Six-channel grid generator প্রস্তুত হয়েছে।")
save_runtime_session()


In [ ]:
if "RUNTIME_READY" not in globals():
    from pathlib import Path
    _candidates = [
        Path.cwd() / ".final_ultra_runtime_bootstrap.py",
        Path("D:/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/content/drive/MyDrive/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/mnt/data/Final_Ultra_Project/final_ultra/cache/runtime_bootstrap.py"),
        Path.cwd() / "final_ultra" / "cache" / "runtime_bootstrap.py",
        Path.cwd() / "cache" / "runtime_bootstrap.py",
    ]
    _bootstrap = next((p for p in _candidates if p.exists()), None)
    if _bootstrap is None:
        _hits = list(Path.cwd().rglob("runtime_bootstrap.py"))
        _bootstrap = _hits[0] if _hits else None
    if _bootstrap is None:
        raise RuntimeError("Recovery bootstrap পাওয়া যায়নি। প্রথমবার হলে setup থেকে Kernel-restart Recovery cell পর্যন্ত run করুন।")
    exec(compile(_bootstrap.read_text(encoding="utf-8"), str(_bootstrap), "exec"), globals())
    print("🔄 Kernel restart-এর পর runtime cache থেকে restore হয়েছে।")

X_GRIDS = build_or_load_grids()
fig,axes=plt.subplots(2,3,figsize=(13,8))
channel_names=["H-bond","Hydrophobicity","Metallicity","Excluded volume","Positive ionization","Negative ionization"]
for ax,img,name in zip(axes.flat,np.asarray(X_GRIDS[0]),channel_names):
    im=ax.imshow(img,cmap="magma",origin="lower"); ax.set_title(name); ax.axis("off"); fig.colorbar(im,ax=ax,fraction=.046,pad=.03)
plt.suptitle("Example Six-channel Molecular Grid",fontsize=15,fontweight="bold"); plt.tight_layout()
plt.savefig(FIG_DIR/"05_six_channel_grid_example.png",dpi=220,bbox_inches="tight"); plt.show(); plt.close()
print("✅ Grid cache load এবং example visualization সম্পন্ন হয়েছে।")


In [ ]:
class GridDataset(Dataset):
    def __init__(self,grid_path,indices,augment=False): self.grid_path=grid_path; self.indices=np.asarray(indices); self.augment=augment; self._grid=None
    def _arr(self):
        if self._grid is None: self._grid=np.load(self.grid_path,mmap_mode="r")
        return self._grid
    def __len__(self): return len(self.indices)
    def __getitem__(self,k):
        idx=int(self.indices[k]); x=np.asarray(self._arr()[idx],dtype=np.float32)
        if self.augment:
            rot=np.random.randint(0,4); x=np.rot90(x,rot,axes=(1,2)).copy()
            if np.random.rand()<.5: x=x[:,:,::-1].copy()
        return {"x":torch.from_numpy(np.ascontiguousarray(x)),"y":torch.from_numpy(Y_TENSOR_SAFE[idx]),
                "mask":torch.from_numpy(LABEL_MASK[idx]),"idx":torch.tensor(idx,dtype=torch.long)}

class MultiChannelCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(6,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),
            nn.Conv2d(128,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),nn.AdaptiveAvgPool2d(1)
        )
        self.head=nn.Sequential(nn.Flatten(),nn.Linear(128,256),nn.ReLU(),nn.Dropout(.35),nn.Linear(256,len(TARGETS)))
    def forward(self,x): return self.head(self.features(x))

def build_cnn_resources(train_idx,val_idx,test_idx,fold,stage):
    model=MultiChannelCNN().to(DEVICE)
    return {"model":model,"train_loader":make_loader(GridDataset(GRID_PATH,train_idx,True),RESOURCE["cnn_batch"],True),
            "val_loader":make_loader(GridDataset(GRID_PATH,val_idx,False),RESOURCE["cnn_batch"],False),
            "test_loader":make_loader(GridDataset(GRID_PATH,test_idx,False),RESOURCE["cnn_batch"],False),
            "forward_fn":lambda m,b:m(b["x"]),"pos_weight":compute_pos_weight(train_idx)}

print("✅ Multi-channel 2D CNN architecture প্রস্তুত হয়েছে।")
save_runtime_session()


In [ ]:
if "RUNTIME_READY" not in globals():
    from pathlib import Path
    _candidates = [
        Path.cwd() / ".final_ultra_runtime_bootstrap.py",
        Path("D:/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/content/drive/MyDrive/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/mnt/data/Final_Ultra_Project/final_ultra/cache/runtime_bootstrap.py"),
        Path.cwd() / "final_ultra" / "cache" / "runtime_bootstrap.py",
        Path.cwd() / "cache" / "runtime_bootstrap.py",
    ]
    _bootstrap = next((p for p in _candidates if p.exists()), None)
    if _bootstrap is None:
        _hits = list(Path.cwd().rglob("runtime_bootstrap.py"))
        _bootstrap = _hits[0] if _hits else None
    if _bootstrap is None:
        raise RuntimeError("Recovery bootstrap পাওয়া যায়নি। প্রথমবার হলে setup থেকে Kernel-restart Recovery cell পর্যন্ত run করুন।")
    exec(compile(_bootstrap.read_text(encoding="utf-8"), str(_bootstrap), "exec"), globals())
    print("🔄 Kernel restart-এর পর runtime cache থেকে restore হয়েছে।")

CNN_RESULT = run_torch_cv(
    "Multi-channel 2D CNN",
    build_cnn_resources,
    RESOURCE["cnn_epochs"], lr=2e-4, weight_decay=1e-5
)


## Model 8 — Compact MTL-DNN

Architecture: ECFP4 2048 + 7 fixed descriptors → 1024 → 512 → 128 → multi-task logits. This follows the compact MTL-DNN paper setup while retaining strict masking and fold-safe preprocessing.


In [ ]:
if "RUNTIME_READY" not in globals():
    from pathlib import Path
    _candidates = [
        Path.cwd() / ".final_ultra_runtime_bootstrap.py",
        Path("D:/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/content/drive/MyDrive/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/mnt/data/Final_Ultra_Project/final_ultra/cache/runtime_bootstrap.py"),
        Path.cwd() / "final_ultra" / "cache" / "runtime_bootstrap.py",
        Path.cwd() / "cache" / "runtime_bootstrap.py",
    ]
    _bootstrap = next((p for p in _candidates if p.exists()), None)
    if _bootstrap is None:
        _hits = list(Path.cwd().rglob("runtime_bootstrap.py"))
        _bootstrap = _hits[0] if _hits else None
    if _bootstrap is None:
        raise RuntimeError("Recovery bootstrap পাওয়া যায়নি। প্রথমবার হলে setup থেকে Kernel-restart Recovery cell পর্যন্ত run করুন।")
    exec(compile(_bootstrap.read_text(encoding="utf-8"), str(_bootstrap), "exec"), globals())
    print("🔄 Kernel restart-এর পর runtime cache থেকে restore হয়েছে।")

COMPACT_RESULT = run_torch_cv(
    "Compact MTL-DNN",
    lambda tr,va,te,f,s: build_tabular_resources(tr,va,te,f,s,"compact"),
    RESOURCE["dnn_epochs"], lr=1e-4, weight_decay=1e-5
)


## Model 9 — Molecular R-GCN + Fingerprint/Descriptor Fusion

### Important scientific distinction

The uploaded KG-GNN paper's R-GCN used a large external ToxKG containing chemical-gene-pathway relations and a matched 6,565-compound subset. Those KG triples were not included in the uploaded files. Therefore, claiming the paper's ≈0.925 mean AUC from `tox21.csv` alone would be scientifically invalid.

This notebook implements a real relation-specific graph convolutional network where molecular **bond types are relations** (single, double, triple, aromatic, other). Rich atom features are learned by R-GCN layers and fused with ECFP4 + 13 fold-standardized descriptors. This model is fully reproducible from the uploaded CSV, but its score must be measured rather than assumed.


In [ ]:
ATOM_NUMS=[1,5,6,7,8,9,14,15,16,17,35,53,3,4,11,12,19,20,26,30]
HYBRIDS=[Chem.rdchem.HybridizationType.SP,Chem.rdchem.HybridizationType.SP2,Chem.rdchem.HybridizationType.SP3,
         Chem.rdchem.HybridizationType.SP3D,Chem.rdchem.HybridizationType.SP3D2,Chem.rdchem.HybridizationType.UNSPECIFIED]
CHIRALS=[Chem.rdchem.ChiralType.CHI_UNSPECIFIED,Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
         Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,Chem.rdchem.ChiralType.CHI_OTHER]

def one_hot_unknown(value,choices):
    return [float(value==x) for x in choices]+[float(value not in choices)]

def atom_feature(atom):
    feat=[]
    feat+=one_hot_unknown(atom.GetAtomicNum(),ATOM_NUMS)
    feat+=one_hot_unknown(atom.GetDegree(),list(range(6)))
    feat+=one_hot_unknown(atom.GetFormalCharge(),[-2,-1,0,1,2])
    feat+=one_hot_unknown(atom.GetHybridization(),HYBRIDS[:-1])
    feat+=one_hot_unknown(atom.GetChiralTag(),CHIRALS[:-1])
    feat += [float(atom.GetIsAromatic()),float(atom.IsInRing()),float(atom.GetMass()/200.0),float(atom.GetTotalValence()/8.0)]
    return np.asarray(feat,dtype=np.float32)

def bond_relation(bond):
    bt=bond.GetBondType()
    if bt==Chem.rdchem.BondType.SINGLE:return 0
    if bt==Chem.rdchem.BondType.DOUBLE:return 1
    if bt==Chem.rdchem.BondType.TRIPLE:return 2
    if bt==Chem.rdchem.BondType.AROMATIC:return 3
    return 4

def smiles_to_rel_graph(smiles):
    mol=Chem.MolFromSmiles(smiles)
    x=np.stack([atom_feature(a) for a in mol.GetAtoms()]).astype(np.float32)
    src=[];dst=[];rel=[]
    for b in mol.GetBonds():
        i=b.GetBeginAtomIdx();j=b.GetEndAtomIdx();r=bond_relation(b)
        src += [i,j]; dst += [j,i]; rel += [r,r]
    edge_index=np.asarray([src,dst],dtype=np.int64)
    edge_type=np.asarray(rel,dtype=np.int64)
    return {"x":x,"edge_index":edge_index,"edge_type":edge_type}

GRAPH_ROOT=FEATURE_ROOT/"rel_graph_shards"; GRAPH_ROOT.mkdir(parents=True,exist_ok=True)
GRAPH_MANIFEST=GRAPH_ROOT/"manifest.json"; GRAPH_SHARD_SIZE=500

def build_graph_cache():
    n_shards=math.ceil(len(df)/GRAPH_SHARD_SIZE)
    for s in range(n_shards):
        path=GRAPH_ROOT/f"shard_{s:03d}.joblib"
        if path.exists() and not FORCE_REBUILD: continue
        lo=s*GRAPH_SHARD_SIZE; hi=min(len(df),(s+1)*GRAPH_SHARD_SIZE)
        graphs=[]
        for i in tqdm(range(lo,hi),desc=f"Graph shard {s+1}/{n_shards}",leave=False):
            graphs.append(smiles_to_rel_graph(df.iloc[i]["std_smiles"]))
        joblib.dump(graphs,path,compress=3)
        print(f"   ✅ Graph shard {s+1}/{n_shards} save হয়েছে।")
    manifest={"n":len(df),"shard_size":GRAPH_SHARD_SIZE,"n_shards":n_shards,"atom_dim":len(atom_feature(Chem.MolFromSmiles('C').GetAtomWithIdx(0))),"relations":5}
    GRAPH_MANIFEST.write_text(json.dumps(manifest,indent=2))
    print("✅ Molecular relational graph cache সম্পন্ন হয়েছে।")

def load_all_graphs():
    manifest=json.loads(GRAPH_MANIFEST.read_text()); graphs=[]
    for s in range(manifest["n_shards"]): graphs.extend(joblib.load(GRAPH_ROOT/f"shard_{s:03d}.joblib"))
    return graphs[:manifest["n"]]

print("✅ Molecular relation-graph feature functions প্রস্তুত হয়েছে।")
save_runtime_session()


In [ ]:
if "RUNTIME_READY" not in globals():
    from pathlib import Path
    _candidates = [
        Path.cwd() / ".final_ultra_runtime_bootstrap.py",
        Path("D:/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/content/drive/MyDrive/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/mnt/data/Final_Ultra_Project/final_ultra/cache/runtime_bootstrap.py"),
        Path.cwd() / "final_ultra" / "cache" / "runtime_bootstrap.py",
        Path.cwd() / "cache" / "runtime_bootstrap.py",
    ]
    _bootstrap = next((p for p in _candidates if p.exists()), None)
    if _bootstrap is None:
        _hits = list(Path.cwd().rglob("runtime_bootstrap.py"))
        _bootstrap = _hits[0] if _hits else None
    if _bootstrap is None:
        raise RuntimeError("Recovery bootstrap পাওয়া যায়নি। প্রথমবার হলে setup থেকে Kernel-restart Recovery cell পর্যন্ত run করুন।")
    exec(compile(_bootstrap.read_text(encoding="utf-8"), str(_bootstrap), "exec"), globals())
    print("🔄 Kernel restart-এর পর runtime cache থেকে restore হয়েছে।")

build_graph_cache()
GRAPH_LIST = load_all_graphs()
manifest=json.loads(GRAPH_MANIFEST.read_text())
print("✅ Graph cache load হয়েছে।")
print("📌 Molecules:",len(GRAPH_LIST),"| Atom feature dim:",manifest["atom_dim"],"| Relation types:",manifest["relations"])


In [ ]:
class GraphMultiTaskDataset(Dataset):
    def __init__(self,graphs,tabular,indices): self.graphs=graphs; self.tabular=tabular; self.indices=np.asarray(indices,dtype=int)
    def __len__(self): return len(self.indices)
    def __getitem__(self,k):
        idx=int(self.indices[k]); g=self.graphs[idx]
        return {"node_x":torch.from_numpy(g["x"]),"edge_index":torch.from_numpy(g["edge_index"]),"edge_type":torch.from_numpy(g["edge_type"]),
                "tabular":torch.from_numpy(np.asarray(self.tabular[k],dtype=np.float32)),"y":torch.from_numpy(Y_TENSOR_SAFE[idx]),
                "mask":torch.from_numpy(LABEL_MASK[idx]),"idx":torch.tensor(idx,dtype=torch.long)}

def graph_collate(items):
    node_x=[]; edges=[]; types=[]; batch=[]; tabs=[];ys=[];masks=[];idxs=[];offset=0
    for b,it in enumerate(items):
        n=it["node_x"].shape[0]; node_x.append(it["node_x"]); edges.append(it["edge_index"]+offset); types.append(it["edge_type"])
        batch.append(torch.full((n,),b,dtype=torch.long)); tabs.append(it["tabular"]);ys.append(it["y"]);masks.append(it["mask"]);idxs.append(it["idx"]);offset+=n
    return {"node_x":torch.cat(node_x),"edge_index":torch.cat(edges,dim=1) if edges else torch.empty((2,0),dtype=torch.long),
            "edge_type":torch.cat(types) if types else torch.empty((0,),dtype=torch.long),"batch":torch.cat(batch),
            "tabular":torch.stack(tabs),"y":torch.stack(ys),"mask":torch.stack(masks),"idx":torch.stack(idxs)}

def make_graph_loader(dataset,shuffle):
    return DataLoader(dataset,batch_size=RESOURCE["graph_batch"],shuffle=shuffle,num_workers=RESOURCE["num_workers"],
                      pin_memory=(DEVICE.type=="cuda"),collate_fn=graph_collate,persistent_workers=(RESOURCE["num_workers"]>0))

class RelGraphConv(nn.Module):
    def __init__(self,in_dim,out_dim,num_rel=5):
        super().__init__(); self.weight=nn.Parameter(torch.empty(num_rel,in_dim,out_dim)); self.self_lin=nn.Linear(in_dim,out_dim,bias=False); self.bias=nn.Parameter(torch.zeros(out_dim)); nn.init.xavier_uniform_(self.weight)
    def forward(self,x,edge_index,edge_type):
        out=self.self_lin(x); deg=torch.ones(x.shape[0],device=x.device,dtype=x.dtype)
        if edge_index.numel()>0:
            src,dst=edge_index
            agg=torch.zeros(x.shape[0],self.weight.shape[-1],device=x.device,dtype=x.dtype)
            dadd=torch.zeros(x.shape[0],device=x.device,dtype=x.dtype)
            for r in range(self.weight.shape[0]):
                mask=edge_type==r
                if mask.any():
                    s=src[mask]; d=dst[mask]; msg=x[s]@self.weight[r]; agg.index_add_(0,d,msg); dadd.index_add_(0,d,torch.ones_like(d,dtype=x.dtype))
            out=out+agg/(dadd.clamp_min(1).unsqueeze(1)); deg=deg+dadd
        return out+self.bias

def global_mean_max(x,batch):
    B=int(batch.max().item())+1 if batch.numel() else 0; D=x.shape[1]
    mean=torch.zeros(B,D,device=x.device,dtype=x.dtype); mean.index_add_(0,batch,x)
    cnt=torch.bincount(batch,minlength=B).clamp_min(1).to(x.dtype).unsqueeze(1); mean=mean/cnt
    try:
        mx=torch.full((B,D),-torch.inf,device=x.device,dtype=x.dtype)
        mx.scatter_reduce_(0,batch[:,None].expand(-1,D),x,reduce="amax",include_self=True)
    except Exception:
        mx=torch.stack([x[batch==i].max(dim=0).values for i in range(B)])
    return torch.cat([mean,mx],dim=1)

class MolecularRGCNFusion(nn.Module):
    def __init__(self,atom_dim,tab_dim):
        super().__init__()
        self.c1=RelGraphConv(atom_dim,128); self.n1=nn.LayerNorm(128)
        self.c2=RelGraphConv(128,256); self.n2=nn.LayerNorm(256)
        self.c3=RelGraphConv(256,256); self.n3=nn.LayerNorm(256)
        self.tab=nn.Sequential(nn.Linear(tab_dim,512),nn.LayerNorm(512),nn.LeakyReLU(.1),nn.Dropout(.25),nn.Linear(512,256),nn.LeakyReLU(.1))
        self.head=nn.Sequential(nn.Linear(512+256,512),nn.LayerNorm(512),nn.LeakyReLU(.1),nn.Dropout(.35),nn.Linear(512,256),nn.LeakyReLU(.1),nn.Dropout(.25),nn.Linear(256,len(TARGETS)))
    def forward(self,node_x,edge_index,edge_type,batch,tabular):
        h=F.leaky_relu(self.n1(self.c1(node_x,edge_index,edge_type)),.1); h=F.dropout(h,.15,self.training)
        h=F.leaky_relu(self.n2(self.c2(h,edge_index,edge_type)),.1); h=F.dropout(h,.20,self.training)
        h2=F.leaky_relu(self.n3(self.c3(h,edge_index,edge_type)),.1); h=h+h2
        graph=global_mean_max(h,batch); tab=self.tab(tabular); return self.head(torch.cat([graph,tab],dim=1))

def build_rgcn_resources(train_idx,val_idx,test_idx,fold,stage):
    Xtr,Xva,art=prepare_tabular_fold(train_idx,val_idx,mode="dnn",compact=False)
    dte=art["imputer"].transform(np.asarray(X_DESC[test_idx])); dte=art["scaler"].transform(dte)
    Xte=np.concatenate([np.asarray(X_ECFP[test_idx],dtype=np.float32),dte.astype(np.float32)],axis=1).astype(np.float32)
    atom_dim=GRAPH_LIST[0]["x"].shape[1]; model=MolecularRGCNFusion(atom_dim,Xtr.shape[1]).to(DEVICE)
    return {"model":model,"train_loader":make_graph_loader(GraphMultiTaskDataset(GRAPH_LIST,Xtr,train_idx),True),
            "val_loader":make_graph_loader(GraphMultiTaskDataset(GRAPH_LIST,Xva,val_idx),False),
            "test_loader":make_graph_loader(GraphMultiTaskDataset(GRAPH_LIST,Xte,test_idx),False),
            "forward_fn":lambda m,b:m(b["node_x"],b["edge_index"],b["edge_type"],b["batch"],b["tabular"]),
            "pos_weight":compute_pos_weight(train_idx)}

print("✅ Custom relation-specific R-GCN + tabular fusion architecture প্রস্তুত হয়েছে।")
save_runtime_session()


In [ ]:
if "RUNTIME_READY" not in globals():
    from pathlib import Path
    _candidates = [
        Path.cwd() / ".final_ultra_runtime_bootstrap.py",
        Path("D:/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/content/drive/MyDrive/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/mnt/data/Final_Ultra_Project/final_ultra/cache/runtime_bootstrap.py"),
        Path.cwd() / "final_ultra" / "cache" / "runtime_bootstrap.py",
        Path.cwd() / "cache" / "runtime_bootstrap.py",
    ]
    _bootstrap = next((p for p in _candidates if p.exists()), None)
    if _bootstrap is None:
        _hits = list(Path.cwd().rglob("runtime_bootstrap.py"))
        _bootstrap = _hits[0] if _hits else None
    if _bootstrap is None:
        raise RuntimeError("Recovery bootstrap পাওয়া যায়নি। প্রথমবার হলে setup থেকে Kernel-restart Recovery cell পর্যন্ত run করুন।")
    exec(compile(_bootstrap.read_text(encoding="utf-8"), str(_bootstrap), "exec"), globals())
    print("🔄 Kernel restart-এর পর runtime cache থেকে restore হয়েছে।")

# Graph cache is restored from disk after a kernel restart if it is not present in memory.
if "GRAPH_LIST" not in globals():
    GRAPH_LIST = load_all_graphs()
RGCN_RESULT = run_torch_cv(
    "Molecular R-GCN Fusion",
    build_rgcn_resources,
    RESOURCE["rgcn_epochs"], lr=2e-4, weight_decay=2e-5
)


## Model 10 — Leakage-safe Soft-Voting Ensemble

The primary ensemble uses equal weights fixed **before** examining test-fold performance. It averages only outer-fold out-of-fold probabilities from completed base models. This avoids choosing weights directly from the same test predictions. An equal-weight ensemble is less aggressive than test-tuned weighting, but methodologically safer.


In [ ]:
if "RUNTIME_READY" not in globals():
    from pathlib import Path
    _candidates = [
        Path.cwd() / ".final_ultra_runtime_bootstrap.py",
        Path("D:/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/content/drive/MyDrive/Drug_Toxicity/final_ultra/cache/runtime_bootstrap.py"),
        Path("/mnt/data/Final_Ultra_Project/final_ultra/cache/runtime_bootstrap.py"),
        Path.cwd() / "final_ultra" / "cache" / "runtime_bootstrap.py",
        Path.cwd() / "cache" / "runtime_bootstrap.py",
    ]
    _bootstrap = next((p for p in _candidates if p.exists()), None)
    if _bootstrap is None:
        _hits = list(Path.cwd().rglob("runtime_bootstrap.py"))
        _bootstrap = _hits[0] if _hits else None
    if _bootstrap is None:
        raise RuntimeError("Recovery bootstrap পাওয়া যায়নি। প্রথমবার হলে setup থেকে Kernel-restart Recovery cell পর্যন্ত run করুন।")
    exec(compile(_bootstrap.read_text(encoding="utf-8"), str(_bootstrap), "exec"), globals())
    print("🔄 Kernel restart-এর পর runtime cache থেকে restore হয়েছে।")

BASE_MODELS=[
    "Random Forest","Extra Trees","XGBoost","SVM-RBF","DeepTox-style MT-DNN",
    "Multitask CapsNet + RBM","Multi-channel 2D CNN","Compact MTL-DNN","Molecular R-GCN Fusion"
]
available=[]; arrays=[]
for name in BASE_MODELS:
    p=RESULT_DIR/model_slug(name)/"oof_probabilities.npy"
    if p.exists():
        arr=np.load(p)
        if arr.shape==(len(df),len(TARGETS)):
            available.append(name); arrays.append(arr)
if len(arrays)<2:
    raise RuntimeError("Ensemble-এর জন্য অন্তত 2টি completed base model প্রয়োজন।")

stack=np.stack(arrays,axis=0)
ensemble_prob=np.nanmean(stack,axis=0).astype(np.float32)
metrics=[]
for fold in range(N_FOLDS):
    idx=np.where(df["fold"].to_numpy()==fold)[0]
    metrics.append(metrics_from_predictions(idx,ensemble_prob[idx],fold,"Soft-Voting Ensemble"))
metrics=pd.concat(metrics,ignore_index=True)
per_task,fold_macro,overall=save_model_outputs("Soft-Voting Ensemble",ensemble_prob,metrics)

weights_df=pd.DataFrame({"Base model":available,"Weight":[1/len(available)]*len(available)})
weights_df.to_csv(RESULT_DIR/model_slug("Soft-Voting Ensemble")/"ensemble_weights.csv",index=False)
print("✅ Ensemble-এ ব্যবহৃত base models:")
display(weights_df.style.format({"Weight":"{:.3f}"}).set_caption("Pre-declared Equal Ensemble Weights"))
show_model_result("Soft-Voting Ensemble",per_task,overall)
ENSEMBLE_RESULT={"oof":ensemble_prob,"metrics":metrics,"per_task":per_task,"fold_macro":fold_macro,"overall":overall}


# Final Results, Visualizations and Research Exports


In [ ]:
def collect_overall_results():
    rows=[]
    for p in RESULT_DIR.glob("*/overall_summary.json"):
        try: rows.append(json.loads(p.read_text(encoding="utf-8")))
        except Exception: pass
    if not rows: return pd.DataFrame()
    out=pd.DataFrame(rows).drop_duplicates("Model",keep="last").sort_values("Mean AUC",ascending=False).reset_index(drop=True)
    return out

overall_results=collect_overall_results()
if overall_results.empty:
    raise RuntimeError("কোনো model result পাওয়া যায়নি। আগে model cells run করুন।")
overall_results.to_csv(RESULT_DIR/"FINAL_overall_results.csv",index=False)

print("🏁 Final model comparison তৈরি হয়েছে।")
display(overall_results.style
        .format({"Mean AUC":"{:.4f}","AUC SD":"{:.4f}","Mean Accuracy":"{:.4f}","Accuracy SD":"{:.4f}",
                 "Best Endpoint AUC":"{:.4f}","Best Endpoint Accuracy":"{:.4f}","Target AUC Gap":"{:+.4f}"})
        .background_gradient(subset=["Mean AUC"],cmap="YlGn")
        .background_gradient(subset=["Mean Accuracy"],cmap="Blues")
        .set_caption("Final Ultra — Overall 3-fold Model Performance"))

best=overall_results.iloc[0]
if best["Mean AUC"]>=TARGET_MEAN_AUC:
    print(f"🎯 Research target achieved: best mean AUC = {best['Mean AUC']:.4f} ≥ {TARGET_MEAN_AUC:.2f}")
else:
    print(f"📌 Best observed mean AUC = {best['Mean AUC']:.4f}; target {TARGET_MEAN_AUC:.2f} এখনো অর্জিত হয়নি।")
    print("🔬 Score বাড়ানোর জন্য test-fold tuning বা label leakage করা হবে না; next step হবে external ToxKG, pretraining, অথবা training-only ablation।")


In [ ]:
plot_df=overall_results.sort_values("Mean AUC")
fig,axes=plt.subplots(1,2,figsize=(17,7))
axes[0].barh(plot_df["Model"],plot_df["Mean AUC"],xerr=plot_df["AUC SD"],color="#2A9D8F",alpha=.9)
axes[0].axvline(TARGET_MEAN_AUC,color="#D62828",ls="--",lw=1.5,label="Target AUC = 0.90")
axes[0].set_xlim(max(.45,plot_df["Mean AUC"].min()-.05),1.01);axes[0].set_title("Model Comparison — Mean ROC-AUC",fontweight="bold");axes[0].legend()
for i,v in enumerate(plot_df["Mean AUC"]): axes[0].text(v+.003,i,f"{v:.3f}",va="center")

axes[1].scatter(overall_results["Mean AUC"],overall_results["Mean Accuracy"],s=100,c=np.arange(len(overall_results)),cmap="viridis")
for _,r in overall_results.iterrows(): axes[1].annotate(r["Model"],(r["Mean AUC"],r["Mean Accuracy"]),xytext=(5,4),textcoords="offset points",fontsize=8)
axes[1].axvline(TARGET_MEAN_AUC,color="#D62828",ls="--",lw=1);axes[1].set_xlabel("Mean ROC-AUC");axes[1].set_ylabel("Mean Accuracy");axes[1].set_title("AUC vs Accuracy — Imbalance-aware Interpretation",fontweight="bold")
plt.tight_layout();plt.savefig(FIG_DIR/"FINAL_model_comparison.png",dpi=240,bbox_inches="tight");plt.show(); plt.close()
print("✅ Final model comparison plots save হয়েছে।")


In [ ]:
auc_rows=[];acc_rows=[]
for _,r in overall_results.iterrows():
    p=RESULT_DIR/model_slug(r["Model"])/"per_task_summary.csv"
    if p.exists():
        t=pd.read_csv(p).set_index("Endpoint")
        auc_rows.append(pd.Series(t["AUC_Mean"],name=r["Model"]))
        acc_rows.append(pd.Series(t["Accuracy_Mean"],name=r["Model"]))
auc_heat=pd.DataFrame(auc_rows);acc_heat=pd.DataFrame(acc_rows)
auc_heat.to_csv(RESULT_DIR/"FINAL_auc_heatmap_table.csv");acc_heat.to_csv(RESULT_DIR/"FINAL_accuracy_heatmap_table.csv")

plt.figure(figsize=(15,max(5,.55*len(auc_heat))))
sns.heatmap(auc_heat,annot=True,fmt=".3f",cmap="RdYlGn",vmin=.5,vmax=1,linewidths=.4,cbar_kws={"label":"Mean ROC-AUC"})
plt.title("AUC Heatmap — All Models × Tox21 Endpoints",fontweight="bold");plt.xlabel("Endpoint");plt.ylabel("Model");plt.tight_layout()
plt.savefig(FIG_DIR/"FINAL_auc_heatmap.png",dpi=240,bbox_inches="tight");plt.show(); plt.close()

plt.figure(figsize=(15,max(5,.55*len(acc_heat))))
sns.heatmap(acc_heat,annot=True,fmt=".3f",cmap="YlGnBu",vmin=.5,vmax=1,linewidths=.4,cbar_kws={"label":"Mean Accuracy"})
plt.title("Accuracy Heatmap — All Models × Tox21 Endpoints",fontweight="bold");plt.xlabel("Endpoint");plt.ylabel("Model");plt.tight_layout()
plt.savefig(FIG_DIR/"FINAL_accuracy_heatmap.png",dpi=240,bbox_inches="tight");plt.show(); plt.close()
print("✅ AUC এবং Accuracy heatmap তৈরি হয়েছে।")


In [ ]:
best_name=overall_results.iloc[0]["Model"]
best_slug=model_slug(best_name)
best_per=pd.read_csv(RESULT_DIR/best_slug/"per_task_summary.csv").sort_values("AUC_Mean",ascending=False)
best_prob=np.load(RESULT_DIR/best_slug/"oof_probabilities.npy")

fig,axes=plt.subplots(1,2,figsize=(17,6))
axes[0].bar(best_per["Endpoint"],best_per["AUC_Mean"],yerr=best_per["AUC_SD"].fillna(0),color="#2A9D8F")
axes[0].axhline(.9,color="#D62828",ls="--");axes[0].set_ylim(0,1.03);axes[0].set_title(f"{best_name} — Endpoint AUC",fontweight="bold");axes[0].tick_params(axis="x",rotation=40)
axes[1].bar(best_per["Endpoint"],best_per["Accuracy_Mean"],yerr=best_per["Accuracy_SD"].fillna(0),color="#F4A261")
axes[1].set_ylim(0,1.03);axes[1].set_title(f"{best_name} — Endpoint Accuracy",fontweight="bold");axes[1].tick_params(axis="x",rotation=40)
plt.tight_layout();plt.savefig(FIG_DIR/"FINAL_best_model_endpoints.png",dpi=240,bbox_inches="tight");plt.show(); plt.close()

best_ep=best_per.iloc[0]["Endpoint"];hard_ep=best_per.iloc[-1]["Endpoint"]
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,ep,title in [(axes[0],best_ep,"Best AUC Endpoint"),(axes[1],hard_ep,"Hardest AUC Endpoint")]:
    j=TARGETS.index(ep);valid=LABEL_MASK[:,j].astype(bool);yt=Y[:,j][valid].astype(int);yp=(best_prob[:,j][valid]>=THRESHOLD).astype(int)
    cm=confusion_matrix(yt,yp,labels=[0,1]);sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",ax=ax,cbar=False,xticklabels=["Non-toxic","Toxic"],yticklabels=["Non-toxic","Toxic"])
    ax.set_title(f"{title}\n{ep}",fontweight="bold");ax.set_xlabel("Predicted");ax.set_ylabel("True")
plt.suptitle(f"Confusion Matrices — {best_name}",fontweight="bold");plt.tight_layout();plt.savefig(FIG_DIR/"FINAL_confusion_best_hardest.png",dpi=240,bbox_inches="tight");plt.show(); plt.close()
print("✅ Best model endpoint plots এবং confusion matrices তৈরি হয়েছে।")


In [ ]:
fold_rows=[]
for _,r in overall_results.iterrows():
    p=RESULT_DIR/model_slug(r["Model"])/"fold_macro_summary.csv"
    if p.exists():
        t=pd.read_csv(p);t["Model"]=r["Model"];fold_rows.append(t)
fold_all=pd.concat(fold_rows,ignore_index=True)
plt.figure(figsize=(15,6))
sns.barplot(data=fold_all,x="Model",y="Mean_AUC",hue="Fold",palette="Set2")
plt.axhline(TARGET_MEAN_AUC,color="#D62828",ls="--",lw=1.2);plt.ylim(0,1.03);plt.xticks(rotation=35,ha="right")
plt.title("3-fold Cross-validation — Macro Mean ROC-AUC",fontweight="bold");plt.tight_layout();plt.savefig(FIG_DIR/"FINAL_cv_fold_auc.png",dpi=240,bbox_inches="tight");plt.show(); plt.close()
print("✅ Fold-wise CV visualization তৈরি হয়েছে।")


In [ ]:
# Plot training curves for completed neural models when history is available.
neural_names=["DeepTox-style MT-DNN","Multitask CapsNet + RBM","Multi-channel 2D CNN","Compact MTL-DNN","Molecular R-GCN Fusion"]
for name in neural_names:
    p=RESULT_DIR/model_slug(name)/"training_histories.json"
    if not p.exists(): continue
    hist=json.loads(p.read_text())
    plt.figure(figsize=(10,5)); plotted=False
    for key,val in hist.items():
        h=val.get("history",[])
        if h and "val_auc" in h[0]:
            plt.plot([x["epoch"] for x in h],[x["val_auc"] for x in h],label=key);plotted=True
    if plotted:
        plt.axhline(.9,color="#D62828",ls="--",lw=1);plt.xlabel("Epoch");plt.ylabel("Validation macro AUC");plt.ylim(0,1.02)
        plt.title(f"{name} — Training Convergence",fontweight="bold");plt.legend(fontsize=8);plt.tight_layout()
        plt.savefig(FIG_DIR/f"training_{model_slug(name)}.png",dpi=200,bbox_inches="tight");plt.show(); plt.close()
print("✅ Available neural training curves তৈরি হয়েছে।")


## 11. Reproducibility package and final research notes


In [ ]:
# Save final environment information.
versions={}
for name,module in [("python",None),("numpy",np),("pandas",pd),("scikit-learn",__import__("sklearn")),("rdkit",__import__("rdkit")),("xgboost",__import__("xgboost")),("torch",torch)]:
    versions[name]=sys.version.split()[0] if module is None else getattr(module,"__version__","unknown")
with open(RESULT_DIR/"FINAL_environment_versions.json","w") as f: json.dump(versions,f,indent=2)

# Create a compact README inside the result folder.
readme=f"""# Final Ultra Tox21 Results\n\n- Dataset SHA256: {RAW_SHA256}\n- Curated molecules: {len(df)}\n- Active endpoints: {len(TARGETS)}\n- Outer CV: {N_FOLDS}-fold scaffold-balanced\n- Missing targets: never imputed as class labels\n- Threshold: {THRESHOLD}\n- Target mean AUC: {TARGET_MEAN_AUC}\n- Device: {DEVICE}\n\nImportant: The molecular R-GCN in this notebook uses bond relations plus fingerprint/descriptor fusion. The external ToxKG used by the cited paper was not part of the uploaded data.\n"""
(RESULT_DIR/"README.md").write_text(readme,encoding="utf-8")

print("✅ Final result tables, figures, environment versions এবং README save হয়েছে।")
print("📁 Results:",RESULT_DIR)
print("📁 Figures:",FIG_DIR)
print("📁 Models/checkpoints:",MODEL_DIR)
print("📁 OOF predictions:",PRED_DIR)
print("\n🔬 Final researcher conclusion:")
print("• Missing target labels were masked, never converted into true 0/1 labels.")
print("• Scaffold groups were frozen before model fitting.")
print("• All learned preprocessing was fitted on training data only.")
print("• Deep models are epoch-checkpointed; classical models are fold-task checkpointed.")
print("• Reported results come from the current run, not copied from literature.")


## References from the uploaded papers

1. Mayr, A., Klambauer, G., Unterthiner, T., & Hochreiter, S. (2016). *DeepTox: Toxicity Prediction using Deep Learning*. Frontiers in Environmental Science, 3:80.  
2. Wang, Y. et al. (2021). *Multitask CapsNet: An Imbalanced Data Deep Learning Method for Predicting Toxicants*. ACS Omega, 6, 26545-26555.  
3. Yuan, Q. et al. (2019). *Toxicity Prediction Method Based on Multi-Channel Convolutional Neural Network*. Molecules, 24, 3383.  
4. Rao, K. V. et al. (2025). *A Multi-Task Graph Convolutional Network for Molecular Toxicity Prediction Using the Tox21 Dataset*.  
5. Narayana, V. L. et al. (2026 PDF header). *Deep Learning-Based Toxicity Prediction of Drug Compounds Using Multi-Task Neural Networks on the Tox21 Dataset*.  
6. Xie, J. et al. (2025). *Graph Neural Network-Based Toxicity Prediction by Integrating Molecular Fingerprints and Knowledge Graph Features*. Toxics, 13, 953.  
7. `Drug_Path.pdf` — uploaded research roadmap and dataset audit.

---

### Final execution advice

For the strongest reproducible run, use Google Colab T4 or the 12 GB GPU PC, keep `SMOKE_TEST=False`, place the dataset and notebook inside the persistent `Drug_Toxicity` project folder, and run top-to-bottom once. After an interruption, rerun the interrupted model cell; it will restore the saved session and continue from checkpoints.
